# <center>多智能体协作模式</center>

&emsp;&emsp;在此之前，我们大多写过单个智能体——一个大语言模型挂上几个工具，让它自己循环着把活干完。这门课要做的，是带我们把视角从"一个智能体"切换到"一支智能体团队"：四种主流的多智能体协作模式——Workflow 流程编排、Supervisor 集中调度、Hierarchical 分层协同、Swarm 自主协作——逐一讲透，每一种模式都用五个仍在活跃维护的主流框架各写一段能直接运行的最小实现，最后落成四个完整的案例项目：销售数据报表流水线、多轮研究编排、多智能体软件交付、在线客服中心。

&emsp;&emsp;这门课的<b>最大里程碑</b>是其中那个软件交付项目——我们会在浏览器里看着一个技术总监智能体现场制定接口定义，前端、后端、测试三个小组并行写代码、互相打回返工，最后一键启动它们造出来的软件。下面这张封面图把这条从单体到团队、再到四个能跑系统的路径拼在了一起。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/opening-multi-agent-overview-f810d7dd.png" width=80%></div>

<br>

&emsp;&emsp;封面图下方那条时间轴就是接下来的节奏：本章先铺一张协作模式的<font color=red>全景地图</font>和框架版图；之后四章每章拆透一种模式，五个框架各写一段能跑的最小代码；第六章把"什么业务用什么模式"的选型判断立起来；第七到第十章四个完整项目把模式落到地；最后一章回顾整条来路。每一章末尾都会留下一个可以直接复用的产物——一段能跑的代码，或者一个能上线的系统。

&emsp;&emsp;这门课默认我们已经会写单个智能体——给模型挂过工具、写过 function calling，或者用过任意一个智能体框架。技术栈方面，五个框架的版本以 2026 年 6 月的实测为准：LangGraph 1.2.4、CrewAI 1.14.6、OpenAI Agents SDK 0.17.4、Microsoft Agent Framework 1.0 GA（实测 1.8.x）、Claude Agent SDK 0.2.95；模型统一用 OpenRouter 上的 deepseek，密钥放在课件目录的 `.env` 文件里读取（python-dotenv 自动加载），代码里绝不硬编码。多智能体生态发版很快，遇到版本号对不上时以各家官方最新版为准，机制不变。

&emsp;&emsp;<b>动手前先配好运行环境。</b>本课代码分两类、各用各的环境，开跑前花几分钟配一次即可。<b>一是这本 notebook 里的 第二部分 示例</b>（五个框架的最小实现），跑在课件目录附带的一个"一体化"虚拟环境里——它把五个框架装进同一个 Python，建一次、在 Jupyter 里选中它当内核就行。在课件目录（放 `.ipynb` 这层）下执行：

```bash
python3.12 -m venv .venv
./.venv/bin/pip install -r requirements-lock.txt --no-deps     # 为什么用 lock + --no-deps:见本文件顶部说明
./.venv/bin/python -m ipykernel install --user --name multi-agent --display-name "多智能体课件"
cp .env.example .env          # 再编辑 .env,把 OPENROUTER_API_KEY 填成自己在 https://openrouter.ai/keys 申请的 key
```

&emsp;&emsp;装好后在 Jupyter 右上角把内核切成"多智能体课件"、`.env` 里填好密钥，就能从头一路运行 第二部分 的所有示例 cell。<b>二是 第三部分 的四个完整项目</b>，它们是独立的 Web 服务，各有自己的目录、依赖和虚拟环境，搭建与启动方式在第六章末尾和每章的"跑起来观察"里单独讲，到那时再配也不迟。

## <center>第一章 多智能体协作全景</center>

&emsp;&emsp;一张全局地图，是走进任何复杂领域前最值得先拿到的东西。在多智能体这个领域，最容易让人迷路的不是某个框架的 API，而是"一共有几种协作形态、它们之间是什么关系、各家框架分别擅长哪一种"。

<div align=center><font size=2 color=#999999>第一章：从单个智能体碰到上限，到四种协作结构的演进地图</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L1-pattern-map-37ae4aee.png" width=80%></div>

<br>

&emsp;&emsp;上一段我们已经会写的单个智能体，是这张图最左边那个被任务压住的机器人。本章先讲清它碰到上限的三种信号、多智能体能换来什么、又要付出什么代价；再理顺 Workflow 到 Agent 这条自主性谱系；然后展开本课主线的四种协作模式，给每种模式配一个直观的业务对应；接着摊开六个主流框架在 2026 年 6 月的真实版图（其中 AutoGen 已进入维护模式，所以后面写代码时只用其余仍在活跃维护的五个）；最后用一节讲清智能体之间靠什么传递信息——剥开各家 API，底下只有两种通信范式。学完这一章，我们手里就有了一张能在任意业务场景里说出"这该用哪种模式、配哪个框架"的全景地图。

### 1.1 单 Agent 的能力上限

&emsp;&emsp;先说清楚"为什么要从单个智能体走向多个"，否则后面所有模式都会显得多此一举。下面这张图把单个智能体的三种瓶颈、多智能体换来的三项价值、以及要付出的代价摆在一起。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L1.1-single-agent-ceiling-da235335.png" width=80%></div>

> <font size=2>【名词解释】<b><font color=red>Agent</font>（Agent，智能体）</b>:一个大语言模型加一组工具，模型自己决定下一步调哪个工具、什么时候收尾，循环着把任务做完的程序。本课沿用业界惯例直接称"智能体"或"Agent"。</font>

&emsp;&emsp;我们已经会写的单个智能体，本质是一个模型加一组工具的循环。这种形态在三种场景下会明显碰到上限，而这三种瓶颈信号，正是该升级到多智能体的依据。

- <b>工具越挂越多</b>：一个智能体上挂了十几个工具时，模型每一步选错工具的概率上升，工具描述本身也会把上下文撑爆。

- <b>上下文越塞越复杂</b>：所有子任务共用一条对话历史，几轮之后既塞不下、又互相污染，模型开始记串。

- <b>一个 prompt 身兼数职</b>：同一个系统提示词很难让模型既当严谨的研究员、又当精确的数学家、还当流畅的写手，几种角色互相干扰。

&emsp;&emsp;多智能体把一个大模型拆成一支团队，换来的是三项核心价值。<b>并行化</b>——多个子智能体各自独立检索、独立推理，互不等待，Anthropic 实测复杂查询的耗时能因此降到约十分之一。<b>上下文隔离</b>——每个智能体有自己独立的上下文窗口，中间过程留在各自内部，只把最终结论交回上层，彼此不污染。<b>专业化分工</b>——每个智能体可以用不同的系统提示词、不同的工具集、甚至不同的模型，各管一摊。

&emsp;&emsp;这份能力不是免费的，代价要直说。Anthropic 在它的研究系统里实测，多智能体系统消耗的 token 大约是普通聊天交互的 15 倍——这个"约 15×"的基线是聊天交互，不是单个智能体。更进一步，在 BrowseComp 这个评测上，token 用量本身就解释了约 80% 的性能方差。BrowseComp 考的是浏览智能体去找很难找到的信息，所以这条结论的潜台词是：多智能体之所以更强，很大一部分原因就是它愿意花更多 token 去解决问题。

> <font size=2>【名词解释】<b><font color=red>token</font>（token，词元）</b>:大语言模型处理文本的最小计费单位，一个汉字或一个英文词大致对应一到几个 token，模型调用的成本和上下文长度都按 token 计。</font>

&emsp;&emsp;收益这一侧同样有实测数据支撑。Anthropic 用 Claude Opus 4 当主管、Claude Sonnet 4 当下属的多智能体研究系统，在它的内部研究评测上，比单体 Claude Opus 4 高出 90.2%。

&emsp;&emsp;把账摆到一起看，结论就清楚了：多智能体不是免费的午餐，15 倍 token 的账应该在立项时就算清楚，而不是上线后看账单才后悔。一个任务到底该不该拆成多智能体，是有判断方法的——这套选型判断力，我们在第六章会专门立起来，本章只需先记住"它有代价、要算账"这一条。

### 1.2 自主性谱系：从固定流程到模型自主

&emsp;&emsp;在铺开四种协作模式之前，有一组更底层的概念要先理顺。同样是"用模型加工具完成任务"，从"把每一步流程都固定在代码里"，到"让模型自己决定下一步走哪条路"，其实是一条连续的自主性谱系——两端分别叫 Workflow 和 Agent，中间是平滑过渡的大片灰色地带，而不是黑白分明的两个阵营。下面这张图把这条谱系和它两端的特征画了出来。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L1.2-autonomy-spectrum-70e824c9.png" width=80%></div>

> <font size=2>【名词解释】<b><font color=red>Workflow</font>（Workflow，工作流）</b>:模型和工具走的是预先固定在代码里的路径，每一步执行什么、按什么顺序，由开发者决定，模型只负责产出内容。</font>

> <font size=2>【名词解释】<b><font color=red>Orchestrator</font>（Orchestrator，编排器）</b>:多智能体系统里负责"派活和收口"的协调角色，决定把哪个子任务交给哪个智能体、什么时候汇总结果。</font>

&emsp;&emsp;谱系的 <font color=red>Workflow</font> 这一端，模型和工具走预先写好的固定路径，执行顺序由开发者定死，结果可预测、出问题好排查；Agent 那一端，模型自己决定走哪条路径、调用什么工具，灵活，但成本更高、行为更难预测。要紧的是，这两端并不是互相竞争的对立选项。2026 年的多智能体研究综述基本形成了一个共识：Workflow 和 Agent 是相辅相成的，真实的生产系统往往是两者的混合体——一条固定的流程编排，中间可能嵌进一个"主管派工人"的自主步骤；一个自主 agent，也可能把某个子任务丢给一小群专家去交叉验证。把它们分成两端依然有用，因为这给了我们一个思考的起点和一套共同的语言，哪怕最后搭出来的系统是个混合体。

&emsp;&emsp;落到实践，有一条朴素的原则值得记住：用能解决问题的最简方案，只在真正需要时才往上加复杂度。具体就是一条阶梯——一次模型调用能解决就别上 Workflow，Workflow 扛不住再上单个 Agent，单个 Agent 也扛不住才上多智能体。

&emsp;&emsp;最早把 Workflow 的常见形态系统地归纳出来的，是 Anthropic 在 2024 年底那篇《Building Effective Agents》。它总结的五种编排原语，到今天仍是搭多智能体系统的底层零件，这里先认识一下，不逐个展开：

- <b>Prompt Chaining（提示链）</b>：把任务拆成顺序的几步，每一步处理上一步的输出。

- <b>Routing（路由）</b>：先对输入分类，再分发到专门的后续处理。

- <b>Parallelization（并行）</b>：把独立子任务并行跑，或同一任务多跑几遍再投票。

- <b>Orchestrator-Workers（编排者-工人）</b>：一个中央模型动态拆任务、派给工人，适合子任务无法提前预知的场景。

- <b>Evaluator-Optimizer（评估-优化）</b>：一个模型负责生成、另一个负责评估，来回迭代改进。

&emsp;&emsp;这五种原语里，Prompt Chaining 对应本章马上要讲的流程编排，Orchestrator-Workers 对应集中调度。把这条自主性谱系和这五个零件摆在一起，正好就是下一节四种协作模式的基础——四种模式说到底就是这条谱系上的不同落点，各自用不同的零件搭起来。各家框架又把这些零件做成了写法差异很大的 API，这正是后面四章逐一对照五个框架的意义。

### 1.3 四种主流协作模式地图

&emsp;&emsp;本课主线锁定四种协作模式。这四种不是随意挑的，它们沿着一条主轴排开：控制权从"完全由代码固定"逐步交还给"每个智能体自己拿主意"。先各用一段话把它们定义清楚，再各配一个直观的业务对应。

<div align=center><font size=2 color=#999999>四种协作模式的结构形态与适用业务对照</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L1.3-four-patterns-45798f0c.png" width=80%></div>

- <b>Workflow 流程编排</b>：执行顺序固定在代码里，上一道工序的输出就是下一道工序的输入，模型只在每个工位上产出内容。直观对应——销售报表生成，查库、统计、画图、成文四道工序固定，是典型的流水线。

- <b>Supervisor 集中调度</b>：一个主管看全局、派活给下属、收口汇总，下属之间互不可见。直观对应——多轮研究编排，主管按主题挑几个专家并行检索，再把结果综合成简报。

- <b>Hierarchical 分层协同</b>：在主管之上再加主管，多层委派、逐层收窄上下文。直观对应——软件交付团队，技术总监管三个组长、组长各管几个工程师，还带质检返工。

- <b>Swarm 自主协作</b>：没有中心，控制权在平级智能体之间接力传递，谁接手由当前智能体自己决定。直观对应——在线客服，用户问着问着话题就变了，会话焦点跟着用户在专员之间漂移。

&emsp;&emsp;主线之外还有两种周边模式，知道它们存在即可，不展开：<b>Collaboration（对等共享）</b>是多个智能体共享同一份状态、平等协作，<b>Router（路由分发）</b>是先判断输入类型再分给对应处理者——本质上 Router 是 Workflow 五原语里 Routing 的智能体版，Collaboration 则是 Swarm 在"不转移控制权、只共享信息"方向上的变体。

&emsp;&emsp;讲到 Swarm，有一处特别容易混淆，必须在这里讲清楚：库和模式是两回事。OpenAI 早年的 Swarm 是一个实验性的库，它在 2025 年 3 月起已经由 OpenAI Agents SDK 接棒，官方 README 推荐迁移、不再积极维护——这是"库退役"。但 Swarm 作为一种协作模式，今天是现役且实现齐全的：langgraph-swarm、OpenAI Agents SDK 的 handoffs、MAF 的 HandoffBuilder 都在生产可用。所以以后听到"Swarm 退役了"，要分清说的是那个实验库，还是这种接力模式——前者退役，后者活得好好的。

> <font size=2>【名词解释】<b><font color=red>handoff</font>（handoff，控制权交接）</b>:Swarm 模式的核心动作，本质是一次特殊的工具调用，当前智能体调用它就把对话控制权移交给另一个智能体。</font>

### 1.4 框架版图 2026

&emsp;&emsp;模式是概念，框架是把概念变成可运行代码的工具。所以这一节先用一张图把六个主流框架的版图摊开，再用表格逐项对照。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L1.4-framework-landscape-f585182f.png" width=80%></div>

&emsp;&emsp;先把六个主流框架在 2026 年 6 月的真实版图摊开——出品方是谁、当前版本、核心概念，再用三把硬指标量一量它们当下的位置：GitHub 星看历史关注度、近 30 天下载量看当下使用面、近 3 个月 commit 看维护还活不活跃。一张表看全。

<div align=center><font size=2 color=#999999>六个主流多智能体框架的三维版图（GitHub 星看历史关注 / 月下载量看当下使用 / 近三月 commit 看维护活跃）</font></div>

<div align="center">
<table width="80%">
<thead><tr>
<th>框架</th><th>出品方</th><th>版本（2026-06）</th><th>核心概念</th><th>GitHub 星（历史关注）</th><th>近 30 天下载（当下使用）</th><th>近 3 月 commit（维护活跃）</th>
</tr></thead>
<tbody>
<tr><td>LangGraph</td><td>LangChain</td><td>1.2.4</td><td>图加 Command 原语</td><td>约 3.4 万</td><td>约 5860 万</td><td>372</td></tr>
<tr><td>OpenAI Agents SDK</td><td>OpenAI</td><td>0.17.4（Pre-1.0）</td><td>极简四原语加 Runner</td><td>约 2.7 万</td><td>约 3160 万</td><td>395</td></tr>
<tr><td>Claude Agent SDK</td><td>Anthropic</td><td>0.2.95</td><td>query 加子智能体委派</td><td>约 0.7 万</td><td>约 1540 万</td><td>267</td></tr>
<tr><td>CrewAI</td><td>CrewAI 公司</td><td>1.14.6</td><td>角色驱动（Crews + Flows）</td><td>约 5.3 万</td><td>约 1520 万</td><td>459</td></tr>
<tr><td>AutoGen</td><td>Microsoft</td><td>维护模式（末版 0.7.5）</td><td>GroupChat 等历史原语</td><td>约 5.9 万</td><td>约 145 万</td><td>3</td></tr>
<tr><td>Microsoft Agent Framework</td><td>Microsoft</td><td>1.0 GA，实测 1.8.x</td><td>Agent 加图工作流</td><td>约 1.1 万</td><td>约 113 万</td><td>627</td></tr>
</tbody>
</table>
</div>

> <font size=2>【名词解释】<b><font color=red>GA</font>（General Availability，正式发布）</b>:软件版本达到生产可用、接口稳定、对外正式发布的状态，相对于 Pre-1.0（接口仍在演进的早期阶段）。</font>

&emsp;&emsp;表里后三列要放在一起读，单看任何一列都会被带偏。GitHub 星数反映的是历史关注度，是好几年累积下来的口碑，但它不告诉我们这个框架现在还有没有人用、还有没有人维护；近 30 天的下载量更贴近当下的真实使用面——这里要留一句保留：自动化构建和依赖传递安装会把下载量放大，它并不严格等于真实用户数，但六家之间量级上的差距，已经足够说明谁用得广、谁还小众；近 3 个月的 commit 数则直接量出维护还活不活跃。三个指标一起看，才看得清一个框架眼下的真实位置。

&emsp;&emsp;有一组对照特别耐人寻味——Claude Agent SDK 的 GitHub 星数全场最低（约 0.7 万），月下载量却排到第三（约 1540 万），比星数高它七八倍的 CrewAI 还略多一点。这个反差透露出一件事：它的安装量很大程度上是被 Claude Code 生态和工具链带动的，大量开发环境因为在用 Claude Code 而顺带装上了它，而不是社区把它当成一个独立框架在专门研究、讨论。所以判断一个框架的真实热度，下载量和星数都得看，还得分清哪一部分是生态顺带来的。下载量这个指标最直接地排出了当下使用面的座次：<font color=red>LangGraph</font> 以约 5860 万的月下载量断层第一，占了六家总和的近一半；OpenAI Agents SDK 约 3160 万排第二，这两家是眼下真正铺得最开的框架，后面四章里它俩对各种模式的原生覆盖面也最广，和使用面的座次正好相互印证。维护活跃度则排出另一个座次：<font color=red>MAF</font> 近 3 个月 627 个 commit 全场第一，月下载量眼下还垫底（约 113 万），但投入力度最猛——它是微软把 AutoGen 接棒过来后正在主推的方向，下载量低只是因为还在铺开的早期；CrewAI 以 459 个 commit 紧随其后，是另一种均衡，星数全场第二、维护也活跃，背后还有商业化在推（官方称社区认证开发者已超过十万）。

&emsp;&emsp;把这三个指标合起来，本课对六家的选型定位就清楚了：当下生态最稳、最值得重点学的是 LangGraph、CrewAI、OpenAI Agents SDK、MAF 这四家；Claude Agent SDK 作为 Claude 生态里的一等选择补充进来；AutoGen 则只当历史对照和迁移对象，不再作为主线框架。

&emsp;&emsp;这张表还有几处需要说清楚，免得读出歧义。Microsoft Agent Framework（后面简称 MAF）于 2026 年 4 月 3 日发布 1.0 GA 这个里程碑版本，而本课四个案例实测用的是它的 agent-framework-core 1.8.x 小版本——"1.0 GA"指的是稳定里程碑、"1.8.x"是当前的小版本号，两者说的是同一个框架的不同维度，单提哪一个都不完整。Claude Agent SDK 是 0.2.95（2026-06-09），它发版极其频繁，实测时以当时的最新版为准即可，机制不变。而 <font color=red>AutoGen</font> 这一列最需要单独交代，因为它身上藏着这张表最该记住的一条选型教训。它是多智能体框架的架构鼻祖，GroupChat 这类协作原语最早就是它提出的，GitHub 星数也因此全场最高（约 5.9 万）。可只要往后三列扫一眼就会发现：它近 3 个月只有 3 个 commit，月下载量也跌到六家垫底（约 145 万）——星数最高、维护却近乎停滞。这正是"选型不能只盯星数"最直白的例证：那 5.9 万星是攒下来的历史人气，不是当下还在投入的信号。如今它的 GitHub README 已经白纸黑字写明进入维护模式，原文是：

> Maintenance Mode - AutoGen is now in maintenance mode. It will not receive new features or enhancements and is community managed going forward.

&emsp;&emsp;翻译过来就是：AutoGen 进入维护模式，不再接收新特性或增强，后续转为社区管理。Microsoft 同时提供了从 AutoGen 迁移到 Agent Framework 的官方指南，而 MAF 正是 AutoGen 与 Semantic Kernel 合并之后的接棒者。所以本课对 AutoGen 的处理是：把它当历史对照——后面每一章的框架对照表里会保留它对应的原语名字（比如 GraphFlow、SelectorGroupChat、Swarm），但<font color=red>代码统一用 MAF 来写</font>，不再单独跑 AutoGen 的代码。

&emsp;&emsp;说到"支持等级"，本课在后面四章会反复用到一个四档归类口径——<b>原生、官方示例级、可拼装、不适配</b>。需要先声明清楚：这是本课为了横向对照各框架对某种模式的支持程度自定的归类口径，便于学员一眼看出"这家是不是为这个模式专门设计的"，<b>并不是某一家官方的术语</b>。"原生"指框架有为该模式专门设计的一等公民 API，"官方示例级"指没有专用 API 但官方文档明确推荐某种写法，"可拼装"指要靠上层代码自己搭，"不适配"指框架的设计世界观根本支撑不了这种模式。

&emsp;&emsp;最后一条要留意的是版本节奏。这六个框架发版快慢差异极大——LangGraph、CrewAI、MAF 已经是稳定的 GA，而 OpenAI Agents SDK 还在 Pre-1.0 阶段、接近月度发版，接口仍在演进。装环境时务必锁定版本号，否则今天跑通的代码，过两周升级一个小版本就可能因为某个参数改名而报错。

### 1.5 智能体之间的通信

&emsp;&emsp;四种模式各自的样子就清楚了，全景地图还差最后一块：智能体之间到底靠什么传递信息？模式描述的是"谁指挥谁、活怎么分"的组织关系，可组织关系要运转起来，成员之间总得有传话的办法。剥开各家框架五花八门的 API，底下其实只有两种通信范式，下面这张图先把它们画出来。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L1.5-communication-56cdfeae.png" width=80%></div>

&emsp;&emsp;第一种是<b>共享内存</b>——智能体之间不直接对话，而是共同读写一块公共的状态数据，一方把结果写上去、另一方从上面读下来，像一群工程师围着同一块白板干活，所以它也叫黑板模式。下一章会见到的 LangGraph 的 State 就是典型：每个节点只管读它、写它，节点之间互不相识，信息全靠这块"白板"中转。

> <font size=2>【名词解释】<b><font color=red>Shared Memory</font>（Shared Memory，共享内存/黑板）</b>:智能体之间不直接互发消息，而是读写同一块公共状态来间接交换信息的通信范式，也称黑板模式（Blackboard）——信息像写在公共白板上，谁需要谁去看。</font>

&emsp;&emsp;第二种是<b>消息传递</b>——一个智能体把信息直接发给另一个具体的智能体，谁发给谁、发了什么，清清楚楚。它在多智能体里又有两个常用变体：<b>调用-返回</b>，把对方当一件工具来调、等它返回结果再继续往下走，主管派活给专家走的就是这条；<b>控制权接力</b>，不等返回，直接把整个对话的控制权连同上下文一起交出去、自己退场，Swarm 的 handoff 正是它。

> <font size=2>【名词解释】<b><font color=red>Message Passing</font>（Message Passing，消息传递）</b>:一个智能体把信息直接发给另一个具体智能体的通信范式，发送方和接收方都明确；按"等不等回信"分成调用-返回和控制权接力两个变体。</font>

&emsp;&emsp;这两种范式管的都是<b>同一个应用内部</b>的传话——共享一块状态也好、互发消息也好，前提是这些智能体生活在同一套代码、同一个框架里。可一旦跨出应用边界，比如我们公司的客服智能体要把任务转给另一家公司的物流智能体，两边框架不同、代码互不可见，靠什么传话？这一层业界已经收敛出了一个开放标准：<font color=red>A2A 协议</font>。它的机制用一句话就能说清：每个智能体对外发布一张"能力名片"（Agent Card），写明自己是谁、能干什么活；别的智能体读这张名片发现它的能力，再用统一的消息把任务派过去、取回产物——本质上是把"消息传递"这条范式从应用内部标准化到了互联网层面。它由 Google 发起、已捐给 Linux Foundation 中立托管，2026 年初发布 1.0 版进入生产可用，微软、亚马逊、IBM 等一百五十多家组织参与支持。这里有一条容易判断错的分界线要说清：要不要用 A2A，看的<b>不是框架是否相同，而是有没有跨过进程和组织的边界</b>。哪怕两个智能体分别用 CrewAI 和 LangGraph 写成，只要跑在同一个进程里、代码互相可见，普通的代码拼接就能传话，用不着协议；反过来，哪怕两边用的是同一个框架，只要各自跑在不同组织的服务器上、互相看不见代码，就得靠 A2A 这样的标准来握手。本课五个框架做的全是应用内协作，所以通篇用不到它。

> <font size=2>【名词解释】<b><font color=red>A2A</font>（Agent2Agent Protocol，智能体互通协议）</b>:让不同公司、不同框架构建的智能体能跨系统互相发现与协作的开放协议——智能体发布 Agent Card（能力名片）供对方发现，再以统一的消息派任务、取产物。Google 发起，现由 Linux Foundation 托管，2026 年发布 1.0 版。</font>

&emsp;&emsp;再往里看一层，A2A 的本质是一套<b>任务委托与结果交换</b>的规范：发起方把用户意图、必要的上下文和输入材料，按标准结构发给对方；对方在自己的框架和进程里执行，再把执行状态、过程消息和最终产物按标准格式还回来。比起在两个系统之间裸传一段提示词，它把工程上缺的环节都补齐了——身份与能力靠 Agent Card（能力名片）声明，一次委托是一个 <b>Task</b>（任务，带完整的生命周期状态），过程沟通走 <b>Message</b>（消息），最终交付物是 <b>Artifact</b>（产物，报告、数据文件都算），长任务还支持流式上报进度，跨组织调用自带认证授权。一次完整的委托大致是这样一来一回：

```text
Agent A → Agent B：帮我做一份销售分析（意图 + 上下文 + 输入材料）
Agent B → Agent A：收到，任务开始（Task 建立，状态 working）
Agent B → Agent A：正在查数据库……（流式进度 Message）
Agent B → Agent A：分析完成（状态 completed，交付 report.md / sales.csv / summary.json 三个 Artifact）
Agent A：取回产物，继续自己的流程，最后汇总给用户
```

&emsp;&emsp;能看出来，这套"派任务、报进度、交产物"的结构，跟本章主管派活给专家的调用-返回如出一辙——A2A 做的就是把这层互动从进程内的函数调用，标准化成跨系统的网络协议。等手上的系统真要跟外部智能体打交道时，沿着 A2A 这个名字去查官方规范即可。

&emsp;&emsp;拿这两种范式回看四种模式，对应关系一目了然：Workflow 的典型实现是把工序产出写进共享状态顺序传下去（也有框架直接把上一步产出当消息传给下一步）；Supervisor 和 Hierarchical 是典型的调用-返回——主管调专家、总监调组长，层层调用、层层返回；Swarm 则是控制权接力。第二到第五章每讲一个框架实现，我们都会点一句它用的是哪种通信方式——同一种协作模式，不同框架可能选不同的通信范式去实现，这是看懂框架差异的一把钥匙。

&emsp;&emsp;到这里，全景地图就铺完了：我们认清了单个智能体的能力上限和多智能体的代价收益，理顺了 Workflow 到 Agent 这条自主性谱系，拿到了四种协作模式的地图和六个框架的版图，也看清了智能体之间通信的两种范式、以及跨系统互通的 A2A 协议这个出口。从下一章开始，我们就沿着这张地图，一种模式一种模式地走进去，用其中仍在活跃维护的五个框架各写一段能跑的代码，把概念变成手边能运行的东西。

## <center>第二章 Workflow 流程编排</center>

&emsp;&emsp;四种协作模式里，流程编排是最贴近我们已有认知的一种。上一章那个被任务压住的单个智能体之所以撑不住，很大一部分原因是它一个人要顺次干完研究、写作、润色好几道活。最自然的解法，就是把这几道活拆开，一道工序配一个专门的智能体，前一个干完把成果交给后一个——这就是流程编排。

&emsp;&emsp;本章我们用一个内容生产的场景把它讲透：研究员先就一个主题列出核心要点，写作者据此写一段博客初稿，编辑再把初稿压成一句话导读。三道工序、三个智能体、固定顺序，正好对应第一章 1.2 节里 Anthropic 那五种工作流原语中的 Prompt Chaining。

<div align=center><font size=2 color=#999999>第二章：三个智能体在一条传送带上依次加工同一份内容</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L2-workflow-pipeline-490ba0a1.png" width=80%></div>

<br>

&emsp;&emsp;接下来七个小节这样走：先讲清流程编排的确定性机制和它的适用边界；然后五个主流框架各写一段能直接运行的最小流水线，每个框架小节都按"生态与支持程度、最小可跑代码、注意事项"三段展开；最后一节用一张对照表把五家在这种模式上的支持程度和手感差异并排放在一起。

### 2.1 流水线机制

&emsp;&emsp;流程编排的核心机制只有一句话：<font color=red>执行顺序固定在代码里</font>，模型只负责在每个工位上产出内容，不参与决定下一步去哪。这跟上一章讲的 Workflow 定义是同一回事——控制流由代码定，模型只填内容。具体到我们的内容生产场景，"研究员跑完轮到写作者、写作者跑完轮到编辑"这个顺序是开发者写在代码里的，模型从头到尾没有"我接下来该交给谁"的决策权。还有一条同样关键：上一道工序的输出，原样成为下一道工序的输入。研究员产出的要点，会被直接喂进写作者的提示词；写作者产出的初稿，又被喂进编辑的提示词。整条链路上的数据流向是单向的、确定的，不存在"绕回去重做"或"跳过某一步"的可能。这种确定性正是流水线的最大优点——结果可预测、出问题好定位，哪一道工序产出不对，一眼就能查到是哪个智能体。

&emsp;&emsp;它的适用前提也很清楚：工序能够被预先划分、且顺序稳定。内容生产、审批流转、数据处理、报表生成这类业务都符合——它们的步骤是固定的，不会因为输入不同就临时改流程。只要业务能用确定性流水线解决，就不必动用更复杂的自主编排。这条"能简单就别上复杂"的原则，是第一章那条升级阶梯的直接体现。

&emsp;&emsp;有两个注意事项要先记在心里。其一，串行链的延迟会随节点数线性叠加——三道工序就是三次模型调用的耗时相加，工位越多，端到端越慢，这是顺序结构换来确定性的固定代价。其二，劣化会向下游放大——如果研究员这一步产出的要点本身就跑偏了，写作者和编辑只会在错误的基础上继续加工，最后整条流水线的产出全错。所以流水线里上游工序的质量约束要比下游更严，越靠前的工位越要把好关。

&emsp;&emsp;机制讲清楚了，动手之前先把这一章五个框架都要用到的环境准备好。下面这个 cell 是全课唯一一次集中读取配置：打开 Jupyter 的顶层异步支持，再从环境变量里把模型名、接入地址、密钥读出来，后面五个框架的代码块直接复用这三个变量，不再重复定义。

In [1]:
# 让 Jupyter 支持在 cell 里直接 await 异步代码（OpenAI/MAF/Claude SDK 都是异步优先）
import nest_asyncio  # 让事件循环可重入，Jupyter cell 里能直接 await
nest_asyncio.apply()

import os
from dotenv import load_dotenv  # python-dotenv：从 .env 文件读配置，而不是手动 export 环境变量
load_dotenv()                    # 把课件目录下 .env 里的 OPENROUTER_API_KEY 加载进来

# 本章每个 agent 都配了真工具（搜索），而"自主调工具"(tool calling)对模型的要求比纯生成文本高得多，
# 所以这里用深度思考模型，而不是最小示例那种非思考的 flash。
MODEL = os.environ.get("MODEL", "deepseek/deepseek-v4-pro")   # LangGraph / CrewAI / OpenAI 三家用它，tool calling 稳
GLM = "z-ai/glm-5.1"                                          # MAF / Claude SDK 两家对 deepseek 的 tool calling 兼容差，换 GLM 5.1
BASE_URL = "https://openrouter.ai/api/v1"        # 走 OpenRouter 这个统一入口
KEY = os.environ["OPENROUTER_API_KEY"]           # 从 .env 读出来，绝不硬编码在代码里

# 给每个 agent 配的搜索工具底座——用 DuckDuckGo 真实联网搜，各框架再用自己的装饰器把它包成工具
from ddgs import DDGS
def web_search_raw(query: str) -> str:
    """联网搜索资料，返回前 3 条结果的标题和摘要。输入查询词。"""
    print(f"    [工具] web_search 被调用 → 查询：{query}")   # 打印一行，方便我们看到 agent 真在自主调搜索工具
    try:
        results = DDGS().text(query, max_results=3)
        return "\n".join(f"- {r['title']}：{r['body'][:90]}" for r in results) or "无结果"
    except Exception as e:
        return f"搜索失败：{e}"

&emsp;&emsp;这个 cell 完成了三件事：`nest_asyncio.apply()` 让后面 OpenAI、MAF、Claude 三家的异步代码能在 Jupyter cell 里直接 `await`，不用套 `asyncio.run()`；`load_dotenv()` 把课件目录下 `.env` 文件里的密钥读进来；`MODEL` 统一锁定 flash 模型、`KEY` 从 `.env` 读出。运行前请先把课件目录里的 `.env.example` 复制成 `.env`、填上自己的 OpenRouter 密钥——把密钥放进 `.env`（且 `.env` 不进版本库）是基本规范，既避免散落各处难维护，也杜绝密钥被硬编码进代码后随仓库泄漏。

### 2.2 LangGraph：StateGraph 串行边

&emsp;&emsp;LangGraph 是我们五个框架里第一个用到的，所以先花一点篇幅讲清它的全貌，后面几章再用到它时就不重复了。它由 LangChain 团队出品，在第一章 1.4 框架版图那张表里它是 1.2.4 的 GA 版本，是目前生产环境采用最广的图编排底座。它把多智能体系统建模成一张"图"——每个智能体或处理步骤是图上的一个节点，节点之间用"边"连接表示执行流向，整张图共享一份状态数据。对流程编排这种模式，它是<b>原生</b>支持：`StateGraph` 加上串行的边，本来就是为"确定性控制流"这件事设计的。

> <font size=2>【名词解释】<b><font color=red>StateGraph</font>（State Graph，状态图）</b>:LangGraph 的核心构造，把工作流建成一张图——节点是处理步骤、边是执行流向、所有节点读写同一份共享状态。</font>

&emsp;&emsp;它有三个核心概念要先认清。<b>State</b> 是流水线上传递的一张共享表，每个节点往里填自己那一格，本例里它有 topic、points、draft、final 四个字段，用 Python 的 `TypedDict` 声明。<b>节点</b>是一个普通函数，接收当前 State、返回自己要写回 State 的那部分。<b>边</b>用 `add_edge` 把节点按顺序连起来，`START` 和 `END` 是图的入口和出口。把这三者拼起来，就是一条控制流固定在边里的流水线。

&emsp;&emsp;下面这段代码用 LangGraph 把"研究员、写作者、编辑"三道工序建成一张顺序图，作用是把 2.1 节那条传送带变成能跑的程序——三道工序固定顺序、上一道的产出喂给下一道，模型只在每个工位上产内容、不决定流向。实现逻辑分三步：先用 `State` 定义一张贯穿全程的共享表，每道工序往里填自己那一格（要点、初稿、定稿）；再把三道工序各写成一个节点函数，每个函数里调一次模型、把产出写回 `State`；最后用三个 `add_edge` 把 `START → research → write → edit → END` 这条流向焊死、`compile()` 成可执行的图，`invoke` 一次就从头跑到尾，最后从 `State` 里取出三个字段。控制流全由这几条边定死，这正是 Workflow 的本质。通信方式上，这是第一章讲的<b>共享内存/黑板</b>的标准样本：三个节点从头到尾没有互相说过一句话，全靠读写同一份 `State` 交换信息。代码复用了上面环境准备 cell 里的 `MODEL`、`BASE_URL`、`KEY`。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L2.2-langgraph-stategraph-56979e9a.png" width=80%></div>

In [ ]:
from typing import TypedDict  # 给 LangGraph 的图状态定义带类型的字段结构
from langchain_openai import ChatOpenAI  # LangChain 封装的 OpenAI 兼容聊天模型
from langchain_core.tools import tool    # 把普通函数包成 LangChain 工具，挂给 agent
from langgraph.graph import StateGraph, START, END  # 建图核心：StateGraph 定状态图，START/END 是起止虚拟节点
from langgraph.prebuilt import create_react_agent   # 把"模型 + 工具"组装成能自主调工具的 agent

# 把 env-prep 里的搜索函数包成 LangChain 工具
@tool
def web_search(query: str) -> str:
    """联网搜索资料，返回前 3 条结果的标题和摘要。输入查询词。"""
    return web_search_raw(query)

llm = ChatOpenAI(model=MODEL, base_url=BASE_URL, api_key=KEY, temperature=0.3)

# 三道工序各是一个带搜索工具的真 agent：能自主决定要不要搜、搜什么——这才叫 agent
researcher = create_react_agent(llm, tools=[web_search],
    prompt="你是研究员。先用 web_search 搜索一两次主题资料，再提炼 3 个最该写进技术博客的要点，逗号分隔，只输出要点。")
writer = create_react_agent(llm, tools=[web_search],
    prompt="你是技术撰稿人。根据研究要点写一段 80 字以内的博客片段，只输出正文。")
editor = create_react_agent(llm, tools=[web_search],
    prompt="你是编辑。把博客片段精简润色成一句 25 字以内的导读，只输出这句话。")

# State 是流水线上传递的一张共享表，每个节点把对应 agent 的产出填进去
class State(TypedDict):
    topic: str       # 输入：写作主题
    points: str      # 研究员产出：研究要点
    draft: str       # 写作者产出：博客初稿
    final: str       # 编辑产出：精简定稿

def research(state: State):                              # 工序一：研究员 agent 搜资料、提要点
    print("【研究员】启动")
    r = researcher.invoke({"messages": [("user", f"主题：{state['topic']}")]})
    out = r["messages"][-1].content
    print(f"【研究员】产出：{out}\n")
    return {"points": out}

def write(state: State):                                 # 工序二：撰稿 agent 据要点写初稿
    print("【写作者】启动")
    r = writer.invoke({"messages": [("user", f"研究要点：{state['points']}")]})
    out = r["messages"][-1].content
    print(f"【写作者】产出：{out}\n")
    return {"draft": out}

def edit(state: State):                                  # 工序三：编辑 agent 精简定稿
    print("【编辑】启动")
    r = editor.invoke({"messages": [("user", f"博客片段：{state['draft']}")]})
    out = r["messages"][-1].content
    print(f"【编辑】产出：{out}\n")
    return {"final": out}

# 三道工序按顺序焊死在边里——这就是 Workflow：每个工位是带工具的 agent，但工位顺序由代码定死
g = StateGraph(State)
g.add_node("research", research)
g.add_node("write", write)
g.add_node("edit", edit)
g.add_edge(START, "research")
g.add_edge("research", "write")
g.add_edge("write", "edit")
g.add_edge("edit", END)
app = g.compile()  # 把状态图编译成可执行的图

result = app.invoke({"topic": "多智能体协作模式"})
print("【研究要点】", result["points"])
print("【博客初稿】", result["draft"])
print("【精简定稿】", result["final"])

&emsp;&emsp;真实运行的产出大致是这样：

```text
【研究员】启动
    [工具] web_search 被调用 → 查询：多智能体协作模式
    [工具] web_search 被调用 → 查询：multi-agent collaboration patterns 2026
    [工具] web_search 被调用 → 查询：多智能体协作 六种架构 协作形式
【研究员】产出：五种核心协作模式全景对比（编排者-工人、评估者-优化者、顺序流水线、并行、群组辩论）及各自触发条件，从"选模式"到"动态组合编织"的演化路径，多智能体的隐形成本（协调开销、token 膨胀、可观测性危机）

【写作者】启动
【写作者】产出：多智能体协作不是五选一，而是五合一。五大模式各有所适，但生产级系统的拐点在于动态组合编织——先并行检索、再编排汇总、最后评估打磨。更关键的盲区在隐形成本：token 膨胀、延迟叠加、调试黑箱。记住收纳原则：单 Agent 够用，就别加。

【编辑】启动
【编辑】产出：多智能体协作：五大模式动态组合，警惕成本，够用即止。
```

&emsp;&emsp;能清楚看到内容在三道工序里被逐步加工：研究员给出三个逗号分隔的要点，写作者把要点扩写成一段成型的博客，编辑再压成一句导读。三个字段的演进，正是 2.1 节那条"上游输出即下游输入"的流向落到了实处。

&emsp;&emsp;这里有一处注意事项，关系到 State 字段怎么写回。本例里每个节点写的是各自不同的字段——研究员写 `points`、写作者写 `draft`，互不重叠，直接赋值不会冲突。但如果改成多个节点往<b>同一个</b>列表字段里追加内容，直接赋值就会让后写的整段覆盖先写的，得用 `Annotated[list, operator.add]` 这类归约器（reducer）声明，告诉 LangGraph 这个字段是"追加合并"而不是"整段替换"。本例没踩到这个坑，是因为各节点各写各的字段，但这是用 LangGraph 建更复杂的图时最常见的陷阱之一，先记一笔。

### 2.3 CrewAI：Process.sequential

&emsp;&emsp;CrewAI 是一家独立公司出品的框架，社区规模到了五万星级别。它跟 LangGraph 走的是完全不同的设计路线——LangGraph 让我们建图连边，关心的是控制流；CrewAI 让我们定义"角色"，关心的是分工。它的世界观是把多智能体系统看成一支由不同角色组成的团队，每个角色有自己的职责、目标和背景设定。它有 Crews 和 Flows 两种模型，本节用的是 Crews。对流程编排，它同样是<b>原生</b>支持：`Process.sequential` 就是它的默认执行形态。

> <font size=2>【名词解释】<b><font color=red>Process</font>（Process，执行流程）</b>:CrewAI 里定义一支团队怎么协作的枚举，sequential 表示任务按列表顺序逐个执行、前一个产出自动进入下一个的上下文。</font>

&emsp;&emsp;它的招牌是用三件套定义一个智能体：<b>role</b>（这个角色是谁）、<b>goal</b>（它要达成什么）、<b>backstory</b>（它的背景设定，用来给模型定调）。任务用 `Task` 定义，每个 Task 绑定一个负责它的 Agent，并写明期望产出。最后用 `Crew` 把一组 Agent 和一组 Task 装在一起，指定 `Process.sequential`，整支团队就按任务列表的顺序跑起来。

&emsp;&emsp;下面这段代码用 CrewAI 实现同一条内容生产流水线，作用和上一节一样是把研究员、写作者、编辑三道工序串起来，但写法完全不同——看不到任何"连边"。实现逻辑分三步：先用 `role`/`goal`/`backstory` 三件套定义三个角色、每个角色都用 `tools=[web_search]` 配上同一个搜索工具（这样它们才是能自主查资料的真 agent，而不是只会接龙的纯文本生成），再用 `Task` 定义三个任务、每个任务绑定一个负责它的角色并写明期望产出，最后用 `Crew` 把这组角色和任务装在一起、声明 `Process.sequential`，`kickoff_async` 一启动（Jupyter 的事件循环里要用异步版、直接 `await`），整支团队就按任务列表顺序跑下来。"上一个 Task 的产出自动进下一个 Task 的上下文"这件衔接的活全由 `Process.sequential` 接管，我们只声明角色和任务、不写一行串接代码。通信方式上这是<b>消息传递</b>——前一个角色的产出作为消息进入下一个角色的上下文，只是传递这个动作由框架代劳，我们看不见。同样复用环境准备 cell 的三个变量。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L2.3-crewai-sequential-d8279fd4.png" width=80%></div>

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM  # CrewAI 五件套：Agent 角色 / Task 任务 / Crew 团队 / Process 编排方式 / LLM 模型
from crewai.tools import tool as crew_tool          # CrewAI 的工具装饰器，把普通函数注册成 agent 可调的工具

# 把 env-prep 里的搜索函数包成 CrewAI 工具
@crew_tool("web_search")
def web_search(query: str) -> str:
    """联网搜索资料，返回前 3 条结果的标题和摘要。输入查询词。"""
    return web_search_raw(query)

# litellm 走 OpenRouter 时模型名要带 openrouter/ 前缀，这是 CrewAI 这条链路的硬要求
llm = LLM(model=f"openrouter/{MODEL}", base_url=BASE_URL, api_key=KEY, temperature=0.3, timeout=180)
os.environ.setdefault("OPENAI_API_KEY", KEY)             # CrewAI 部分组件会读 OPENAI_API_KEY，兜底设一下

# 三个角色，每个都配上搜索工具——能自主决定要不要搜、搜什么，这才叫 agent
researcher = Agent(role="内容研究员", goal="就给定主题先联网搜索、再提炼最该写进技术博客的核心要点",
                   backstory="你擅长先查资料再下笔。", llm=llm, tools=[web_search], verbose=False)
writer = Agent(role="技术写作者", goal="把研究要点写成一段通俗的技术博客",
               backstory="你写的技术博客准确又好读。", llm=llm, tools=[web_search], verbose=False)
editor = Agent(role="文字编辑", goal="把一段博客精简成一句导读",
               backstory="你能把一整段讲解收成一句话。", llm=llm, tools=[web_search], verbose=False)

# Task.description 必须明确"先用 web_search 工具搜索"，否则 agent 可能跳过工具直接答
t_research = Task(description="先用 web_search 工具搜索一两次「{topic}」，再提炼 3 个最该写进技术博客的核心要点，逗号分隔。",
                  expected_output="3 个核心要点，逗号分隔。", agent=researcher)
t_write = Task(description="根据上一步的要点，写一段 80 字以内的技术博客片段。",
               expected_output="一段 80 字以内的博客片段。", agent=writer)
t_edit = Task(description="把上一步的博客精简成一句 25 字以内的导读。",
              expected_output="一句 25 字以内的导读。", agent=editor)

crew = Crew(agents=[researcher, writer, editor],  # 把一组 agent + task 编成一个团队
            tasks=[t_research, t_write, t_edit],
            process=Process.sequential, verbose=False)   # sequential = 流程编排

result = await crew.kickoff_async(inputs={"topic": "多智能体协作模式"})  # Jupyter 自带事件循环，须用异步版启动（过程中 web_search 被调用时会打印）
print("\n【各工序产出】")
for t in [t_research, t_write, t_edit]:
    print(f"【{t.agent.role}】产出：", t.output.raw if t.output else "(无)")
print("\n【最终导读】", result)

&emsp;&emsp;和 LangGraph 一对比，CrewAI 的取舍就清楚了：没有 State、没有节点函数、没有 `add_edge`，换来的是更快的原型速度——角色化写法很直观，三个角色定义清楚，流水线就成了。运行产出如下：

```text
    [工具] web_search 被调用 → 查询：多智能体协作模式
    [工具] web_search 被调用 → 查询：多智能体协作模式 架构 设计模式 核心要点

【各工序产出】
【内容研究员】产出： 六种协作架构选型, 专长角色的专业化分工, 从 ReAct 到 MCP 的通信协议栈
【技术写作者】产出： 打造可靠的多智能体系统，离不开三块基石：六种协作架构选型、专长角色专业化分工，以及 ReAct 到 MCP 的通信协议栈。
【文字编辑】产出： 多智能体系统三基石：六种协作架构、专长角色分工、ReAct 到 MCP 通信协议栈。

【最终导读】 多智能体系统三基石：六种协作架构、专长角色分工、ReAct 到 MCP 通信协议栈。
```

&emsp;&emsp;`kickoff_async` 的返回值就是最后一道工序的产出（编辑给的那句导读），中间各工序的产出可以从每个 Task 的 `output.raw` 单独取出。要注意的一点已经写在代码注释里：litellm 这条接入链路走 OpenRouter 时，模型名必须带 `openrouter/` 前缀，少了前缀会因为找不到模型提供方而报错——这是 CrewAI 接入第三方端点最容易卡住的地方。

### 2.4 OpenAI Agents SDK：代码驱动编排

&emsp;&emsp;OpenAI Agents SDK 由 OpenAI 出品，是它早年那个 Swarm 实验库的生产化接棒者（这一点我们在第一章 1.3 节区分"库和模式"时讲过）。它的设计哲学是极简——整个 SDK 只围绕四个原语：Agent、Runner、Tools、Handoffs。它目前还在 Pre-1.0 阶段，接近月度发版。对流程编排，它的支持是<b>官方示例级</b>——它没有提供流水线专用的 API，但官方文档明确推荐"代码驱动编排"作为确定性场景的首选姿势：我们自己用普通的 Python 代码，把上一个智能体的产出喂给下一个。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L2.4-openai-code-orchestration-25c30101.png" width=80%></div>

> <font size=2>【名词解释】<b><font color=red>Runner</font>（Runner，运行器）</b>:OpenAI Agents SDK 里负责实际驱动一个 Agent 跑起来的执行器，Runner.run 输入一个 Agent 和它的输入、返回运行结果。</font>

&emsp;&emsp;这里要解释一下"官方示例级"这个口径在 OpenAI SDK 上的具体含义。前面两家（LangGraph、CrewAI）都有专门为流水线设计的构造——一个让我们连边、一个让我们用 `Process.sequential`。OpenAI SDK 没有这种东西，它把"怎么串"这件事完全交还给开发者：我们用 `Runner.run` 跑第一个 Agent，拿到它的 `final_output`，再当作输入喂给第二个 Agent 的 `Runner.run`，如此接力。这正是"能做，但要自己串"的意思。

&emsp;&emsp;下面这段代码用 OpenAI SDK 实现内容生产流水线，作用还是那三道工序，但它的"串接"是五家里最肉眼可见的。实现逻辑分两步：先把三道工序各定义成一个 `Agent`（各带一句 `instructions`、各用 `tools=[web_search]` 挂上搜索工具、彼此之间没有任何连线），再在 `main` 里手写三行 `Runner.run` 把它们接力起来——跑完第一个 Agent 拿到 `r1.final_output`，当输入喂给第二个 Agent 的 `Runner.run`，再接力到第三个。真正把三个独立 Agent 串成流水线的就是这三行手写代码，没有任何框架在背后帮忙，这正是 OpenAI SDK 极简哲学的体现。通信方式同样是<b>消息传递</b>，而且是手动版——`r1.final_output` 喂给下一个 `Runner.run`，那一行就是消息本身。注意这段是异步代码，按本课 Jupyter 约定直接在 cell 里 `await`，不套 `asyncio.run()`。

In [ ]:
from openai import AsyncOpenAI  # OpenAI 官方异步客户端，指向 OpenRouter 兼容端点
from agents import Agent, Runner, OpenAIChatCompletionsModel, function_tool, set_tracing_disabled  # 多导入 function_tool：把普通函数包成 agent 可调的工具

set_tracing_disabled(True)                               # 关掉默认连 OpenAI 的 tracing，走第三方端点必须
client = AsyncOpenAI(base_url=BASE_URL, api_key=KEY)     # 复用环境准备 cell 的 BASE_URL/KEY
model = OpenAIChatCompletionsModel(model=MODEL, openai_client=client)

# 把 env-prep 里的搜索函数包成 OpenAI SDK 工具
@function_tool
def web_search(query: str) -> str:
    """联网搜索资料，返回前 3 条结果的标题和摘要。输入查询词。"""
    return web_search_raw(query)

# 三个 Agent 各管一道工序，每个都挂上搜索工具；它们之间没有 handoff，纯靠下面的代码把输出接力下去
researcher = Agent(name="researcher", model=model, tools=[web_search],
                   instructions="先用 web_search 搜索一两次用户给的主题，再提炼 3 个最该写进技术博客的核心要点，逗号分隔，不要解释。")
writer = Agent(name="writer", model=model, tools=[web_search],
               instructions="根据用户给的研究要点，写一段 80 字以内的技术博客片段。")
editor = Agent(name="editor", model=model, tools=[web_search],
               instructions="把用户给的博客片段精简成一句 25 字以内的导读。")

async def main():
    topic = "多智能体协作模式"
    print("【研究员】启动")
    r1 = await Runner.run(researcher, topic)             # 工序一：研究
    print(f"【研究员】产出：{r1.final_output}\n")
    print("【写作者】启动")
    r2 = await Runner.run(writer, r1.final_output)       # 上一步产出当这一步输入
    print(f"【写作者】产出：{r2.final_output}\n")
    print("【编辑】启动")
    r3 = await Runner.run(editor, r2.final_output)       # 再接力一棒
    print(f"【编辑】产出：{r3.final_output}\n")

await main()       # Jupyter 里直接 await，不用 asyncio.run()

&emsp;&emsp;运行产出如下：

```text
【研究员】启动
    [工具] web_search 被调用 → 查询：多智能体协作模式
    [工具] web_search 被调用 → 查询：multi-agent collaboration PEV A2A protocol
【研究员】产出：选对拓扑（顺序流水线、层级委派、扁平辩论、混合编排四类模式各有场景），搭好闭环（PEV 三层架构把 Mind/Hand/Verifier 解耦、每步执行后强制验证），跑通通信（A2A 协议做能力发现、消息收发与会话持久化）

【写作者】启动
【写作者】产出：多智能体协作落地，关键在于选对拓扑、搭好闭环、跑通通信。四类模式——顺序流水线、层级委派、扁平辩论、混合编排，各有场景匹配；PEV 三层架构将 Mind/Hand/Verifier 彻底解耦，每一步执行后强制验证闭环；A2A 协议通过 Agent Card 能力发现、JSON-RPC 消息收发与会话状态持久化，再辅以指数退避重试，让智能体间的协作容错与可观测性真正可行。

【编辑】启动
【编辑】产出：多智能体协作落地关键：选对拓扑、搭好闭环、跑通通信。
```

&emsp;&emsp;有两处注意事项。一是 `set_tracing_disabled(True)`——OpenAI SDK 默认会把运行轨迹回传到 OpenAI 的服务，我们走的是 OpenRouter 第三方端点，不关掉这个追踪会因为凭证不匹配而报错，所以走第三方端点时这一行是必须的。二是异步写法——这个 SDK 是异步优先的，在 Jupyter 里我们靠环境准备 cell 里的 `nest_asyncio` 直接 `await`，千万别再套一层 `asyncio.run()`，否则会报"事件循环已在运行"的错。

### 2.5 MAF：SequentialBuilder 顺序编排

&emsp;&emsp;Microsoft Agent Framework（本课简称 MAF）由 Microsoft 出品，是 AutoGen 和 Semantic Kernel 两个框架合并之后的接棒者——第一章 1.4 节讲 AutoGen 进入维护模式时提到的那个接棒者就是它。它于 2026 年 4 月 3 日发布 1.0 GA 这个稳定里程碑，本课实测用的是 agent-framework-core 1.8.x 小版本，有 Python 和 .NET 双实现，提供五种官方编排模式，是企业级编排视角下最完整的一家。对流程编排，MAF 是<b>原生</b>支持：它在 `agent_framework.orchestrations` 子包里提供了 <font color=red>SequentialBuilder</font>，一行就能把一组智能体串成顺序流水线。这一节我们就用它。还有一点要先交代：从这家起，模型统一换成 GLM 5.1——deepseek 经 MAF 这条链路调工具时不太稳，换成 GLM 5.1 实测可用。

&emsp;&emsp;下面这段代码用 `SequentialBuilder` 实现内容生产流水线。实现逻辑分三步：先用 `client.as_agent()` 把三道工序各包成一个带搜索工具的智能体（MAF 的 `as_agent` 能直接收一个普通 Python 函数当工具）；再用 `SequentialBuilder(participants=[...]).build()` 把三个智能体按顺序串成一条工作流；最后 `await wf.run()` 一次跑完，从结果里把每个 agent 的发言依次取出来打印。和前两家的手动接力不同，这里"上一步产出喂给下一步"由 `SequentialBuilder` 自动接管，我们只声明参与者的先后顺序。通信方式上它走的是<b>共享内存/黑板</b>——每个 agent 把发言追加到同一条共享对话里，下一个接着往下读，和第二章 LangGraph 的 State 是同一类。这段是异步代码，直接 `await`。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L2.5-maf-sequential-0bbf8b68.png" width=80%></div>

In [ ]:
from agent_framework.orchestrations import SequentialBuilder  # MAF 顺序编排构造器：把一组 agent 串成流水线
from agent_framework.openai import OpenAIChatClient            # MAF 聊天客户端，as_agent 把它变成一个 agent

# 搜索函数直接当工具传给 as_agent（MAF 能自动把普通函数转成工具 schema）
def web_search(query: str) -> str:
    """联网搜索资料，返回前 3 条结果的标题和摘要。输入查询词。"""
    return web_search_raw(query)

# 这一节起模型换成 GLM 5.1（deepseek 经 MAF 调工具不稳）
client = OpenAIChatClient(model=GLM, api_key=KEY, base_url=BASE_URL)

# 三个带搜索工具的 agent；as_agent 是 MAF 最稳的智能体单元
researcher = client.as_agent(name="researcher", tools=[web_search],
    instructions="你是研究员。先用 web_search 最多搜两次，再提炼 3 个最该写进技术博客的要点，逗号分隔，只输出要点。")
writer = client.as_agent(name="writer", tools=[web_search],
    instructions="你是技术撰稿人。根据上文研究要点写一段 80 字以内的博客片段，只输出正文。")
editor = client.as_agent(name="editor", tools=[web_search],
    instructions="你是编辑。把上文博客片段精简成一句 25 字以内的导读，只输出这句话。")

# SequentialBuilder 把三个 agent 按顺序串成一条流水线
wf = SequentialBuilder(participants=[researcher, writer, editor]).build()

async def main():
    result = await wf.run("主题：多智能体协作模式")   # 一次跑完整条流水线（过程中 web_search 被调用会打印）
    # SequentialBuilder 把三个 agent 的发言按先后累积在 result 里，依次取出打印
    roles = ["研究员", "写作者", "编辑"]
    speeches, seen = [], set()
    for ev in result:                                  # 遍历工作流事件
        data = getattr(ev, "data", None)
        msgs = data if isinstance(data, list) else [data]
        for m in msgs:
            t = (getattr(m, "text", "") or "").strip()
            if t and not t.startswith("主题：") and t not in seen:
                seen.add(t); speeches.append(t)
    for role, text in zip(roles, speeches):
        print(f"【{role}】产出：{text}")

await main()       # Jupyter 里直接 await

&emsp;&emsp;运行产出如下（每个 agent 的发言从 `SequentialBuilder` 的结果里按顺序取出）：

```text
    [工具] web_search 被调用 → 查询：多智能体协作模式
    [工具] web_search 被调用 → 查询：multi-agent collaboration patterns 2026
【研究员】产出：编排拆分、团队并行、共享状态三大协作模式, 分别适配任务分解、独立推进与协同研发, 选型取决于子任务耦合度
【写作者】产出：多智能体协作三大核心模式：编排拆分、团队并行、共享状态，分别适配任务分解、独立推进与协同研发，让 AI 从单打独斗走向高效协作。
【编辑】产出：三大协作模式让多智能体从单打独斗走向高效协同
```

&emsp;&emsp;`SequentialBuilder` 的好处是把"接力"这件事收进了框架，我们只声明参与者顺序、不写一行串接代码，比前两家手动 `Runner.run`、`step` 一棒棒传更省事。代价是它把过程藏进了内部那条共享对话，想看清每个 agent 各自说了什么，得像上面那样从结果里把发言一条条捞出来——这正是高层编排普遍的取舍：省了串接代码，却也少了一点过程的透明度。

### 2.6 Claude Agent SDK：自定义工具串行

&emsp;&emsp;Claude Agent SDK 由 Anthropic 出品，它的定位和前四家不太一样——它本质是 Claude Code 命令行工具的可编程封装，自带文件读写、Bash 执行、检索这一类工具，定位更偏向编码智能体的场景。在第一章 1.4 表里它是 0.2.95 版本，发版极其频繁。对流程编排，它的支持是<b>官方示例级</b>——它同样没有流水线专用 API，靠的是在代码层多次调用 `query()` 串行接力。

> <font size=2>【名词解释】<b><font color=red>query</font>（query，查询调用）</b>:Claude Agent SDK 的核心入口函数，发起一次与 Claude 的交互，返回一个异步生成器，需要用 async for 逐条消费它产出的消息块。</font>

&emsp;&emsp;先把它的运行条件交代清楚。安装上很省事：`pip install claude-agent-sdk` 时会自动捆绑一份 Claude Code 命令行程序，不需要再单独安装任何东西。模型凭证上它有四条路可选：配一个 Anthropic 的 API 密钥、用本机已登录的 Claude Code、走 AWS Bedrock 或 Google Vertex 这类云厂商，以及把请求指向任何 Anthropic 兼容端点的第三方网关。本课走的是最后一条——把 `ANTHROPIC_BASE_URL`、`ANTHROPIC_AUTH_TOKEN`、`ANTHROPIC_MODEL` 三个环境变量指向 OpenRouter，模型就换成了 GLM 5.1，和前四家从同一个 OpenRouter 取模型，不需要任何 Anthropic 凭证。之所以这家选 GLM 5.1 而不是其余几家用的 deepseek，是因为 deepseek 经这条链路调工具时不太稳，GLM 5.1 实测干净可用。

&emsp;&emsp;下面这段代码用 Claude Agent SDK 实现内容生产流水线，比前几家多做两件事：给 agent 配上真正的搜索工具，以及把模型换成 GLM 5.1。实现逻辑分三层：先用 `@tool` 把搜索函数注册成一个进程内工具、再用 `create_sdk_mcp_server` 打包成一个 MCP server 挂给 agent，这是 Claude SDK 给 agent 配自定义工具的标准姿势；然后写一个 `step` 函数封装"发一次 query、收集文本回复"——因为 `query()` 返回的是异步生成器，得用 `async for` 逐条取消息、只挑出助手回复里的文本块拼起来；最后在 `main` 里调三次 `step` 串行接力，每道工序就是一次 `query()`，靠 f-string 把上一步产出拼进下一步提示词。研究员那一步会真的调起搜索工具，运行时能看到工具被调用的打印。通信方式还是手动的<b>消息传递</b>，载体就是提示词本身。还有个细节：Claude Code 启动时会把所在目录的项目文件读进上下文，所以代码里用 `tempfile.mkdtemp()` 建了个干净空目录来跑，免得它读到无关内容把主题带偏。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L2.6-claude-query-e5f4762e.png" width=80%></div>

In [2]:
import tempfile  # 建一个干净空目录跑，避免 Claude Code 读到当前项目文件、污染对话上下文
from claude_agent_sdk import (query, ClaudeAgentOptions, AssistantMessage, TextBlock,
                              tool, create_sdk_mcp_server)  # query 发起对话 / tool+create_sdk_mcp_server 自定义进程内工具 / 消息类型解析回复

# 用 @tool 把搜索函数注册成 Claude SDK 的进程内工具：工具体须是 async、返回 MCP 的 content 结构
@tool("web_search", "联网搜索资料，返回前 3 条结果的标题和摘要", {"query": str})
async def web_search(args):
    text = web_search_raw(args["query"])
    return {"content": [{"type": "text", "text": text}]}

# 把工具打包成一个进程内 MCP server，挂给 agent 用（不需要起外部进程）
search_server = create_sdk_mcp_server(name="search", version="1.0.0", tools=[web_search])

# 让 Claude SDK 从 OpenRouter 取 GLM 5.1 模型（和前四家用的是同一个 OpenRouter）
CLAUDE_ENV = {"ANTHROPIC_BASE_URL": "https://openrouter.ai/api",
              "ANTHROPIC_AUTH_TOKEN": KEY,
              "ANTHROPIC_MODEL": GLM}
CLEAN_DIR = tempfile.mkdtemp()   # 临时干净目录，隔离项目上下文

def make_opts():                 # 每道工序共用一套配置：挂搜索工具 + 路由到 OpenRouter + 干净目录
    return ClaudeAgentOptions(
        model=GLM, env=CLAUDE_ENV, cwd=CLEAN_DIR,
        permission_mode="bypassPermissions",          # 跳过工具授权交互
        mcp_servers={"search": search_server},        # 挂载上面的进程内 MCP server
        allowed_tools=["mcp__search__web_search"],    # 白名单：允许 agent 调这个搜索工具
        max_turns=8)

async def step(role: str, prompt: str) -> str:        # 一道工序 = 一次 query，收集回复文本
    print(f"【{role}】启动")
    text = ""
    async for msg in query(prompt=prompt, options=make_opts()):  # query 异步流式返回消息块
        if isinstance(msg, AssistantMessage):            # 只取助手回复里的文本块
            for block in msg.content:
                if isinstance(block, TextBlock):
                    text += block.text
    text = text.strip()
    print(f"【{role}】产出：{text}\n")
    return text

async def main():
    topic = "多智能体协作模式"
    points = await step("研究员", f"请先用 web_search 工具搜索「{topic}」，再从结果里提炼 3 个最该写进技术博客的核心要点，逗号分隔，不要解释。")
    draft = await step("写作者", f"根据这些要点写一段 80 字以内的技术博客片段，只输出正文：{points}")   # 上一步产出拼进这一步 prompt
    final = await step("编辑", f"把这段博客精简成一句 25 字以内的导读，只输出这句话：{draft}")

await main()       # 用别家模型 GLM 5.1 经 OpenRouter，无需 Anthropic 凭证；Jupyter 里直接 await

【研究员】启动
    [工具] web_search 被调用 → 查询：多智能体协作模式
【研究员】产出：编排架构选型（顺序/并行/层级模式适用场景），专长型智能体分工与任务拆解机制，跨智能体通信与状态同步策略

【写作者】启动
【写作者】产出：顺序编排适合流水线依赖，并行模式加速独立子任务，层级模式处理递归分解；专长型智能体按能力边界分工，任务拆解至最小可分配单元；跨智能体通过共享状态存储与事件总线实现异步通信与一致性同步。

【编辑】启动
【编辑】产出：多模式编排按依赖拆任务，专长型智能体按能力分工，跨智能体以共享状态与事件总线异步通信。



&emsp;&emsp;运行产出如下：

```text
【研究员】启动
    [工具] web_search 被调用 → 查询：多智能体协作模式
【研究员】产出：分工协同突破单一智能体能力局限, 六种协作架构与六种协作形式, 多智能体在同一项目中并行协作

【写作者】启动
【写作者】产出：单一智能体再强也有天花板，多智能体协作让 AI 团队各展所长——六种架构搭配六种协作形式，分工协同突破单兵瓶颈，同一项目并行推进，1+1 远大于 2。

【编辑】启动
【编辑】产出：多智能体协作：六架构六形式，破单兵瓶颈，1+1>2
```

&emsp;&emsp;有两处注意。其一是消费方式——`query()` 返回的是异步生成器，必须用 `async for` 逐条取消息，不能像普通函数那样直接拿返回值。其二是自定义工具的格式——工具体必须是 `async` 函数、且返回 `{"content": [...]}` 这种结构，直接返回字符串会被框架判为格式错误。另外实测时会发现，模型回复有时偏啰嗦，会在主答案后附上几句说明，引用它的产出时取主句即可。

### 2.7 支持程度与生态对照

&emsp;&emsp;五个框架各写完一段流水线，现在把它们在流程编排这种模式上的支持程度和原语并排放在一张表里看。这里沿用第一章 1.4 节说过的那套本课内部归类口径——原生、官方示例级、可拼装、不适配，再强调一次，这是本课为横向对照自定的口径，不是某一家的官方术语。

<div align=center><font size=2 color=#999999>五个框架对 Workflow 流程编排的支持程度与生态对照</font></div>

<div align="center">
<table width="80%">
<thead><tr>
<th>框架</th><th>出品方</th><th>本模式支持等级</th><th>模式专用原语</th><th>一句话点评</th>
</tr></thead>
<tbody>
<tr><td>LangGraph</td><td>LangChain</td><td>原生</td><td>StateGraph + add_edge 串行边</td><td>控制粒度最细，能 per-node 控状态</td></tr>
<tr><td>CrewAI</td><td>CrewAI 公司</td><td>原生</td><td>Process.sequential</td><td>角色化最直观，原型最快</td></tr>
<tr><td>MAF</td><td>Microsoft</td><td>原生</td><td>SequentialBuilder</td><td>一等公民构造器，一行串成顺序流水线</td></tr>
<tr><td>OpenAI Agents SDK</td><td>OpenAI</td><td>官方示例级</td><td>Runner.run 代码驱动编排</td><td>无专用 API，手动接力，极简可控</td></tr>
<tr><td>Claude Agent SDK</td><td>Anthropic</td><td>官方示例级</td><td>多次 query() 串行</td><td>无专用 API，朴素拼接，可经 OpenRouter 换模型</td></tr>
<tr><td>AutoGen</td><td>Microsoft</td><td>历史对照</td><td>GraphFlow</td><td>维护模式，本课用 MAF 实现</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;从这张表能看出一个总的判断：流程编排是四种模式里门槛最低的，五家无论原生还是官方示例级都能干净地实现，差异不在"能不能做"，而在"控制粒度"。LangGraph、CrewAI、MAF 三家是<font color=red>原生</font>支持，各有侧重——LangGraph 给的是最细的 per-node 控制，能在每个节点单独检查状态、控制 token、加条件路由；CrewAI 给的是最快的角色化原型；MAF 给的是一行就能建链的一等公民构造器 SequentialBuilder。OpenAI SDK 和 Claude SDK 是<b>官方示例级</b>，没有专用 API，靠手写代码把上游产出接力给下游，胜在极简和完全可控。

&emsp;&emsp;落到选型上，结论也清楚。如果只是要一条线性的内容流水线、追求最快出原型，CrewAI 的角色化写法是最短路径；如果这条流水线需要在每个节点做状态检查、token 控制、条件路由或断点续跑这类精细控制，就换 LangGraph。这也和业界共识一致：CrewAI 适合角色驱动的线性内容工作流，LangGraph 适合生产级、需要可观测的复杂流水线。流程编排最大的优点是可预测、稳定，代价则是灵活性差、串行延迟会线性叠加——这两条 2.1 节已经讲过，到这里正好闭合。

&emsp;&emsp;本章我们把流程编排从机制讲到了五家的代码实现：它的本质是控制流固定、模型只产内容、上游输出即下游输入，适合工序固定的业务，代价是延迟叠加和劣化向下游放大。五个框架我们各跑通了一段最小流水线，也摸清了它们的手感差异。从下一章开始，我们松开一点对控制流的"固定"——把"派谁去做什么"的决策权交给一个中央协调者，进入 Supervisor 集中调度模式。

## <center>第三章 Supervisor 集中调度</center>

&emsp;&emsp;流程编排有一个绕不开的硬限制：它要求工序能预先划分、顺序还稳定。可现实里有大量任务并不长这样——比如要写一份某主题的研究简报，技术维度该问谁、应用维度该问谁、问完了怎么综合，这条路径没法在写代码时就定死，得看具体问题临时拿主意。这一类"派谁去做什么得当场决策"的任务，正是 Supervisor 集中调度要解决的。

&emsp;&emsp;Supervisor 模式的做法，是把第二章里"固定在代码里"的那个派单决策权，交还给模型。我们设一个中央协调者（习惯上叫主管，Supervisor），它手下挂着一组各有专长的子智能体；任务来了，由主管看着全局决定派谁、派什么、什么时候收口综合，子智能体只管干好自己被派到的那份活。这是从确定性控制流向模型自主决策迈出的第一步，也是第一章那条升级阶梯上紧挨着 Workflow 的下一级。下面这张图把这种中央派单、并行回流的星形结构画了出来。

<div align=center><font size=2 color=#999999>第三章：主管居中派单给各专家，专家各自作答后结果回流主管综合</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L3-supervisor-star-03761bee.png" width=80%></div>

<br>

&emsp;&emsp;接下来这一章和第二章同构地走：先讲清 Supervisor 的中央派单机制和它的两种控制权形态；然后五个主流框架各写一段能直接运行的最小 Supervisor，每个框架小节仍按"生态与支持程度、最小可跑代码、注意事项"三段展开；最后一节用一张对照表把五家在这种模式上的支持程度并排放在一起。和上一章不同的是，这一章里我们会第一次看到模型自己拿派单的主意，也会看到当模型这个主意不靠谱时，怎么用确定性兜底把它拽回来。

### 3.1 集中调度机制

&emsp;&emsp;Supervisor 的核心机制可以浓缩成一句话：一个中央协调者看着全局，决定把任务的哪一部分派给哪个子智能体，等结果收齐后再综合成最终产出。和流程编排最根本的区别就在这个"决定"上——流水线里"下一步去哪"是代码预先定死的，Supervisor 里"这一步派给谁"是主管这个智能体临场判断的。所以同一个研究简报任务，问的主题不同，主管派单的路径就可能不同，这正是它能应对"路径不能预先定死"那类任务的原因。这个结构还有一条容易被忽略但很关键的特征：<font color=red>子智能体之间互不可见</font>。在我们的研究简报例子里，技术专家不知道有个应用专家存在，应用专家也不知道技术专家说了什么，它们各自只跟主管单线通信，彼此之间没有任何连接——这正是上面那张图里两个专家之间故意不画连线的原因。所有的信息汇聚和分发都经由主管这一个中心点完成。这种"中心辐射"的结构让协作变得干净可控，主管始终掌握全局，但也埋下了这个模式最典型的隐患，待会儿在注意事项里会专门讲。

> <font size=2>【名词解释】<b><font color=red>控制权</font>（对应英文 control flow）</b>:指此刻由哪个智能体在执行、下一步轮到谁接手的主导权。谁握着控制权，谁就决定接下来做什么、要不要把它交给别人。四种协作模式的根本区别正在于控制权怎么流动——固定在代码里（Workflow）、收在一个主管手里（Supervisor 与 Hierarchical），或在平级智能体之间直接传递（Swarm）。</font>

&emsp;&emsp;还有一点必须先讲透，它会贯穿后面五个框架——主管把活派给子智能体，有两种截然不同的形态，区别在于<font color=red>控制权转不转移</font>。第一种形态，子智能体被当成图上的一个节点，主管把控制权<b>转移</b>给它：主管说"接下来交给技术专家"，控制流就真的跳到技术专家那里，技术专家在自己的上下文里跑，跑完再把控制权交回主管。这种形态下，"派单"本质上是一次控制权的让渡。

&emsp;&emsp;第二种形态，子智能体被当成主管手里的一件<b>工具</b>，控制权<b>始终不转移</b>：主管像调用一个普通函数那样"调用"技术专家这个工具，技术专家在内部跑完，把结果当作工具的返回值交回来，主管拿到结果后继续自己的推理——从头到尾，控制权一直攥在主管手里，子智能体只是被它调用了一下而已。这种形态有个专门的叫法，叫 agents-as-tools（把智能体当工具），我们在 3.3 节讲 OpenAI SDK 时会把它的代码完整展开。

&emsp;&emsp;这两种形态的区分非常重要，因为它正好是 Supervisor 模式和后面第五章 Swarm 模式的分界线：Supervisor 里主管始终是中心、控制权要么短暂让渡后必定收回（第一种）、要么压根不让渡（第二种），主管对全局始终有掌控；而 Swarm 里智能体之间是把控制权<b>真正交出去且不收回</b>的自主协作——那个真正的控制权移交动作叫 handoff，我们留到第五章再展开。现在只需记住：本章看到的所有"派单"，无论哪种形态，主管最终都拿得回全局；这是 Supervisor 区别于 Swarm 的根本。机制讲清了，下面进入五个框架的实现。

&emsp;&emsp;这一章五个框架的代码，仍然统一复用第二章 2.1 节那个环境准备 cell 里读出来的 `MODEL`、`BASE_URL`、`KEY` 三个变量，不再重复定义。统一的演示场景是研究简报：就"多智能体协作模式在 2026 年的现状"这个问题，主管把技术维度派给技术专家、应用维度派给应用专家，收齐两份后综合成一段研究简报——这正是 Supervisor 最典型的适用业务，也呼应了本课第三部分会做的多轮研究编排。

### 3.2 LangGraph：create_supervisor

&emsp;&emsp;第二章我们已经认识了 LangGraph 的图编排底座，这一节它换上专门为 Supervisor 准备的高层构造。对集中调度这种模式，LangGraph 是<b>原生</b>支持：它的 `langgraph-supervisor` 扩展提供了 `create_supervisor`，一个函数就能把"主管 + 一组专家"装配成上面讲的星形结构，省去我们手动连边、手动写派单工具的活。

> <font size=2>【名词解释】<b><font color=red>create_supervisor</font>（create supervisor，创建主管）</b>:LangGraph 用来一键装配 Supervisor 结构的函数，传入一组子智能体和一个主管模型，它会自动给主管注入 transfer_to_X 形式的派单工具，让主管能把任务转交给指定专家。</font>

&emsp;&emsp;这里的子智能体（worker）用另一个构造函数 `create_react_agent` 创建——它能造出一个会自己决定要不要调工具、能多轮推理的 ReAct 智能体。我们的两个专家不挂任何外部工具，纯靠模型知识各管一个维度即可。

> <font size=2>【名词解释】<b><font color=red>create_react_agent</font>（create ReAct agent，创建 ReAct 智能体）</b>:LangGraph 用来造一个能自主推理、按需调工具的智能体的函数，本节用它造两个专家 worker。在本课锁定的 1.2.4 版本里，它从 `langgraph.prebuilt` 导入即可；LangChain 已规划在后续版本里把这个能力迁移到 `langchain.agents.create_agent`，但那需要额外安装 langchain 主包，本课环境里 `prebuilt` 这个写法仍是可用且推荐的，照用即可。</font>

&emsp;&emsp;下面这段代码用 LangGraph 把研究简报场景搭起来，作用就是落地 3.1 节那张星形图：两个专家各管一个维度，主管居中派单并综合。实现逻辑分三步：先用 `create_react_agent` 造出技术、应用两个专家（不挂外部工具，纯靠模型知识各管一摊）；再用 `create_supervisor` 把两个专家和主管模型装配起来——它会自动给主管装上 `transfer_to_tech_expert`、`transfer_to_apply_expert` 两件派单工具；最后 `invoke` 一次跑完整协作。和第二章流水线最大的不同是，这里没有任何 `add_edge` 把执行顺序焊死——先派谁、要不要两个都派，全由主管在运行时临场决定，所以代码末尾我们遍历整条消息轨迹打印，就是为了看清这个临场决策的全过程。通信方式上，它底层用的是<b>控制权交接</b>（`transfer` 工具把控制权移给专家、专家干完再交回），但一来一回的整体语义仍是<b>调用-返回</b>：派出去、干完活、回到主管。代码复用第二章环境准备 cell 里的三个变量。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L3.2-langgraph-supervisor-b2209655.png" width=80%></div>

In [ ]:
import os  # 读环境变量（密钥、模型名）
from langchain_openai import ChatOpenAI  # LangChain 封装的 OpenAI 兼容聊天模型，LangGraph 节点里用它调 LLM
from langgraph.prebuilt import create_react_agent          # 本课锁定 1.2.4，从 prebuilt 导入即可，当前版本可用
from langgraph_supervisor import create_supervisor  # LangGraph 官方扩展：一键装配"主管 + 一组专家"的集中调度结构

# 复用第二章环境准备 cell 的 MODEL/BASE_URL/KEY
model = ChatOpenAI(model=MODEL, base_url=BASE_URL, api_key=KEY, temperature=0.3, timeout=180, max_retries=1)

# 两个专家 agent（无外部工具，凭模型知识各管一个维度）；name 是主管派单时的寻址依据
tech_expert = create_react_agent(model, tools=[], name="tech_expert",  # 造一个会自主推理、按需调工具的 ReAct 智能体
    prompt="你是技术专家。只从技术原理和架构角度回答问题，80 字以内。")
apply_expert = create_react_agent(model, tools=[], name="apply_expert",
    prompt="你是应用专家。只从落地场景和典型案例角度回答问题，80 字以内。")

# 主管：create_supervisor 自动给它注入 transfer_to_tech_expert / transfer_to_apply_expert 派单工具
supervisor = create_supervisor(  # 一键装配"主管 + 一组专家"的星形集中调度
    agents=[tech_expert, apply_expert],
    model=model,
    prompt=("你是研究主管。把用户问题的技术维度交给 tech_expert，应用维度交给 apply_expert，"
            "两份都收齐后，综合成一段 150 字以内的研究简报。")
).compile()  # 把状态图编译成可执行的图

# 用 stream 边跑边打印——messages 是共享状态(黑板)，跑完再打顺序是按对话结构整理的；要看真实时间线得流式看
seen = set()                                                # 记录已打印的消息，去重
for _, state in supervisor.stream(
        {"messages": [{"role": "user", "content": "多智能体协作模式在 2026 年的现状"}]},
        subgraphs=True, stream_mode="values"):              # subgraphs=True 才看得到专家子图内部的事件
    for m in state.get("messages", []):
        if m.id in seen: continue
        seen.add(m.id)
        who = getattr(m, "name", None) or m.type
        content = m.content if isinstance(m.content, str) else str(m.content)
        if content.strip():
            print(f"[{who}] {content[:220]}")

&emsp;&emsp;真实运行的轨迹大致是这样：

```text
[human] 多智能体协作模式在 2026 年的现状
[transfer_to_tech_expert] Successfully transferred to tech_expert
[tech_expert] 2026年多智能体协作主流采用混合式架构：集中式编排器负责全局任务分解与调度，边缘 LLM Agent 执行子任务并本地闭环。通信依赖 MCP/A2A 协议，实现异构 Agent 间工具调用与上下文共享。记忆层普遍引入向量数据库与知识图谱……
[tech_expert] Transferring back to supervisor
[transfer_back_to_supervisor] Successfully transferred back to supervisor
[transfer_to_apply_expert] Successfully transferred to apply_expert
[apply_expert] 2026 年，多智能体系统已从纯中心化或纯去中心化模式，演进为混合式架构：集中式编排器负责任务分解、全局调度和冲突仲裁，边缘 LLM Agent 执行子任务……
[apply_expert] Transferring back to supervisor
[transfer_back_to_supervisor] Successfully transferred back to supervisor
[supervisor] 两份报告已收齐，现在综合成研究简报：
## 研究简报：多智能体协作模式（2026年现状）
2026 年，多智能体系统以混合式架构为主流——中心编排器负责全局调度，边缘 LLM Agent 执行子任务并本地闭环，角色可动态切换……
```

&emsp;&emsp;这条轨迹把 Supervisor 的运转看得清清楚楚：主管通过派单工具先把活转交给技术专家、收回后再转给应用专家，两份回答收齐后，自己把它们综合成最后那段简报——每一步都按发生顺序打了出来。注意主管的派单顺序、综合措辞都是模型当场生成的，不是我们预先定好的——这就是把决策权交给模型的直观体现。

&emsp;&emsp;打印方式也藏着一个值得记一笔的细节：代码里用的是 `stream` 边跑边打印，而不是 `invoke` 跑完后再打印 `result["messages"]`。这两种姿势看到的顺序可能不一样——`messages` 正是 1.5 节讲的<b>共享内存/黑板</b>，它是一份按对话结构整理的最终状态，专家的回答插回黑板的位置由消息配对规则决定，和实际执行的先后并不严格一致；想看"谁先干活、谁后干活"的真实时间线，就要用 `stream` 在事件发生的当下把它打出来。状态是状态、日志是日志，观察多智能体系统时分清这两样，能省下不少"顺序怎么不对"的困惑。

&emsp;&emsp;有一处注意事项。`create_supervisor` 默认走的是上一节说的<b>第一种控制权形态</b>：专家是图上的真实节点，主管通过 `transfer_to_X` 把控制权转移过去、专家跑完再交回来。这一点和下一节 OpenAI SDK 的 agents-as-tools 形态正好相反，两节对照着看，能把两种控制权形态彻底分清。另外，运行时若看到 `create_react_agent` 那条迁移提示，那是版本演进的正常提示，不影响本课代码运行，无需处理。

### 3.3 OpenAI Agents SDK：agents-as-tools

&emsp;&emsp;OpenAI Agents SDK 实现 Supervisor 用的是另一种思路，也就是 3.1 节讲的<b>第二种控制权形态</b>——把每个专家智能体包装成主管手里的一件工具。对集中调度这种模式，它是<b>原生</b>支持：`agent.as_tool()` 是 SDK 内建的一等能力，专门用来把一个 Agent 转成另一个 Agent 可调用的工具。

> <font size=2>【名词解释】<b><font color=red>as_tool</font>（as tool / agents-as-tools，把智能体当工具）</b>:OpenAI Agents SDK 的方法，把一个 Agent 包装成另一个 Agent 可以调用的工具。主管调用这个工具时，专家在内部跑完、把结果当返回值交回，<b>控制权始终留在主管手里、不发生转移</b>——这正是它和 handoff（第五章那种真正把控制权交出去的接力）的根本分界。</font>

&emsp;&emsp;请把这条名词解释和 handoff 严格区分开，它是贯穿 Supervisor 与 Swarm 两章的分界点：as_tool 是"调用了一下，结果拿回来，主管还是主管"，控制权不转移；handoff 是"主管把方向盘交出去，接下来由对方说了算"，控制权真正交出去——前者属于本章 Supervisor，后者属于第五章 Swarm，第五章会专门展开。

&emsp;&emsp;下面这段代码用 OpenAI SDK 实现同一个研究简报场景，走的是它的招牌路径 agents-as-tools。实现逻辑就两步：先把两个专家定义成普通的 `Agent`；再在定义主管时，用 `as_tool` 把它们包成 `ask_tech`、`ask_apply` 两件工具挂进主管的 `tools` 列表。运行时主管把专家当普通工具调用，调一次拿回一份专家答案，先调哪个、何时调完综合，全由主管自己定。通信方式是标准的<b>调用-返回</b>，而且和上一节恰成对照：LangGraph 那边控制权转移给专家、专家跑完交回；这边控制权全程没离开过主管——它把专家当函数调，自己一直坐镇。同样复用环境准备 cell 的三个变量。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L3.3-openai-astool-5aaaf069.png" width=80%></div>

In [ ]:
import asyncio  # 驱动异步的 agent 调用
from openai import AsyncOpenAI  # OpenAI 官方异步客户端，指向 OpenRouter 兼容端点
from agents import Agent, Runner, OpenAIChatCompletionsModel, set_tracing_disabled  # OpenAI Agents SDK：Agent 智能体 / Runner 执行器 / 模型接兼容端点 / 关闭轨迹上报

set_tracing_disabled(True)                                  # 关掉 SDK 默认上报，走第三方端点时必须
client = AsyncOpenAI(base_url=BASE_URL, api_key=KEY)        # 复用环境准备 cell 的 BASE_URL/KEY
model = OpenAIChatCompletionsModel(model=MODEL, openai_client=client)

# 两个专家 Agent
tech_expert = Agent(name="tech_expert", model=model,
                    instructions="你是技术专家。只从技术原理和架构角度回答，80 字以内。")
apply_expert = Agent(name="apply_expert", model=model,
                     instructions="你是应用专家。只从落地场景和典型案例角度回答，80 字以内。")

# 主管：把两个专家 as_tool 挂成自己的工具；主管自己决定何时调哪个、最后综合（控制权不转移）
manager = Agent(name="manager", model=model,
    instructions=("你是研究主管。用 ask_tech 工具问技术维度，用 ask_apply 工具问应用维度，"
                  "两个都问完后综合成一段 150 字以内的研究简报。"),
    tools=[
        tech_expert.as_tool(tool_name="ask_tech", tool_description="就问题咨询技术专家"),  # 把一个 agent 包装成另一个 agent 可调用的工具（agents-as-tools）
        apply_expert.as_tool(tool_name="ask_apply", tool_description="就问题咨询应用专家"),
    ])

async def main():
    r = await Runner.run(manager, "多智能体协作模式在 2026 年的现状")  # OpenAI Agents SDK 执行入口：跑一个 agent 直到产出
    print("【研究简报】", r.final_output)

await main()       # Jupyter 里直接 await

&emsp;&emsp;运行产出如下：

```text
【研究简报】 ## 多智能体协作模式 2026 年研究简报

技术层面，多智能体协作以 AutoGen、CrewAI 等 Transformer 框架为主，核心难点在于通信延迟与冲突消解，动态角色分配与混合推理协议是关键突破。应用层面，已在智能制造、智慧交通、金融风控等领域规模化落地，典型如阿里物流调度系统与百度自动驾驶车队协同，商业化进展显著，部分已实现盈利。整体呈现技术标准化加速、产业渗透加深的态势。
```

&emsp;&emsp;这段产出是主管两次调用工具、拿回技术和应用两份答案后，自己综合出来的最终简报。值得留意的是它清晰地分了"技术维度""应用维度"再收口综合，这个结构正是主管 prompt 引导的结果。

### 3.4 CrewAI：Process.hierarchical

&emsp;&emsp;第二章我们用 CrewAI 的 `Process.sequential` 搭了流程编排，这一节只要把执行流程的枚举值换成 `Process.hierarchical`，同一套角色化写法就变成了 Supervisor。对集中调度这种模式，CrewAI 也是<b>原生</b>支持：指定 `Process.hierarchical` 并给一个 `manager_llm`，CrewAI 会自动替我们造一个主管，由它动态把任务委派给手下的专家、再汇总结果。

&emsp;&emsp;这里有一个必须当场讲清的命名陷阱，它是 CrewAI 这一节最重要的教学点。

> <font size=2>【名词解释】<b><font color=red>Process.hierarchical</font>（hierarchical process，层级流程）</b>:CrewAI 表示"中央委派"执行形态的枚举值。它的名字里带"hierarchical（层级）"，但它的机制其实就是本章的 Supervisor——一个 manager 把任务派给多个平级专家再汇总，并不是真正的多分层协同。第四章我们要讲的 Hierarchical 分层协同指的是"主管手下还有主管"的真正多层嵌套，和 CrewAI 这个枚举值的命名是两码事，千万别因为名字一样就混为一谈。</font>

&emsp;&emsp;记住这条区分：CrewAI 的 `Process.hierarchical` 是<b>单层</b>中央委派，本质是 Supervisor；第四章的 Hierarchical 才是<b>多层</b>嵌套组织。框架的命名沿用了"hierarchical"这个词，但它指的是"有个 manager 居上派单"这层意思，而不是真有多级。下面看代码。

&emsp;&emsp;下面这段代码用 CrewAI 实现研究简报。和第二章流水线版对照着看会很清楚，实现上只动了两个地方：`Task` 不再绑定某个具体的 agent（把"派给谁"的决策权让出来），`Crew` 里的 `process` 从 `sequential` 换成 `hierarchical`、并多传一个 `manager_llm`——CrewAI 会用它自动生成一个隐形的 manager 居中派单。就这两处改动，原本按顺序逐个执行的团队就变成了 Supervisor。这恰好印证 CrewAI 的设计哲学：同一套角色定义，切换 `Process` 就能在协作模式之间转换。通信方式同样是<b>调用-返回</b>——manager 调度两位专家、收齐综合，只是这个 manager 本身是框架替我们造的。同样复用环境准备 cell 的三个变量。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L3.4-crewai-hierarchical-da28419f.png" width=80%></div>

In [ ]:
import os  # 读环境变量（密钥、模型名）
from crewai import Agent, Task, Crew, Process, LLM  # CrewAI 五件套：Agent 角色 / Task 任务 / Crew 团队 / Process 编排方式 / LLM 模型

os.environ.setdefault("OPENAI_API_KEY", KEY)               # CrewAI 底层 litellm 会查这个变量
# litellm 走 OpenRouter 时模型名要带 openrouter/ 前缀，这是 CrewAI 这条链路的硬要求
llm = LLM(model=f"openrouter/{MODEL}", base_url=BASE_URL, api_key=KEY, temperature=0.3, timeout=180)

# 两个专家；hierarchical 模式下不绑 Task，由 manager 决定派给谁
tech_expert = Agent(role="技术专家", goal="从技术原理和架构角度分析问题",
                    backstory="你精通多智能体协作模式的技术机制。", llm=llm, verbose=False)
apply_expert = Agent(role="应用专家", goal="从落地场景和典型案例角度分析问题",
                     backstory="你熟悉多智能体协作模式的产业落地。", llm=llm, verbose=False)

# Task 不绑具体 agent，交给 manager 调度
task = Task(description="就「多智能体协作模式在 2026 年的现状」产出一份研究简报，综合技术与应用两个维度。",
            expected_output="一段 150 字以内的研究简报。")

# Process.hierarchical + manager_llm：CrewAI 自动建一个 manager 负责派单和汇总
crew = Crew(agents=[tech_expert, apply_expert], tasks=[task],  # 把一组 agent + task 编成一个团队
            process=Process.hierarchical, manager_llm=llm, verbose=False)  # CrewAI 层级编排：主管自动调度下属

result = await crew.kickoff_async()  # Jupyter 自带事件循环，须用异步版启动
print("【研究简报】", result)

&emsp;&emsp;运行产出如下：

```text
【研究简报】 两位专家提供了很好的素材，现在我来整合产出最终简报。

**多智能体协作模式 2026 年现状研究简报**

2026年，多智能体协作技术趋向成熟：架构转向去中心化图拓扑，基于A2A协议实现动态任务编排，MCP统一语义接口实现跨模型互操作，强化学习驱动协作策略自优化。应用层面，金融风控、医疗影像、供应链调度等领域形成"感知-决策-执行"闭环，工业场景多机器人协作效率提升超30%。标准化协议与安全框架正加速推动人机混合团队规模化落地。
```

### 3.5 MAF：Magentic 编排

&emsp;&emsp;到这一节，我们已经看了三家"让模型自主派单"的 Supervisor。MAF 对这种模式同样是<b>原生</b>支持，而且给出的是五家里最重型的一版：<font color=red>Magentic 编排</font>。这个名字来自微软研究院的 Magentic-One 多智能体架构，MAF 把它产品化成了 `MagenticBuilder`——一个真正的 LLM 主管，不只临场决定派给哪个专家，还会先把任务拆成一份计划（任务账本），派单过程中持续盯进度（进度账本），发现卡住了甚至会推翻计划重新规划。前三家的主管是"调度员"，Magentic 的主管更像一个"项目经理"。

&emsp;&emsp;这一节模型用 GLM 5.1——deepseek 经 MAF 这条链路跑重型编排时会发散（实测过：反过来追问用户、往外吐代码块），GLM 5.1 配合轮数上限实测可用。这也正是本课"模型不确定性、确定性兜底"主题在 MAF 上的落点：主管的规划和派单交给模型自主，但用 `max_round_count` 这道确定性保险丝兜住它跑飞的可能。

> <font size=2>【名词解释】<b><font color=red>MagenticBuilder</font>（Magentic Builder，Magentic 编排构造器）</b>:MAF 官方五种编排模式里的编排者模式构造器，传入一组专家（participants）和一个主管智能体（manager_agent），主管自主完成任务规划、逐轮派单、进度跟踪与最终综合；`max_round_count` 限制总轮数，防止主管无限循环。</font>

&emsp;&emsp;下面这段代码用 Magentic 编排实现研究简报。实现逻辑分三步：先用 `as_agent` 创建技术、应用两个专家和一个主管（三个都是普通智能体，主管的"项目经理"能力由框架注入）；再用 `MagenticBuilder` 把专家列表和主管装配起来，`max_round_count=8` 给它装上轮数保险丝，`intermediate_output_from='all'` 让每个智能体的过程产出都能拿到；最后 `await wf.run()` 一次跑完，先打印各步过程产出、再打印主管综合的最终简报。运行时"先派谁、派几轮、何时收口"全由主管模型自主决定，没有一行代码固定派单顺序。通信方式是<b>调用-返回</b>：主管把任务逐轮派给专家、收回结果更新进度账本。同样复用环境准备 cell 的变量。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L3.5-maf-magentic-51fa3400.png" width=80%></div>

In [ ]:
from agent_framework.openai import OpenAIChatClient   # MAF 聊天客户端，as_agent 把它变成一个 agent
from agent_framework.orchestrations import MagenticBuilder  # MAF 官方编排者模式：主管自主规划、派单、盯进度

# 这一节模型用 GLM 5.1（deepseek 经 MAF 跑重型编排会发散）
client = OpenAIChatClient(model=GLM, api_key=KEY, base_url=BASE_URL)

# 两个专家 + 一个主管，都是普通 as_agent；主管的"项目经理"能力由 MagenticBuilder 注入
tech_expert = client.as_agent(name="tech_expert",
    instructions="你是技术专家，只从技术原理和架构角度回答，80 字以内中文。")
apply_expert = client.as_agent(name="apply_expert",
    instructions="你是应用专家，只从落地场景和典型案例角度回答，80 字以内中文。")
manager = client.as_agent(name="manager",
    instructions="你是研究主管，负责规划任务、把活派给合适的专家、最后综合成 150 字以内中文研究简报。")

# Magentic 编排：主管自主规划与派单，max_round_count 是防失控的轮数保险丝
wf = MagenticBuilder(participants=[tech_expert, apply_expert],
                     manager_agent=manager,
                     max_round_count=8,
                     intermediate_output_from="all").build()

async def main():
    result = await wf.run("就「多智能体协作模式在 2026 年的现状」产出一份研究简报："
                          "技术维度问 tech_expert，应用维度问 apply_expert，收齐后综合，150 字以内中文。")
    for i, o in enumerate(result.get_intermediate_outputs(), 1):   # 各轮过程产出（专家回答、主管中间综合）
        print(f"【过程产出 {i}】{o.text}\n")
    print("【主管简报】", result.get_outputs()[0].text)             # 主管最终综合

await main()       # Jupyter 里直接 await

&emsp;&emsp;运行产出如下：

```text
【过程产出 1】通信依语义协议跨框架互联；编排转去中心化事件驱动；共识借 BFT 与密码学保状态信任一致。

【过程产出 2】聚焦企业级流程自动化。场景：跨域供应链协同与具身群控；案例：汽车制造跨厂排产、金融反欺诈多源联防。

【过程产出 3】企业级流程自动化正迈向跨域协同与具身群控新范式。技术端依托多智能体协同与实时数据融合实现全局动态寻优；应用端聚焦打破孤岛、提升业务韧性……

【主管简报】 【2026年多智能体协作现状简报】
技术层基于语义协议跨框架互联与去中心化事件驱动，以 BFT 与密码学保障状态信任一致；应用层该可信体系深度赋能企业级流程自动化，精准支撑跨域供应链协同与具身群控，落地汽车跨厂排产与金融反欺诈联防，构建高可信跨域协同新范式。
```

&emsp;&emsp;过程产出里能看到主管这位"项目经理"的工作痕迹：前两条分别是技术专家、应用专家被派单后的回答，后面是主管把两份材料逐步揉成简报的中间稿——派单顺序、揉合方式都是它自主决定的。和前三家相比，Magentic 的特点是过程更重：主管在背后维护着任务账本和进度账本，所以同样一个研究简报，它的轮数和耗时都更多，换来的是面对更复杂、更长程的任务时不容易跑丢。简单任务用它有点高射炮打蚊子，前三家的轻量主管更划算——这也是同一种模式里不同框架的取舍差异。还要交代一句实测边界：MAF 高层 builder 要分个体看，同版本里 `GroupChatBuilder` 配 GLM 经第三方端点会在消息缓存上报错跑不通，而 `MagenticBuilder` 和第二章用过的 `SequentialBuilder` 实测都稳——选型时别因为一个 builder 翻车就整体否定，也别因为一个能跑就全盘信任，逐个实测是唯一靠谱的办法。

### 3.6 Claude Agent SDK：subagents

&emsp;&emsp;Supervisor 是 Claude Agent SDK 真正的看家模式。第二章里它做流程编排只是"官方示例级"地靠多次 `query()` 硬串，到了集中调度这里，它有专门为之设计的一等能力——subagents（子智能体）。这种"一个主智能体编排一组子智能体"的结构，业界叫 orchestrator-worker，正是 Anthropic 自己那套多智能体研究系统的核心架构。对 Supervisor 模式，它是<b>原生</b>支持，而且是它最拿手的那一种。

> <font size=2>【名词解释】<b><font color=red>subagents</font>（subagents，子智能体）</b>:Claude Agent SDK 的看家机制，在主智能体下挂一组各有专长的子智能体。主智能体根据每个子智能体的 description 自动决定把任务委派给谁，子智能体在各自隔离的上下文里干活——这正是 orchestrator-worker（编排者-工作者）结构的直接实现。</font>

> <font size=2>【名词解释】<b><font color=red>AgentDefinition</font>（agent definition，智能体定义）</b>:Claude Agent SDK 用来定义一个 subagent 的结构，三个关键字段是 description（它擅长什么，主智能体据此决定要不要派给它）、prompt（它的角色设定）、tools（它能用哪些工具）。</font>

&emsp;&emsp;和第二章 2.6 节一样，这一节也用 GLM 5.1 经 OpenRouter 驱动 Claude SDK——设好那三个 `ANTHROPIC_*` 环境变量即可，模型同样从 OpenRouter 取，不需要 Anthropic 凭证。

&emsp;&emsp;下面这段代码用 Claude SDK 的 subagents 实现研究简报。实现逻辑分两步：先用 `AgentDefinition` 定义两个专家，每个写清 `description`（它是干什么的，给主智能体看的"岗位说明"）和 `prompt`（它具体怎么干活）；再把它们以 `agents={...}` 字典塞进 `ClaudeAgentOptions`，连同主智能体的总指令一起交给 `query()`。运行时主智能体读各 subagent 的 `description`，自主决定把技术维度派给谁、应用维度派给谁——我们只下达一句"分别委派、收齐综合"的总指令，具体派单和综合全由它完成。通信方式是<b>调用-返回</b>：subagent 在自己隔离的上下文里干活，只把最终结论交回主智能体。消费方式沿用 2.6 节认识的 `query()` 加 `async for`。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L3.6-claude-subagents-1b7395d2.png" width=80%></div>

In [3]:
import tempfile  # 建干净空目录跑，避免读到当前项目文件污染上下文
from claude_agent_sdk import query, ClaudeAgentOptions, AgentDefinition, AssistantMessage, TextBlock  # Claude Agent SDK：query 发起对话 / AgentDefinition 定义子智能体 / 消息类型解析回复

# 和 2.6 一样，让 Claude SDK 从 OpenRouter 取 GLM 5.1 模型
CLAUDE_ENV = {"ANTHROPIC_BASE_URL": "https://openrouter.ai/api",
              "ANTHROPIC_AUTH_TOKEN": KEY, "ANTHROPIC_MODEL": GLM}

# 两个 subagent 用 AgentDefinition 定义；主 agent 按 description 自动决定派给谁
options = ClaudeAgentOptions(
    model=GLM, env=CLAUDE_ENV, cwd=tempfile.mkdtemp(),   # 模型路由到 OpenRouter；干净目录隔离项目上下文
    permission_mode="bypassPermissions",                  # 委派 subagent 走内置工具，跳过授权交互
    agents={
        "tech_expert": AgentDefinition(
            description="技术专家，只回答技术原理和架构维度的问题",
            prompt="你是技术专家。只从技术原理和架构角度回答，80 字以内。", tools=[]),
        "apply_expert": AgentDefinition(
            description="应用专家，只回答落地场景和案例维度的问题",
            prompt="你是应用专家。只从落地场景和典型案例角度回答，80 字以内。", tools=[]),
    },
    max_turns=20,
)

async def main():
    text = ""
    prompt = ("作为研究主管，就「多智能体协作模式在 2026 年的现状」分别委派 tech_expert 问技术维度、"
              "apply_expert 问应用维度，收齐后综合成一段 150 字以内的研究简报。")
    async for msg in query(prompt=prompt, options=options):  # query 返回异步生成器，逐条消费
        if isinstance(msg, AssistantMessage):
            for block in msg.content:
                if isinstance(block, TextBlock):
                    text += block.text
    print("【研究简报】", text.strip())

await main()       # 用 GLM 5.1 经 OpenRouter；Jupyter 里直接 await

【研究简报】 我将以研究主管身份，分别委派技术专家和应用专家从两个维度调研，然后综合成简报。两位专家的调研已收齐，综合简报如下：

---

## 研究简报：多智能体协作模式 2026 年现状

2026 年，多智能体协作从实验走向生产：架构上，编排式仍为主流，A2A/MCP 协议推动对等式协作快速崛起，混合拓扑渐成常态；技术上，共享记忆升级为知识图谱+情景记忆混合方案，任务自动 DAG 生成与监督式容错动态重分配成为标配；应用已在软件开发、金融风控、智能客服等场景规模化落地，流程效率提升超 40%，涌现出单智能体无法实现的跨域推理能力；挑战集中在通信开销、错误传播放大与可观测性不足，业界通过编排层集中协调、护栏回滚机制与全链路因果审计体系应对。


&emsp;&emsp;运行产出如下：

&emsp;&emsp;运行产出如下：

```text
【研究简报】 2026年多智能体协作已形成分层编排为主、去中心化与黑板模型并存的架构态势；MCP/A2A 协议标准化与 LangGraph/CrewAI 等编排框架成熟是关键突破。落地集中于代码生成、深度研究、智能客服三大场景。核心瓶颈：错误级联放大、编排成本高、可观测性不足，简单任务性价比仍不及单 Agent。
```

&emsp;&emsp;这份简报是主智能体委派两个 subagent、收齐各自在隔离上下文里产出的答案后综合而成的。可以注意到这份产出比前几家更具批判性，主动点出了错误级联、编排成本、可观测性这些瓶颈，还给出"简单任务性价比不及单 Agent"的判断——这正好呼应第一章那笔成本账。

### 3.7 支持程度与生态对照

&emsp;&emsp;五个框架的 Supervisor 都写完了，照例把它们在集中调度这种模式上的支持程度和原语并排放进一张表。沿用第一章 1.4 节那套本课内部归类口径——原生、官方示例级、可拼装、不适配，这是为横向对照自定的口径，不是某一家的官方术语。

<div align=center><font size=2 color=#999999>五个框架对 Supervisor 集中调度的支持程度与生态对照</font></div>

<div align="center">
<table width="80%">
<thead><tr>
<th>框架</th><th>出品方</th><th>本模式支持等级</th><th>模式专用原语</th><th>一句话点评</th>
</tr></thead>
<tbody>
<tr><td>LangGraph</td><td>LangChain</td><td>原生</td><td>create_supervisor + create_react_agent</td><td>控制最细，派单轨迹完整可见</td></tr>
<tr><td>OpenAI Agents SDK</td><td>OpenAI</td><td>原生</td><td>agent.as_tool（agents-as-tools）</td><td>最简，控制权不转移、不到百行</td></tr>
<tr><td>CrewAI</td><td>CrewAI 公司</td><td>原生</td><td>Process.hierarchical + manager_llm</td><td>改个枚举值就从流水线变 Supervisor</td></tr>
<tr><td>MAF</td><td>Microsoft</td><td>原生</td><td>MagenticBuilder（编排者自主规划派单）</td><td>五家里最重型的主管：任务账本 + 进度账本 + 轮数保险丝</td></tr>
<tr><td>Claude Agent SDK</td><td>Anthropic</td><td>原生</td><td>AgentDefinition + subagents</td><td>Anthropic 看家模式，生产实证，可经 OpenRouter 换模型</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;这张表给出一个很一致的判断：五家对 Supervisor 全都是<font color=red>原生</font>支持，没有一家只是勉强拼装。这印证了业界对这个模式家族的定位——把本章的 supervisor 和下一章的 hierarchical 合在一起的 orchestrator-worker 家族，占了 2026 年约七成的生产级多智能体部署，是当下最主流的结构；其中集中调度更被 Anthropic 列为"单个智能体不够用时第一个该考虑的设计"，因为它对子智能体的假设最少、几乎什么场景都套得上。五家全员原生，恰恰是这个模式通用性的直接证据。

&emsp;&emsp;五家虽然都原生，侧重各不同。要派单轨迹看得清、能在每个节点精细控状态，选 LangGraph 的 `create_supervisor`；要代码最短、最快搭出一个能用的主管，选 OpenAI SDK 的 agents-as-tools，不到一百行就能跑；已经在用 CrewAI 写流水线、想顺手加个主管，把 `Process` 换成 `hierarchical` 就行；要面对更复杂、更长程的任务，MAF 的 MagenticBuilder 给的是带任务账本和进度账本的重型主管；而如果团队本就在 Anthropic 生态里、要的是经过生产实证的 orchestrator-worker，Claude SDK 的 subagents 是它的看家本领。

&emsp;&emsp;选型上的总结论也清楚：Supervisor 是四种模式里最通用、最容易控制的一种，单个智能体扛不住时，它通常是第一个该试的方案。它的代价我们在 3.1 节已经讲透——中央主管是单点瓶颈，主管的上下文会随专家结果膨胀，并行派单是用 token 换时间。把这三条代价和它"最易控、最通用"的优点放在一起权衡，就能判断一个任务该不该上 Supervisor：任务能拆成几个独立子任务交给专家、又需要一个中心来统筹综合，它就是合适的；如果子任务之间要频繁互相传递、对话焦点会随过程漂移，那就不是 Supervisor 的主场了。

&emsp;&emsp;本章我们把 Supervisor 集中调度从机制讲到了五家的代码实现：它的本质是一个中央主管临场决定派谁、收齐后综合，子智能体互不可见、只跟主管单线通信，派单又分控制权转移和不转移两种形态。我们也看到了五家主管的轻重之分——从轻量调度到 MAF 带双账本的重型项目经理。但 Supervisor 终究只有一层——一个主管管一组专家。当产出物复杂到一个主管下面还需要再分组、每组还得有自己的小主管时，单层中央就不够用了。下一章，我们把这个结构再叠一层，进入 Hierarchical 分层协同。

## <center>第四章 Hierarchical 分层协同：两家成立与三家边界</center>

&emsp;&emsp;上一章结尾留了个问题：当一个主管管不过来时怎么办？答案直白得有点出人意料——给主管再配主管。这就是 Hierarchical 分层协同的全部出发点。当产出物复杂到一个中央主管下面还得再分几个组、每组还得有自己的小主管来盯细节时，单层的 Supervisor 就被压垮了，我们需要把组织结构再往下叠一层。

&emsp;&emsp;放到具体业务里就很好理解。上一章的研究简报，一个主管派给两个专家就够了；可一旦要交付一个完整软件，技术总监一个人盯不过来前端、后端、测试所有细节，他更自然的做法是把活分给前端组长和后端组长，每个组长再带着自己的工程师干活。总监只跟两位组长对话，组长各自对自己的工程师负责——这种"上层只对组长说话、组长对组员负责"的多层委派，正是本章要拆解的分层协同。下面这张组织树把这个三层结构画了出来。

<div align=center><font size=2 color=#999999>第四章：技术总监、组长、工程师构成的三层组织，每一层主管都自主决定派给哪个下级</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L4-hierarchy-tree-31da0b6a.png" width=80%></div>

<br>

&emsp;&emsp;先把本章的判据立在前头：真正的分层协同，要求<b>每一层的主管都由模型自主决定派给哪个下级，而且下级主管还能再自主往下派</b>——层层都是自主决策，不是代码替模型把路线定死。按这把判据衡量，五个框架直接分成两拨：LangGraph 和 OpenAI Agents SDK 能原生、干净地把两层以上的自主委派搭出来；CrewAI、MAF、Claude Agent SDK 做不到——它们最多原生支持单层的主管或交接，再往上叠只能靠外层代码模拟，那已经不是框架自己的分层语义了。所以接下来五个小节这样走：先讲清分层协同的机制；然后 LangGraph、OpenAI SDK 两家各写一段能直接运行的多层自主委派；接着用一个小节把做不到的三家各自卡在哪里说清楚；最后照例用对照表收口。

### 4.1 层级机制

&emsp;&emsp;分层协同的核心机制，用一句话概括就是：<font color=red>多层委派</font>。上层不直接指挥最底层的执行者，而是只跟它的直接下级说话——总监把需求交给组长，组长把拆好的任务交给工程师，工程师产出后逐层往上汇报。每一层都只管自己这一跨的派单和收口，上下文也随之逐层收窄：工程师只看到组长给的那一小块任务，组长只看到本组的进展，总监只拿到两个组的最终结论。这种逐层收窄正是分层协同最大的价值——它让顶层不至于淹没在底层的实现细节里，总监不用知道前端用了哪个状态管理库，只需要知道"前端这块已经完成"。

&emsp;&emsp;为什么说它跟上一章是同一回事的升级？因为 **Hierarchical 本质就是 Supervisor 的嵌套复合**。把组织树拆开看：技术总监对两个组长，是一个 Supervisor——总监是主管，两个组长是它的下属；前端组长对它手下的工程师，又是一个 Supervisor；后端组长对它的工程师，还是一个 Supervisor。整个分层协同，就是把上一章那个"一主管多专家"的结构，一层套一层地嵌套起来。所以理解了第三章的 Supervisor，分层协同在概念上就没有新东西，只是把同一个模式递归地用了几次。这也解释了为什么本章不再从零讲派单机制——它复用的就是上一章那套。

&emsp;&emsp;为了把这个三层结构讲透，本章统一换一个比研究简报更复杂的演示场景：**软件交付**。需求很简单，"做一个待办事项小应用"，但交付一个软件天然需要分组分工——前端组负责界面和交互，后端组负责接口和存储，每个组里组长先把需求拆成本组要做的核心任务，再由工程师给出具体实现，最后技术总监把两个组的方案汇总成一份交付简报。这个场景正好呼应本课第三部分会做的多智能体软件交付案例，到那时我们会用最重的项目把分层协同彻底落地。

### 4.2 LangGraph：嵌套主管

&emsp;&emsp;讲真正的多层自主，LangGraph 是最该先看、也实现得最干净的一家。对这种模式它是<b>原生</b>支持，靠的就是上一章那个 `create_supervisor` 能<b>嵌套</b>这个特性：一个用 `create_supervisor` 装配、编译好的主管，本身又能被当成"一个 agent"，交给更上一层的 `create_supervisor` 去管理。一层套一层，每一层的主管都是上一章那个会自主派单的 Supervisor，多层自主就成立了。

> <font size=2>【名词解释】<b><font color=red>嵌套主管</font>（supervisor-of-supervisors，主管的主管）</b>:把一个编译好的 Supervisor 当成普通 agent，再交给更上层的 `create_supervisor` 管理，形成"主管管主管、主管再管专家"的多层结构。每一层主管都用自己的 `transfer_to_X` 派单工具自主决定把活交给哪个下层——这正是分层协同要求的"上层自主派下层、下层还能再自主往下派"。</font>

&emsp;&emsp;具体怎么搭？分三层往上垒。<b>最底层是四个 worker</b>：前端的界面工程师、状态工程师，后端的接口工程师、存储工程师，各用 `create_react_agent` 造出来、各管一摊。<b>中间层是两个团队主管</b>：前端组长用 `create_supervisor` 把界面、状态两个 worker 管起来，后端组长同理管接口、存储两个 worker——这两个组长编译时各起一个 `name`（`frontend_team`、`backend_team`），就成了能被上层寻址的"一个 agent"。<b>最顶层是 CTO 主管</b>：再用一次 `create_supervisor`，把这两个团队主管当 agent 管起来。运行时，CTO 自主决定先把活交给哪个团队，团队组长拿到后又自主决定先派给本组哪个 worker——每一层的派单都是模型当场拿的主意，没有一行代码预先规定"先做谁再做谁"。

&emsp;&emsp;这就是"多层自主"的完整含义：控制权从 CTO 交到组长、再交到 worker，干完一层层交回来，每一次转交都由当前那一层的主管自主决策。通信方式上它走的是<b>控制权交接</b>——和上一章单层 Supervisor 是同一套 `transfer` 机制，只是这里叠了两层。下面这段代码就是这么搭的，复用第二章 2.1 节环境准备 cell 里的 `MODEL`、`BASE_URL`、`KEY` 三个变量。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L4.2-langgraph-nested-supervisor-9eec5e69.png" width=80%></div>

In [ ]:
from langchain_openai import ChatOpenAI  # LangChain 封装的 OpenAI 兼容聊天模型
from langgraph.prebuilt import create_react_agent     # 造底层 worker（会自主推理的 ReAct 智能体）
from langgraph_supervisor import create_supervisor    # 装配主管；它的产物可以再被上层 create_supervisor 嵌套

# 复用第二章环境准备 cell 的 MODEL/BASE_URL/KEY
model = ChatOpenAI(model=MODEL, base_url=BASE_URL, api_key=KEY, temperature=0.3, timeout=180, max_retries=1)

# ---- 最底层：四个 worker，各管一摊 ----
fe_ui = create_react_agent(model, tools=[], name="fe_ui",
    prompt="你是前端界面工程师。给出待办应用的界面与交互实现，60 字以内。")
fe_state = create_react_agent(model, tools=[], name="fe_state",
    prompt="你是前端状态工程师。给出待办应用的前端状态管理与数据持久化方案，60 字以内。")
be_api = create_react_agent(model, tools=[], name="be_api",
    prompt="你是后端接口工程师。给出待办应用的 REST 接口设计，60 字以内。")
be_db = create_react_agent(model, tools=[], name="be_db",
    prompt="你是后端存储工程师。给出待办应用的数据库表与存储方案，60 字以内。")

# ---- 中间层：两个团队主管，各自管两个 worker（这一层 LLM 自主派单），编译时起 name 供上层寻址 ----
frontend_team = create_supervisor(
    agents=[fe_ui, fe_state], model=model,
    prompt="你是前端组长。把界面交互交给 fe_ui，状态与持久化交给 fe_state，两份都收齐后用一句话汇报前端方案。",
    supervisor_name="fe_lead",
).compile(name="frontend_team")     # 编译成"一个 agent"，名字叫 frontend_team

backend_team = create_supervisor(
    agents=[be_api, be_db], model=model,
    prompt="你是后端组长。把接口设计交给 be_api，存储方案交给 be_db，两份都收齐后用一句话汇报后端方案。",
    supervisor_name="be_lead",
).compile(name="backend_team")

# ---- 最顶层：CTO 主管把两个团队主管当 agent 管起来（顶层 LLM 自主派单）----
top = create_supervisor(
    agents=[frontend_team, backend_team], model=model,   # 注意：传进来的是两个"主管"，不是 worker——这就是嵌套
    prompt=("你是技术总监。把前端相关工作交给 frontend_team，后端相关工作交给 backend_team，"
            "两个团队都汇报完后，综合成一段 120 字以内的交付简报。"),
    supervisor_name="cto",
).compile()

# 同 3.2：stream 流式打印才是真实时间线（嵌套图要 subgraphs=True 才看得到组内事件）
seen = set()
for _, state in top.stream(
        {"messages": [{"role": "user", "content": "做一个待办事项小应用"}]},
        subgraphs=True, stream_mode="values"):
    for m in state.get("messages", []):
        if m.id in seen: continue
        seen.add(m.id)
        who = getattr(m, "name", None) or m.type
        content = m.content if isinstance(m.content, str) else str(m.content)
        if content.strip():
            print(f"[{who}] {content[:200]}")

&emsp;&emsp;代码里最值得盯一眼的是最顶层 `create_supervisor(agents=[frontend_team, backend_team], ...)` 这一行——传进去的不是 worker，而是两个本身就是主管的编译产物，"主管管主管"的嵌套落点就在这。运行的协作轨迹大致是这样：

```text
[human] 做一个待办事项小应用
[cto] 好的，我来安排。前端团队负责界面与交互，后端团队负责数据与接口，两边同步推进。
[transfer_to_frontend_team] Successfully transferred to frontend_team
[fe_lead] 好的，现在先由 fe_ui 产出界面与交互方案。请稍候。
[transfer_to_fe_ui] Successfully transferred to fe_ui
[fe_ui] 输入框+添加按钮，列表展示待办，点击切换完成状态，滑动删除。
[fe_ui] Transferring back to fe_lead
[fe_lead] 界面方案已收到。接下来收状态与持久化方案。
[transfer_to_fe_state] Successfully transferred to fe_state
[fe_state] 用 useState 管理待办，localStorage 持久化，初始化加载，变化时保存。
[fe_state] Transferring back to fe_lead
[fe_lead] 两份方案已收齐。汇报如下——前端方案：顶部输入框+添加按钮录入待办，列表展示并点击切换完成、滑动删除；状态层 useState 管理、localStorage 持久化。
[frontend_team] Transferring back to cto
[transfer_to_backend_team] Successfully transferred to backend_team
[be_lead] 先由 be_api 出接口设计。
[transfer_to_be_api] Successfully transferred to be_api
[be_api] RESTful API：GET/POST /todos、PUT/DELETE /todos/:id，返回 { id, title, done, created_at }。
[be_api] Transferring back to be_lead
[be_lead] 接口设计已收到。接下来找 be_db 要存储方案：
[transfer_to_be_db] Successfully transferred to be_db
[be_db] todos 表：todos(id, title, done, created_at)，唯一索引 id。存储用 PostgreSQL。
[be_db] Transferring back to be_lead
[be_lead] 接口与存储方案均已收齐，汇报如下——后端方案：RESTful API 操作 todos 表，数据库采用 PostgreSQL。
[backend_team] Transferring back to cto
[cto] 简报：前端用输入框+列表展示，点击切换/滑动删除，useState+localStorage 管理；后端提供 RESTful API 操作 PostgreSQL 待办表。两端对齐，可启动联调。
```

&emsp;&emsp;这条轨迹把"多层自主"看得清清楚楚：CTO 先 `transfer_to_frontend_team`，前端组长拿到后让两个工程师各答一段、自己汇总成一句、再 `transfer_back_to_cto`；CTO 接着 `transfer_to_backend_team`，后端走同样一遍；两个团队都交回后，CTO 综合成交付简报。每一次 `transfer` 都是当前那一层的主管模型自主决定的——顶层 CTO 决定派给哪个团队，团队组长决定派给本组哪个工程师，全程没有代码预先规定派单顺序。这就是 LangGraph 做分层协同最大的价值：嵌套结构把"多层自主"原原本本表达了出来。

### 4.3 OpenAI Agents SDK：嵌套 agents-as-tools

&emsp;&emsp;能把多层自主委派搭起来的，还有 OpenAI Agents SDK。它没有层级专用的构造器，靠的是上一章那件 `as_tool` 反复嵌套——把一个智能体包装成另一个智能体的工具。上一章是单层：主管把专家当工具。这一章我们把它<b>嵌套两层</b>，实现逻辑全藏在 `tools=[...]` 的嵌套里：`make_lead` 造组长时，把工程师 `as_tool` 塞进组长的工具列表（第一层）；定义总监时，再把两个组长 `as_tool` 塞进总监的工具列表（第二层）。运行时总监调"问前端组长"这件工具，组长在自己内部又会调"问前端工程师"——一次总监的工具调用连环触发下一层的工具调用，层级就这么拼出来了。关键在于，这两层调用没有一步是代码预先定死的：总监要不要问组长、先问哪个组长，是总监模型自己决定的；组长拿到任务后要不要调工程师，也是组长模型自己决定的——<b>每一层都是模型自主决策</b>，所以按本章的判据，它和 LangGraph 一样成立。通信方式是<b>调用-返回</b>的两层嵌套版，每一层都是上层把下层当函数调、等结果返回。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L4.3-openai-nested-4fe87087.png" width=80%></div>

In [ ]:
import asyncio  # 驱动异步的 agent 调用
from openai import AsyncOpenAI  # OpenAI 官方异步客户端，指向 OpenRouter 兼容端点
from agents import Agent, Runner, OpenAIChatCompletionsModel, set_tracing_disabled  # OpenAI Agents SDK：Agent 智能体 / Runner 执行器 / 模型接兼容端点 / 关闭轨迹上报

set_tracing_disabled(True)                              # 关掉 SDK 默认上报，走第三方端点时必须
client = AsyncOpenAI(base_url=BASE_URL, api_key=KEY)    # 复用第二章环境准备 cell 的 BASE_URL/KEY
model = OpenAIChatCompletionsModel(model=MODEL, openai_client=client)

def make_lead(side: str) -> Agent:                      # 组长：把工程师当工具挂上（第一层嵌套）
    eng = Agent(name=f"{side}_eng", model=model,
                instructions=f"你是{side}工程师，给出{side}实现方案，80 字以内。")
    return Agent(name=f"{side}_lead", model=model,
                 instructions=f"你是{side}组长。用工具问{side}工程师拿实现方案，然后一句话汇报本组结论。",
                 tools=[eng.as_tool(tool_name=f"ask_{side}_eng", tool_description=f"问{side}工程师实现方案")])  # 把一个 agent 包装成另一个 agent 可调用的工具（agents-as-tools）

fe_lead = make_lead("前端")
be_lead = make_lead("后端")
# 总监：把两个组长当工具挂上（第二层嵌套）
cto = Agent(name="cto", model=model,
            instructions="你是技术总监。用工具分别问前端组长、后端组长拿本组结论，综合成 100 字以内交付简报。",
            tools=[fe_lead.as_tool(tool_name="ask_fe_lead", tool_description="问前端组长本组结论"),
                   be_lead.as_tool(tool_name="ask_be_lead", tool_description="问后端组长本组结论")])

async def main():
    # max_turns 要给够：两层嵌套意味着总监一次调用会触发组长再调工程师，工具调用轮次成倍增加
    r = await Runner.run(cto, "做一个待办事项小应用", max_turns=20)  # OpenAI Agents SDK 执行入口：跑一个 agent 直到产出
    print("【技术总监交付简报】", r.final_output)

await main()       # Jupyter 里直接 await

&emsp;&emsp;运行产出如下：

```text
【技术总监交付简报】 **【待办事项小应用 · 技术交付简报】**

**前端**：React + TypeScript，组件拆分为 AddTodo → TodoList → TodoItem，useReducer 单向数据流，CSS Modules 隔离样式，localStorage 持久化。

**后端**：Node.js + Express + MySQL + JWT 认证，提供注册登录及待办 CRUD 接口，支持分页与 pending→doing→done 状态流转，表结构已定稿。

**结论**：前后端方案均已明确，技术栈对齐，组件与接口定义清晰，可以进入开发阶段。
```

&emsp;&emsp;两层嵌套确实跑通了，总监拿到的简报里能看出前端、后端两条线各自的产出已被它综合到一起。

### 4.4 做不到的三家：CrewAI、MAF 与 Claude SDK

&emsp;&emsp;剩下三家，按本章开头那把判据——每一层都由模型自主派单、下层还能再自主往下派——都不成立。但三家不成立的方式各不相同，把"卡在哪"说清楚，比硬写一段似是而非的多层代码更有价值：拿不存在的原生能力去硬拼，拼出来的东西既不稳定也不可维护，生产里真要多层组织，换框架比硬拼更划算。

&emsp;&emsp;<b>CrewAI：单层自主成立，跨层是代码串的</b>。上一章我们见过它的 `Process.hierarchical`——一个 Crew 配上 `manager_llm`，框架自动生成的 manager 确实是模型自主派单，所以单层的 Supervisor 它是真成立的。但它没有任何<font color=red>跨 Crew 的层级原语</font>：要搭"总监—组长—工程师"，只能在上层代码里依次 `kickoff()` 跑多个 Crew、再手动把上一个 Crew 的产出拼进下一个 Crew 的任务描述。"总监把活派给哪个组"这一跳是我们的代码固定的，组与组之间框架互不知情——多层语义不是 CrewAI 表达的，是我们用代码的执行顺序模拟出来的，按本章判据不算分层协同。

&emsp;&emsp;<b>MAF：没有层级原语，多层只能靠代码固定</b>。MAF 的五种官方编排模式里没有<font color=red>层级专用的构造器</font>，第三章我们也实测过它的高层编排原语配第三方端点时并不稳。要多层，只剩一条路：用 `as_agent` 把总监、组长、工程师各建出来，再用代码把"组长先答、工程师再答、总监最后汇总"的调用顺序逐行固定。这条路能跑——它本质就是普通的异步函数编排——但每一层"派给谁"都是代码决定的，模型只在被点到的位置产出内容。按本章判据，这不是分层协同，是套了组织名词的<font color=red>流程编排</font>。

&emsp;&emsp;<b>Claude Agent SDK：被一条官方规则直接封死</b>。它的 subagents 是上一章的看家本领——主智能体下挂一组子智能体。可分层协同要的是"组长这层 subagent 还能再往下派工程师"，也就是 subagent 自己再 spawn（创建）下一层的 subagent。而这一点，恰恰被官方一条明文规则挡死了：

> <font size=2>官方文档原文（来源 code.claude.com/docs/en/sub-agents）：<b>"Subagents cannot spawn other subagents."</b>（子智能体不能创建其它子智能体。）官方给出的替代方向是：如果工作流需要嵌套委派，改用 Skills，或从主对话里串联多个子智能体。</font>

&emsp;&emsp;这条规则的意思很直接：subagent 是终点，它不能再往下生出自己的 subagent。整个结构被锁死在"主智能体 + 一层 subagent"两层，没有第三层。所以本章这种"总监—组长—工程师"的三层组织，在 Claude SDK 里根本搭不出来——组长这一层的 subagent，没法再往下派工程师那一层。它对分层协同的态度，是**结构上的不适配**，不是"能拼但难拼"，而是设计上就堵死了这条路。

&emsp;&emsp;关键在于，**这不是一个缺陷，而是一个刻意的设计选择**。Claude SDK 的世界观是"一主多仆的扁平结构"——一个强大的主智能体，编排一组各有专长、但彼此平级的 subagent。Anthropic 把整个框架的复杂度都押在了"一层编排 + 把单个 subagent 做强"这件事上，而不是去支持任意深度的嵌套组织。官方文档里那句"嵌套委派请改用 Skills 或从主对话串联"也印证了这个取向：它宁可让开发者把复杂度收回到主对话里用别的机制解决，也不打开 subagent 无限嵌套这个口子。对于更大规模的编排，官方给的方向是 Workflow 类工具（目前主要在 TypeScript 侧），而不是让 subagent 长出层级。

> <b>【提示】框架世界观决定模式上限，这是选型判断力的核心</b>：这一节最该带走的，不是"哪三家做不了多层"这个结论本身，而是它背后的判断方法——**一个框架支持哪些协作模式，根子上是由它的设计哲学决定的，而不是功能多寡**。LangGraph 把世界看成可组合的图，主管套主管对它天然成立，层级支持最完整；CrewAI 和 MAF 的世界观停在"一支团队、一条编排"，跨团队的组织层级不在它们的语义里；Claude SDK 把世界看成"一主多仆"，层级嵌套对它天然不成立。选型时，与其逐个去查"这个框架有没有 X 功能"，不如先想清楚"这个框架的世界观是什么、它把复杂度押在了哪里"——世界观对上了，模式自然顺；世界观拧着来，再怎么拼装都别扭。这堵墙我们下一章还会再撞一次：Swarm 自主协作需要智能体之间互相传递控制权，而这同样不是每家世界观里都有的东西。同一个设计哲学，划定了一个框架在多种模式上的边界。

### 4.5 支持程度与生态对照

&emsp;&emsp;五家在分层协同上的处境都摆出来了，照例并排放进一张对照表。沿用第一章 1.4 节那套本课内部归类口径——原生、官方示例级、可拼装、不适配，这是为横向对照自定的口径，不是某一家的官方术语。分层协同这一栏的分化，比前面任何一章都大：能不能让"每一层都自主派单"，直接把五家切成了两拨。

<div align=center><font size=2 color=#999999>五个框架对 Hierarchical 分层协同的支持程度与生态对照</font></div>

<div align="center">
<table width="80%">
<thead><tr>
<th>框架</th><th>出品方</th><th>本模式支持等级</th><th>模式实现原语</th><th>一句话点评</th>
</tr></thead>
<tbody>
<tr><td>LangGraph</td><td>LangChain</td><td>原生</td><td>create_supervisor 嵌套（主管套主管）</td><td>多层自主最完整，每层都是真主管</td></tr>
<tr><td>OpenAI Agents SDK</td><td>OpenAI</td><td>原生</td><td>嵌套 agents-as-tools</td><td>每层模型自主调用，极简但 token 开销大</td></tr>
<tr><td>CrewAI</td><td>CrewAI 公司</td><td>不适配（多层）</td><td>单层 hierarchical + 外层代码串 Crew</td><td>单层自主成立，跨层是代码模拟、非框架语义</td></tr>
<tr><td>MAF</td><td>Microsoft</td><td>不适配（多层）</td><td>as_agent + 代码固定调用顺序</td><td>无层级原语，写出来本质是流程编排</td></tr>
<tr><td>Claude Agent SDK</td><td>Anthropic</td><td>不适配</td><td>（subagent 不能 spawn subagent）</td><td>官方规则封死，两层到顶</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;这张表给出的判断和前两章很不一样：分层协同是四种模式里<font color=red>"框架方差最大"</font>的一种。Supervisor 那一章五家全员原生，差别只在控制粒度；到了分层协同这里，真正能原生、可递归地把两层以上自主委派搭出来的，只剩 LangGraph 和 OpenAI SDK 两家——其余三家最多原生支持单层的主管或团队，再往上叠就只能靠外层代码模拟，而那已经不是框架自己的分层语义。同一个需求，选不同的框架，要么顺水推舟，要么从一开始就在跟框架的世界观较劲。

&emsp;&emsp;所以分层协同的选型结论也最鲜明：**产出物复杂、需要分组分工时，首选 LangGraph 的嵌套主管**——每层都是真主管、结构本身就表达层级；如果团队已经在 OpenAI SDK 生态里，嵌套 agents-as-tools 也成立，代价是 token 开销和 `max_turns` 都要算足。而 CrewAI、MAF、Claude SDK 三家，在这个模式上不是"不好用"，而是压根不该往这儿用——真有多层组织的需求，换框架比硬拼更划算；它们各自的主场分别在角色化流水线、企业级单层编排和 orchestrator-worker。

&emsp;&emsp;本章我们把 Hierarchical 分层协同从机制讲到了五家的实现差异：它的本质是 Supervisor 的嵌套复合，上层只对组长说话、组长对组员负责，上下文逐层收窄，但工程上要两层封顶以控制错误归因和延迟的成本；真正能让"每一层都自主派单"的框架只有两家，其余三家的边界帮我们第一次正面体会到"框架世界观决定模式上限"这个选型判断的核心。到这里，前三种模式都有一个共同点——总有一个明确的上层在统筹，无论是流水线的固定顺序、Supervisor 的中央主管，还是层级里的总监。下一章，我们要把这个上层彻底拿掉，看看当智能体之间平起平坐、控制权可以互相直接交还时会发生什么——这就是 Swarm 自主协作。

## <center>第五章 Swarm 自主协作：三家原生与业界态度</center>

&emsp;&emsp;前三章的控制权都有人管着——第二章里是代码预先定死的固定顺序，第三章里是中央主管派单，第四章里是总监层层委派。它们有一个共同的骨架：总存在一个明确的上层，掌握着"下一步去哪"的全局决策权。这一章我们把这个上层彻底拿掉。当智能体之间平起平坐、谁都没有凌驾于谁之上时，"下一步交给谁"这件事由当前正在说话的那个智能体自己决定，并且它会把控制权真正地交出去、不再收回。这就是 Swarm 自主协作，它也是四种模式里业界争议最大、框架支持最参差的一种。

&emsp;&emsp;这一章的路径和前几章略有不同。机制讲完之后，我们会看到三家框架对它是原生支持、一家可拼装、一家直接不适配——这种五家拉开光谱的局面，本身就是 Swarm 这个模式特性的写照。讲完五家，我们还要专门花一节核证业界对 Swarm 的真实态度：它既被官方实测点出过明确的问题，又在客服这类场景里仍是无可替代的对的工具，这两面都要讲清楚。最后，作为 第二部分 的收官，我们用一张四模式乘五框架的全矩阵把整个第二部分收口。先看这一章的结构：一张没有中心的网，每个节点都能把控制权直接递给任何另一个节点。

<div align=center><font size=2 color=#999999>第五章：智能体连成无中心的网，控制权在平级节点之间直接传递</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L5-swarm-mesh-dbd2006d.png" width=80%></div>

<br>

&emsp;&emsp;接下来这一章按这样的顺序走：先讲清接力的核心机制 handoff 到底是什么、它需要系统记住什么状态、又有哪些容易翻车的地方；然后 OpenAI Agents SDK、LangGraph、MAF 三家原生实现各写一段能直接运行的客服转接代码，每个框架小节仍按"生态与支持程度、最小可跑代码、注意事项"三段展开；接着讲清 CrewAI 和 Claude SDK 这两家的边界；最后一节核证业界态度，并用全矩阵收口整个 第二部分。

### 5.1 接力机制：handoff 是一次特殊的工具调用

&emsp;&emsp;先把 Swarm 最核心的那个动作 handoff 讲透。第一章 1.3 节那张四模式地图里我们给过它一个名词解释——"没有中心，每个智能体都能通过一个'转交'动作把控制权交给另一个智能体"。当时只是点到为止，现在要把这个"转交"动作的底层机制拆开看。它的本质其实很朴素：<b>handoff 就是一次特殊的工具调用</b>。

&emsp;&emsp;回想我们在前面章节里反复见到的工具调用机制——智能体在它的工具列表里挑一个工具，填好参数调用它，拿到返回值再继续。handoff 复用的正是这套机制，只不过被调用的"工具"很特殊：它不是去查天气、查数据库，而是一个名叫 `transfer_to_X` 的转交工具，X 是另一个智能体的名字。当前智能体一旦调用了 `transfer_to_技术专员` 这件工具，框架就把对话的控制权从当前智能体移交给技术专员，由技术专员接着往下处理。这跟第三章 3.2 节讲 Supervisor 两种控制权形态时埋下的那个区别完全对上了：那里主管转移控制权后<font color=red>必定收回</font>，而这里控制权<b>交出去就不收回</b>了——技术专员接手后，下一步交给谁，由技术专员自己再决定，而不是回到某个中心。正因为没有中心，环上任何一个智能体都能转给任何另一个，控制权就这样在平级之间流转。

&emsp;&emsp;这种无中心的设计带来一个前几章不存在的新问题：系统必须记住"现在轮到谁在接待"。在 Supervisor 里这件事不用记，因为控制权总会回到主管，主管就是那个固定的锚点；但在 Swarm 里控制权可能停在网上任意一个节点，如果不专门记下来，下一轮用户再说话时，系统就不知道该把消息交给谁了。这个被记住的"当前接待者"，业界叫它 active_agent。

> <font size=2>【名词解释】<b><font color=red>active_agent</font>（active agent，当前活跃智能体）</b>：Swarm 模式里被系统记住的"当前正在接待用户的那个智能体"。因为 Swarm 没有中心锚点，控制权可能停在任意节点，必须把它单独记下来，多轮会话才能从上一轮的落点继续，而不是每轮都从头开始。</font>

&emsp;&emsp;active_agent 是 Swarm 能支撑多轮对话的关键。设想一个客服场景：用户第一句话被分诊台转给了退款专员，退款专员问"请提供订单号"，用户第二句话回了订单号——这第二句话必须直接送到退款专员手里，而不是又回到分诊台重新分诊一遍。系统能做到这一点，靠的就是它记住了"现在 active_agent 是退款专员"。哪个框架把这个状态记得牢、记得对，直接决定了它的 Swarm 实现能不能撑住真实的多轮客服，这一点我们在 5.3 节 LangGraph 那里会看到一个很具体的体现。

### 5.2 OpenAI Agents SDK：handoffs 是它的招牌

&emsp;&emsp;讲 Swarm 的实现，第一家必须是 OpenAI Agents SDK，因为 handoff 就是它的招牌。我们在第一章 1.3 节区分"Swarm 库和 Swarm 模式"时讲过，OpenAI 早年那个 Swarm 实验库已经由 Agents SDK 接棒；而接棒过来的，正是 handoff 这套接力机制——它是这个 SDK 仅有的四个核心原语（Agent、Runner、Tools、Handoffs）之一，一脉相承地保留了下来。对 Swarm 这种模式，OpenAI SDK 是<b>原生</b>支持，而且客服流转就是它官方文档里的第一个示例场景。换句话说，handoff 不是它顺手能做的事，而是它被设计出来时就摆在台面上的核心能力。

&emsp;&emsp;它的写法也极其直白，实现逻辑就两步：每个智能体定义时，用 `handoffs=[...]` 列出它能把控制权转交给哪些其它智能体，框架会自动为列表里的每一个目标生成一件 `transfer_to_X` 转交工具挂到它身上；然后 `Runner.run` 从某个智能体起跑，控制权就顺着这些转交工具流动。下面这段代码搭一个最小的客服转接场景：分诊台 triage 判断问题类型转给对应专员，退款专员 refund 和技术专员 tech 平级——其中 `refund.handoffs = [tech]` 这一行是点睛之笔，它让退款专员发现质量问题时能把控制权直接递给技术专员、不必回分诊台，这就是平级互转。通信方式正是第一章说的<b>控制权接力</b>：转交之后原专员退场，接棒者带着完整对话上下文继续接待。代码复用第二章 2.1 节环境准备 cell 里读出的 `MODEL`、`BASE_URL`、`KEY` 三个变量。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L5.2-openai-handoffs-f7de1b32.png" width=80%></div>

In [ ]:
import asyncio  # 驱动异步的 agent 调用
from openai import AsyncOpenAI  # OpenAI 官方异步客户端，指向 OpenRouter 兼容端点
from agents import Agent, Runner, OpenAIChatCompletionsModel, set_tracing_disabled  # OpenAI Agents SDK：Agent 智能体 / Runner 执行器 / 模型接兼容端点 / 关闭轨迹上报

set_tracing_disabled(True)                              # 走 OpenRouter 第三方端点，必须关掉回传 OpenAI 的追踪
client = AsyncOpenAI(base_url=BASE_URL, api_key=KEY)     # 复用环境准备 cell 的 BASE_URL/KEY
model = OpenAIChatCompletionsModel(model=MODEL, openai_client=client)

# name 必须英文：handoff 底层是 transfer_to_&lt;name&gt; 工具，中文名会撞名崩溃；中文角色说明放 instructions/handoff_description
refund = Agent(name="refund", model=model, handoff_description="处理退款申请",
               instructions="你是退款专员，处理退款。如果用户反映商品有质量问题、需要先做技术鉴定，转交给技术专员。一句话回复。")
tech = Agent(name="tech", model=model, handoff_description="商品质量技术鉴定",
             instructions="你是技术专员，负责商品质量技术鉴定，给出鉴定结论。一句话回复。")
triage = Agent(name="triage", model=model,
               instructions="你是客服分诊台，只负责判断用户问题类型并转给对应专员：退款相关转 refund，纯技术问题转 tech。不要自己回答业务问题。",
               handoffs=[refund, tech])                  # 分诊台能转给退款、技术两个专员
refund.handoffs = [tech]                                 # 退款专员也能把控制权再转给技术专员（平级互转，不经过分诊台）

async def main():
    # max_turns 是保险丝：万一两个专员互踢死循环，到上限强制停下
    result = await Runner.run(triage, "我买的手机有质量问题，想退款", max_turns=10)  # OpenAI Agents SDK 执行入口：跑一个 agent 直到产出
    print("【最终回复】", result.final_output)
    print("\n【接力轨迹】")
    for item in result.new_items:                        # 遍历运行产生的事件，打印 handoff 发生在哪些 agent 之间
        cls = item.__class__.__name__
        if "Handoff" in cls:                             # 一次控制权转交事件
            print(f"  - 发生 handoff：{cls}")
        elif cls == "MessageOutputItem":                 # 某个 agent 产出了一段回复
            who = getattr(item.agent, "name", "?")
            print(f"  - {who} 回复")

await main()       # Jupyter 里直接 await，不要再套 asyncio.run（环境准备 cell 已开 nest_asyncio）

&emsp;&emsp;运行后能从接力轨迹里看到控制权走过的完整路径：分诊台先判断、转给退款专员，退款专员发现是质量问题、又转给技术专员，最后由技术专员给出鉴定结论——整段两次 handoff，没有任何一次回到中心。运行产出如下：

```text
【最终回复】 请提供手机具体型号及故障现象。

【接力轨迹】
  - triage 回复
  - 发生 handoff：HandoffCallItem
  - 发生 handoff：HandoffOutputItem
  - refund 回复
  - 发生 handoff：HandoffCallItem
  - 发生 handoff：HandoffOutputItem
  - tech 回复
```

&emsp;&emsp;轨迹清清楚楚地呈现了两次平级互转：triage → refund → tech。控制权从分诊台一路递到技术专员手里，技术专员接手后并没有把控制权交回给谁，而是直接给出了最终结论——这正是 Swarm "交出去不收回"的特征。最终回复也确实出自技术专员的口径（"经鉴定……出厂硬件故障"），而不是分诊台或退款专员。

### 5.3 LangGraph：langgraph-swarm 官方双件套之一

&emsp;&emsp;第二章我们认识了 LangGraph 的图编排底座，第三章它换上 `langgraph-supervisor` 做集中调度。到了 Swarm 这里，它又有一个专门的官方扩展包 `langgraph-swarm`，和 `langgraph-supervisor` 并列，是 LangGraph 官方多智能体扩展的双件套。对 Swarm 模式，LangGraph 是<b>原生</b>支持：这个扩展包内建了 active_agent 状态机和 handoff 工具的生成逻辑，我们不用自己手写"记住当前接待者"的代码，框架替我们管好了。

&emsp;&emsp;要注意 `langgraph-swarm` 不在 LangGraph 主包里，是一个独立的扩展包——课件环境的依赖清单里已经包含它，无需另外安装。

&emsp;&emsp;装好之后看代码。下面这段用 `langgraph-swarm` 实现同一个客服转接，实现逻辑分三步：先用 `create_handoff_tool(agent_name=...)` 给退款专员造一件"转给技术专员"的工具、给技术专员对称地造一件"转回退款专员"的，两件工具把双向互联建起来；再用 `create_react_agent` 把两个专员各建成一个挂着转交工具的智能体；最后 `create_swarm` 把它们装配成一个 Swarm、用 `default_active_agent` 指定初始接待者，`.compile(checkpointer=...)` 配上存储——这一步很关键，active_agent（当前由谁接待）就是靠 checkpointer 记住的，没有它跨轮就忘了谁在接待。通信方式是<b>控制权接力</b>，和上一节 OpenAI SDK 同一范式，差别在于 LangGraph 把"记住接待者"做成了显式的状态机。代码同样复用第二章环境准备 cell 的三个变量。

> <font size=2>【名词解释】<b><font color=red>create_swarm / create_handoff_tool</font></b>：`langgraph-swarm` 扩展包的两个核心构造。`create_handoff_tool(agent_name=...)` 生成一件"转交给指定智能体"的工具；`create_swarm(agents, default_active_agent)` 把一组挂好了转交工具的智能体装配成一个 Swarm，并指定初始由谁接待。两者配合，就把"无中心接力 + 记住当前接待者"这套机制现成地搭好了。</font>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L5.3-langgraph-swarm-f083524c.png" width=80%></div>

In [ ]:
from langchain_openai import ChatOpenAI  # LangChain 封装的 OpenAI 兼容聊天模型，LangGraph 节点里用它调 LLM
from langgraph.prebuilt import create_react_agent  # LangGraph 预制件：造一个会自主推理、按需调工具的 ReAct 智能体
from langgraph_swarm import create_swarm, create_handoff_tool  # langgraph-swarm 扩展：create_swarm 建去中心化群组 / create_handoff_tool 造平级转交工具
from langgraph.checkpoint.memory import InMemorySaver  # 把图执行状态存进内存的 checkpointer，支持 interrupt 暂停后恢复

# 复用环境准备 cell 的 MODEL/BASE_URL/KEY
model = ChatOpenAI(model=MODEL, base_url=BASE_URL, api_key=KEY, temperature=0.3, timeout=180, max_retries=1)

# 每个专员挂一件 handoff 工具，能把控制权转给另一个专员（平级，无中心）
refund = create_react_agent(model, name="refund",  # 造一个会自主推理、按需调工具的 ReAct 智能体
    tools=[create_handoff_tool(agent_name="tech", description="商品有质量问题、需技术鉴定时转给技术专员")],  # 造一个"把控制权转交给某 agent"的自主协作工具
    prompt="你是退款专员，处理退款。商品质量问题需要技术鉴定时转给 tech。一句话回复。")
tech = create_react_agent(model, name="tech",
    tools=[create_handoff_tool(agent_name="refund", description="鉴定完转回退款专员")],
    prompt="你是技术专员，做商品质量技术鉴定并给结论。一句话回复。")

checkpointer = InMemorySaver()                           # swarm 几乎必配：记住 active_agent，否则跨轮丢"现在轮到谁"
# create_swarm 把两个专员装配成互联群组，default_active_agent 指定初始接待者
swarm = create_swarm(agents=[refund, tech], default_active_agent="refund").compile(checkpointer=checkpointer)  # 编译图并挂上 checkpointer，状态才能存档、之后恢复

config = {"configurable": {"thread_id": "1"}}            # thread_id 标识一次会话，checkpointer 按它存取 active_agent
# 同 3.2：stream 流式打印接力轨迹的真实时间线
seen = set()
for _, state in swarm.stream(
        {"messages": [{"role": "user", "content": "我的手机有质量问题，想退款"}]},
        config, subgraphs=True, stream_mode="values"):
    for m in state.get("messages", []):
        if m.id in seen: continue
        seen.add(m.id)
        who = getattr(m, "name", None) or m.type
        content = m.content if isinstance(m.content, str) else str(m.content)
        if content.strip():
            print(f"[{who}] {content[:160]}")

&emsp;&emsp;运行后能看到控制权从退款专员转给了技术专员，整个接力被 checkpointer 串成一条连续的会话：

```text
[human] 我的手机有质量问题，想退款
[refund] 我理解您对手机质量问题的困扰。由于质量问题需要技术鉴定，我这就为您转接技术专员处理，请稍等。
[transfer_to_tech] Successfully transferred to tech
[tech] 您好，我是技术专员，负责商品质量技术鉴定。为了更好地帮您处理，请提供以下信息：
1. 手机品牌型号
2. 购买时间及渠道
3. 具体质量问题/故障现象（如无法开机、屏幕异常、通话故障等）
4. 故障是购买时就存在，还是使用后出现的？
请简要描述，我鉴定后即刻给出结论。
```

&emsp;&emsp;轨迹里那行 `[transfer_to_tech] Successfully transferred to tech` 就是 handoff 工具被真正调用的证据——退款专员不是口头说说要转接，而是实打实地调用了转交工具，控制权才真的落到了技术专员手里。这也正好对照了 5.1 节那个"口头转接不真转"的坑：这里因为提示词和工具配置都到位，转接动作落实成了一次工具调用。

### 5.4 MAF：HandoffBuilder 与一个不直观的必填参数

&emsp;&emsp;第三家原生支持是 MAF。它在 `agent-framework-orchestrations` 子包里提供了一个专门的 `HandoffBuilder`，是它五种官方编排模式里负责接力的那一种。对 Swarm 模式，MAF 是<b>原生</b>支持：`HandoffBuilder` 会自动给每个参与者注入转交工具，把它们组织成一个 mesh 结构——所谓 mesh，就是参与者之间互相可达、彼此广播上下文，这和它自家 `GroupChatBuilder` 那种"所有人围着一个中央协调者发言"的星形结构是两种不同的结构。

> <font size=2>【名词解释】<b><font color=red>HandoffBuilder</font></b>：MAF 为 handoff 接力提供的原生构造器。给它一组参与者智能体、指定一个起始智能体，调用 `.build()` 就得到一个能在参与者之间转交控制权的工作流。它内部自动注入 handoff 工具、组织成参与者互相可达的 mesh 结构。</font>

&emsp;&emsp;不过用 `HandoffBuilder` 之前，要先讲清一个在本课实测里踩到的必填参数坑，否则代码连 `build()` 这一关都过不了。MAF 要求 handoff 工作流里<b>每一个</b>参与者都必须开启 `require_per_service_call_history_persistence=True` 这个设置——它的作用是保证 handoff 短路切换时，本地维护的对话历史和服务端的状态保持一致。如果漏掉哪怕一个参与者没设，`build()` 会直接抛出 `ValueError`，报错信息明确写着"Handoff workflows require all participant agents to have this setting"（handoff 工作流要求所有参与者智能体都开启此设置）。下面这段代码实现同一个客服转接，逻辑分三步：先用 `as_agent` 建分诊台、退款、技术三个参与者（每个都带上那个必填参数）；再把它们交给 `HandoffBuilder`，用 `.with_start_agent(triage)` 指定从分诊台起步、`.build()` 装配成工作流——mesh 结构和各参与者身上的 handoff 工具都由 builder 自动注入，不用我们手工造；最后 `wf.run` 一跑，控制权从分诊台起、顺着接力流动。通信方式同样是<b>控制权接力</b>，这是三家原生里框架包办得最多的一家。它同样复用环境准备 cell 的三个变量。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L5.4-maf-handoffbuilder-c35c3e51.png" width=80%></div>

In [ ]:
import asyncio  # 驱动异步的 agent 调用
from agent_framework.openai import OpenAIChatClient  # Microsoft Agent Framework 聊天客户端，as_agent 能把它变成一个 agent
from agent_framework.orchestrations import HandoffBuilder  # MAF 的 Swarm 原生构造器：mesh 拓扑、框架自动注入 handoff 工具

client = OpenAIChatClient(model=MODEL, api_key=KEY, base_url=BASE_URL)   # 复用环境准备 cell 的变量

# 三个参与者：必填 require_per_service_call_history_persistence=True，漏一个 build() 就 ValueError
triage = client.as_agent(name="triage", instructions="你是客服分诊台，判断用户问题类型，转给 refund 或 tech。",  # 把 MAF 客户端封成一个具名 agent
                         require_per_service_call_history_persistence=True)
refund = client.as_agent(name="refund", instructions="你是退款专员，处理退款；商品质量问题需技术鉴定时转给 tech。一句话回复。",
                         require_per_service_call_history_persistence=True)
tech = client.as_agent(name="tech", instructions="你是技术专员，做商品质量技术鉴定并给结论。一句话回复。",
                       require_per_service_call_history_persistence=True)

async def main():
    # HandoffBuilder 自动给三个参与者注入转交工具，with_start_agent 指定从分诊台起步
    wf = (HandoffBuilder(participants=[triage, refund, tech])  # MAF 原生 Swarm：自动给参与者注入 handoff 工具
          .with_start_agent(triage)
          .build())
    result = await wf.run("我的手机有质量问题，想退款")
    outputs = result.get_outputs()                       # 取出接力过程中各参与者的产出
    print(f"【输出数量】{len(outputs)}")
    for o in outputs:
        print("·", getattr(o, "text", o))

await main()       # Jupyter 里直接 await，不要再套 asyncio.run

&emsp;&emsp;运行时控制权从分诊台起，依次接力到退款专员、技术专员。运行产出如下：

```text
【输出数量】3
· 根据您的问题，您需要处理手机质量问题的退款申请，我将为您转接到退款客服团队。

· 您好，关于手机的质量问题退款，需要进行技术鉴定。我将为您转接技术人员处理。

· 经初步了解，您反映手机存在质量问题。请您具体描述一下出现的故障现象（如无法开机、屏幕异常、电池问题等），以便我为您进行技术鉴定并给出结论。
```

&emsp;&emsp;三段输出分别出自分诊台、退款专员、技术专员，控制权一路接力下来，最终由技术专员给出了鉴定结论。这里有一个对照值得专门点出来，它能帮我们更准确地理解 MAF 这个框架。

> <b>【提示】MAF 的高层 builder 不是全盘不可用，要分 builder 看</b>：我们在第三章 3.5 节讲 MAF 做 Supervisor 时遇到过一次挫折——`GroupChatBuilder` 这类高层编排构造器配 OpenRouter 加 deepseek 模型时真的会崩，那是因为群聊编排的发言人选择逻辑（selection）在第三方端点加非原生模型上跑得不稳，模型自主派单容易发散，最后只能退回更底层的 `as_agent` 用代码兜底。但 `HandoffBuilder` 这一节的情况和那次<b>本质不同</b>：它跑通了，而且就是原生跑通的，没有崩。它一开始过不去的那一关，不是模型行为不稳，而仅仅是缺了 `require_per_service_call_history_persistence` 这个必填参数——这是一个纯粹的 API 配置问题，参数补上，`build()` 就过、工作流就正常接力。两件事放一起看，结论就清楚了：<b>不能因为 GroupChat 那次崩，就给 MAF 的高层 builder 全部判死刑</b>。GroupChat 的问题在模型自主决策的稳定性，HandoffBuilder 的问题只在一个不直观的必填参数；前者要换底层写法兜底，后者补个参数即可原生用。选型时对一个框架的判断要落到具体 builder 上，而不是凭一次踩坑就一刀切地否定整个高层编排能力。

### 5.5 CrewAI 与 Claude Agent SDK 的边界

&emsp;&emsp;三家原生讲完，剩下两家在 Swarm 这个模式上都站在边界之外，但站的位置不一样：CrewAI 是<font color=red>可拼装</font>，Claude SDK 是<b>不适配</b>。这两家不写代码，把边界讲清楚就够了——而恰恰是这两条边界，藏着这一章最值得带走的选型判断。

&emsp;&emsp;先说 CrewAI。它在 Swarm 上是可拼装，意思是它没有对等的接力原语，但勉强能用别的东西凑出类似效果。根子还在它的世界观上——我们在第二章 2.3 节认识 CrewAI 时讲过，它把多智能体系统看成"一支有不同角色的团队"，关心的是角色分工和流程编排。在这个世界观里，根本就没有"控制权在平级之间漂移"这个概念：角色是被安排好职责的，流程是被定义好走向的，不存在"一个角色临场决定把对话交给另一个角色、之后再也不收回"这种事。它的 Flows 能用事件链模拟出顺序的控制权转移，但那是"A 做完触发 B"的单向流转，不是 Swarm 那种"谁都能转给谁、转给谁由当前节点临场定"的平级互转。两者形似而神不同。所以本课对 CrewAI 在 Swarm 上的处理是：讲清它的边界即可，不去硬拼一个别扭的接力出来。这跟它在第四章做分层协同时"单层自主成立、跨层只能代码模拟"是同一类情况——它的强项在角色化的流程，不在控制权的自由流转。

&emsp;&emsp;再说 Claude Agent SDK，它在 Swarm 上是不适配。这是我们在这门课里<b>第二次</b>撞上同一堵墙。第四章 4.4 节讲做不到的三家时，我们已经因为"subagent 不能再 spawn subagent"这条官方明文限制，认定它做不了多分层协同。现在 Swarm 又要求一件事：智能体之间要能互相 handoff、平级传递控制权。可 Claude SDK 的 subagent 既不能创建下一层 subagent，也不能在平级之间互相调度——它的结构里只有"主智能体 dispatch 一组 subagent"这一条单向通路，subagent 之间彼此根本不可见，更谈不上互相转交控制权。我们把那条官方原文再请出来看一遍：

> <font size=2>官方文档原文（来源 code.claude.com/docs/en/sub-agents）：<b>"Subagents cannot spawn other subagents."</b>（子智能体不能创建其它子智能体。）官方给出的替代方向是：如果工作流需要嵌套委派，改用 Skills，或从主对话里串联多个子智能体。</font>

&emsp;&emsp;这条限制在第四章挡死了"组长往下派工程师"的多层嵌套，在这里又挡死了"专员之间互相 handoff"的自主协作——同一条规则，连拦两种模式。Anthropic 官方实验性的方向里有一个叫 Agent Teams 的东西，让 teammate 之间能互发消息，看上去更接近 Swarm 的味道，但它不是标准 SDK 的能力、还在实验阶段，这里提一句现状，不展开。真正要带走的，是这堵墙背后的那个判断。

> <b>【提示】框架世界观决定模式上限——这是整个 第二部分 最该带走的选型判断</b>：Claude SDK 在分层协同和 Swarm 上连续两次不适配，根子是同一个：它的世界观是"一主多仆的扁平结构"——一个强大的主智能体，编排一组各有专长但彼此平级、互不可见的 subagent。Anthropic 把整个框架的复杂度都押在了"把单个 subagent 做强 + 主智能体一层编排"上，刻意不打开"subagent 互相嵌套、互相调度"这个口子。这不是它能力弱，而是它的设计哲学根本不往那个方向走。把这个观察推广开，就是这门课 第二部分 最核心的一条选型方法：<b>一个框架支持哪些协作模式，根子上是由它的设计世界观决定的，而不是功能多寡</b>。LangGraph 把世界看成可组合的图，所以主管嵌套、状态机接力对它都天然成立，层级和 Swarm 它都原生支持；Claude SDK 把世界看成一主多仆，所以多层嵌套和平级互派对它都天然不成立。选型时，与其逐个去查"这个框架有没有 X 功能"，不如先想清楚"这个框架的世界观是什么、它把复杂度押在了哪里"——世界观对上了模式自然顺，世界观拧着来再怎么拼装都别扭。第四章那个伏笔，到这里就完整收回来了：同一个设计哲学，划定了它在两种模式上的边界，这正是选型判断力最干净的一个样本。

### 5.6 业界态度与 第二部分 全景

&emsp;&emsp;三家原生跑通、两家边界讲清，Swarm 这种模式还剩一个绕不开的问题要交代：业界对它其实是有保留的。这不是看衰，而是有实测数据支撑的冷静判断，收尾时值得讲清楚。

&emsp;&emsp;LangChain 在它的多智能体架构基准测试里点出了 Swarm 的两个软肋。一是可扩展性——每个智能体都得知道所有其他智能体的存在、各自挂上能转给所有人的 handoff 工具，智能体一多，这套交接关系就成了平方级的负担。二是控制流——handoff 必须顺序执行，一个智能体在把控制权交出去之前没法并行调用多个工具。所以业界的普遍倾向是：默认先上集中调度，它对子智能体的假设最少、最容易控制；Swarm 退为特定场景才用的解，而不是首选。

&emsp;&emsp;但这份保留态度，针对的是<font color=red>拿 Swarm 去做任务编排</font>那种用法。换个场景，Swarm 依然是最贴合的工具——对话焦点随用户漂移的业务，比如客服、咨询，用户问着物流问着就要退款，这种"会话跟着人走"的流转，正是自主协作的主场，硬套集中调度反而别扭。第十章我们会用一个完整的客服项目证明这一点。还要补一句：Anthropic 自己的研究系统选了集中调度结构，这常被拿来当"集中调度更好"的论据，但那是 Anthropic 在它那个特定研究任务上的工程选择，官方并没有发话说"推荐集中调度、不推荐 Swarm"——别把一次工程实践当成官方定论。

&emsp;&emsp;讲到这里，第二部分 的五个框架在四种模式上的支持程度就全摸清了。把它们汇成一张矩阵，正好给 第二部分 收个尾。

<div align=center><font size=2 color=#999999>第二部分 全景：四种协作模式 × 五个框架的支持程度矩阵</font></div>

<div align="center">
<table width="92%">
<thead><tr>
<th>框架</th><th>Workflow 流程编排</th><th>Supervisor 集中调度</th><th>Hierarchical 分层协同</th><th>Swarm 自主协作</th>
</tr></thead>
<tbody>
<tr><td>LangGraph</td><td>原生（StateGraph 串行边）</td><td>原生（create_supervisor）</td><td><b>原生</b>（主管嵌套）</td><td>原生（langgraph-swarm）</td></tr>
<tr><td>CrewAI</td><td>原生（Process.sequential）</td><td>原生（Process.hierarchical）</td><td>不适配·多层（跨 Crew 代码模拟）</td><td>可拼装（Flows 形似神不同）</td></tr>
<tr><td>OpenAI Agents SDK</td><td>官方示例级（代码接力）</td><td>原生（agents-as-tools）</td><td><b>原生</b>（嵌套 as_tool）</td><td>原生（handoffs 招牌）</td></tr>
<tr><td>MAF</td><td>原生（SequentialBuilder）</td><td>原生（as_agent + 确定性兜底）</td><td>不适配·多层（无层级原语）</td><td>原生（HandoffBuilder）</td></tr>
<tr><td>Claude Agent SDK</td><td>官方示例级（串行 query）</td><td>原生（subagents 看家）</td><td>不适配（不能 spawn）</td><td>不适配（平级互不可见）</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;这张矩阵把每个框架的"性格"摊在了明面上。LangGraph 全行深蓝——它把世界看成可组合的图，主管能套主管、状态机能接力，四种模式它都原生接得住，是覆盖最全的一家。OpenAI SDK 紧随其后，强项是 handoff、集中调度和嵌套 agents-as-tools 拼出的多层自主，极简四原语让它的代码在这几块都最短。CrewAI 在流水线和集中调度上原生最顺，但到了多层组织就出了它的语义边界——跨 Crew 只能代码模拟，自主协作同样不在它的角色分工世界观里。MAF 在流水线和集中调度、handoff 上都够得着，但层级没有原语、做出来本质是流程编排，配第三方端点时高层原语也要多留个心眼。Claude SDK 则在集中调度上是看家本领，碰到层级和 Swarm 就因为"一主多仆"的世界观直接亮红灯。一句话总结：没有哪个框架在所有模式上都最好，偏科是设计哲学的必然，选型的第一步是看清手上的任务落在矩阵的哪一行，再挑那一行里最顺手的那一列。

&emsp;&emsp;到这里，第二部分 就走完了。我们把四种协作模式逐一拆开，每一种都用五个框架各写了一段能跑的最小代码，也摸清了每个框架的脾气和边界。但有一个最实际的问题还没正面回答：拿到一个真实的业务需求，到底该选哪一种模式、配哪一个框架？接下来的第六章，我们就专门把这套选型判断力立起来——先问"要不要上多智能体"，再判断"该落在哪种模式"，然后正式进入 第三部分，用四个完整的项目把这四种模式一一落到地。

## <center>第六章 场景选型：什么业务用什么模式</center>

&emsp;&emsp;前面五章我们把零件攒齐了。第二到第五章逐一拆开了四种协作模式，每一种都用五个框架各写了一段能跑的最小代码，也摸清了每个框架的脾气和边界。可零件再齐，到了真实项目里还得回答一个绕不开的问题：手上这个业务需求，到底该不该上多智能体？如果上，又该落在哪一种模式、配哪一个框架？这一章不写代码，专门把这套选型判断力立起来。

&emsp;&emsp;这一章是从 第二部分 的"模式怎么实现"通向 第三部分 的"项目怎么落地"的那座桥。我们分三步走：先问"要不要上多智能体"，这是一道在立项前就该想清楚的成本题；再把四种模式的适用边界一条一条划清楚，收口成一棵一眼能查的决策树；最后预览接下来四章要逐一落地的四个完整项目。走完这一章，再拿到任何一个业务需求，我们心里都该有一杆秤：先掂量值不值得拆，再判断拆成哪种结构。

### 6.1 先问要不要上多智能体

&emsp;&emsp;选型的第一步不是"选哪种模式"，而是先问一句更靠前的问题：这个任务到底<font color=red>要不要上多智能体</font>？多智能体不是越多越好的银弹，业界对"该不该拆"这件事，甚至有过一场公开的对垒。把这场对垒的两边论点摆清楚，是建立选型判断力的第一块基石。

&emsp;&emsp;一边是 Cognition（Devin 背后的团队），他们写过一篇标题就很直白的文章，叫《Don't Build Multi-Agents》（别搭多智能体）。他们的核心论点是：多个子智能体并行干活时，每个子智能体只看得到自己那一小块上下文，各自独立做决策，最后产出往往对不齐、拼不拢。他们举的典型场景是让两个子智能体各做一部分活——一个负责这块、一个负责那块，结果两边对同一件事的理解和风格不一致，合起来反而是个四不像。他们由此主张：能用单线程把上下文工程做扎实，就别轻易拆成多智能体，因为拆开之后协调一致性的代价，常常比拆开省下的那点并行时间更贵。

&emsp;&emsp;另一边是 Anthropic，他们公开了自己那套多智能体研究系统的工程实践，给出的是相反方向的实测证据。我们在第一章讲过那两个数字：他们用 Claude Opus 4 当主管、Claude Sonnet 4 当下属的研究系统，在内部研究评测上比单体 Claude Opus 4 高出 90.2%；代价是消耗的 token 大约是普通聊天交互的 15 倍。这里两个限定语第一章已经强调过，本节不重复推导，重点放在这两组看似相反的结论为什么其实不打架。

&emsp;&emsp;两边看似针锋相对，落到任务性质上其实指向同一条判据：<b>读写冲突强的任务慎拆，读密集、可并行的任务受益最大</b>。Cognition 警惕的，是那种子任务之间要互相写入、彼此结果要严丝合缝拼在一起的活——这类任务一拆，子智能体各写各的，对齐成本极高，正是它说的"别拆"。而 Anthropic 的研究系统之所以受益巨大，恰恰因为研究是典型的读密集、可并行任务：多个子智能体各自去检索一片信息源，彼此独立、互不写入对方的产出，最后由主管汇总即可。读取可以无冲突地并行铺开，所以拆开换来的速度和覆盖面，远超那 15 倍 token 的代价。所以这场对垒不必选边站——它真正给我们的，是一把判断尺：手上的任务是"子任务要互相对齐着写"，还是"子任务能各读各的并行干"？前者倾向别拆，后者才是多智能体的主场。

> <font size=2>【提示】这条判据怎么用——先看子任务之间是"读"还是"写"的关系。如果几个子任务要往同一个产出物上协同写入、彼此风格和结论必须一致（比如合写一份代码、合画一套界面），那是读写冲突强，拆开后对齐成本高，倾向单线程做扎实；如果几个子任务只是各自读取、各自检索、最后汇总（比如多源调研、并行检查），那是读密集可并行，拆成多智能体收益最大。</font>

&emsp;&emsp;就算判断下来"确实该上多智能体"，也还有一个更稳妥的推进顺序，我们叫它升级阶梯。它和第一章那条自主性谱系是一脉相承的：不要一上来就奔着最复杂的多智能体结构去，而是<b>先把单智能体的提示词优化到位 → 优化到头还不够，再给它加工具 → 工具也加满、确实出现了上下文真的溢出、或者任务真的需要并行同时干几件事，才拆成多智能体</b>。每往上爬一级，灵活性是增加了，但成本和调试难度也跟着涨。这条"能简单就别上复杂"的原则，我们在第二章讲 Workflow 时已经见过它的一个具体体现——能用确定性流水线解决的，就不必动用自主编排。

### 6.2 四模式的适用边界

&emsp;&emsp;过了"要不要上"这一关，确定了该上多智能体，下一个问题就是该落到哪一种模式。第二部分 我们已经把四种模式的实现都摸透了，这一节换个角度，不讲怎么实现，只讲每一种模式分别适合什么业务、不适合什么业务，最后收口成一棵一眼能查的决策树。每个模式除了"适合什么"，更要紧的是它的<font color=red>硬边界</font>在哪——什么信号出现说明这个模式撑不住了，以及它在 2026 年的实际热度，毕竟这一年多智能体的主流态势已经收敛得相当清楚。下面先用一张表把四组边界和当前定位一览，再逐个展开。需要先说明一句：网上流传的不少"模式适用业务对照表"把话说得偏粗（比如笼统地讲某模式"部分支持""可无限扩展"），下面这些边界是结合 2026 年业界的生产实践、本课五个框架的真实支持矩阵和四个案例的实测，对那类粗略说法做过修正后的结论。

<div align=center><font size=2 color=#999999>四模式适用边界一览：典型业务、硬边界与一问定位</font></div>
<div align="center">
<table width="92%">
<thead><tr><th>模式</th><th>典型业务</th><th>硬边界</th><th>一问定位</th><th>2026 当前定位</th></tr></thead>
<tbody>
<tr><td>Workflow 流程编排</td><td>工序固定、每步依赖上一步</td><td>不会临场变通</td><td>工序固定吗？</td><td>能用就优先，最稳最省</td></tr>
<tr><td>Supervisor 集中调度</td><td>大多数需拆解加质检的开放任务</td><td>主管是上下文与吞吐瓶颈</td><td>要中央把关吗？</td><td>主流默认，约七成部署</td></tr>
<tr><td>Hierarchical 分层协同</td><td>非常复杂庞大的项目</td><td>深度，两层封顶</td><td>团队要分层吗？</td><td>仅超大项目才需要</td></tr>
<tr><td>Swarm 自主协作</td><td>没明确工作流、焦点随用户漂移</td><td>边界最多</td><td>对话要跟人走吗？</td><td>早期模式，现已非主流</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;<b>Workflow 流程编排</b>，适合每一步都依赖上一步产出、流程事先就能定型的活——报表生成、文档处理、内容生产、审批流转都是典型。它是四种模式里最快、最稳、也最便宜的：控制流固定在代码里，不多花一个 token 在"决定下一步干什么"上，出了问题一眼能看出卡在哪道工序。所以选型的第一优先永远是它——能用流程编排解决的，别上更复杂的。它的硬边界也只有一条：<font color=red>不会临场变通</font>。一旦流程需要随输入临时改道、需要模型在运行时决定走哪条路，固定的边就成了枷锁，这时候才该往后面三种模式走。顺带修正一个常见的偏粗说法：有些对照表讲主流框架对流程编排只是"部分支持"，并不准确——本课实测里 CrewAI、LangGraph、MAF 三家原生支持，OpenAI SDK 和 Claude SDK 也能用官方示例级的手写代码干净实现。

&emsp;&emsp;<b>Supervisor 集中调度</b>，是 2026 年业界公认的<font color=red>生产默认</font>——绝大多数场景拿不准用什么，先上它都不会错。适合的是能拆成几个相对独立的子任务、交给专家并行干、再由主管收口质检的活：开放问题的研究、多源信息汇总、带审核环节的生成任务都是主场。它好调试（轨迹都在主管手里）、好控制（路由逻辑清晰），五个框架全部原生支持；把它和分层协同合起来的 orchestrator-worker 家族，约占 2026 年生产级多智能体部署的七成，是当下绝对的主流。还要专门点出它现在最受认可的落地形态——<b>supervisor-as-tool</b>，也就是把每个专家 agent 当成主管手里的一件工具来调（即第三章 3.3 讲的 agents-as-tools）。2026 年 LangChain 这类主流框架明显从"独立的 supervisor 库"转向了这种"专家即工具"的写法，因为它路由更可控、跨框架兼容也更好。但 Supervisor 也有两条实打实的硬边界。一是<b>主管是上下文瓶颈</b>：任务说明、所有专家的返回结果、综合所需的上下文，全要装进主管一个脑子，中间结果一多——业界实测几十份中间产出就能把 128K 的上下文窗口顶爆——主管就先撑不住了。二是<b>主管是吞吐瓶颈</b>：高度并行的活每一份结果都要路由回主管，派单收口的来回本身就是延迟大头，并行度越高，这个中心节点越拖后腿。子任务之间要频繁互相传递、对话焦点随过程漂移的场景，也不是它的主场。

&emsp;&emsp;<b>Hierarchical 分层协同</b>，只在<font color=red>非常复杂庞大的项目</font>里才真正需要——比如协作构建一个架构很大的系统、或通读处理超长的资料，单层 Supervisor 一个主管直接管十几个工人必乱，这才需要分层：技术总监管组长、组长管工程师。换句话说它不是日常默认，多数场景用 Supervisor 就够了，只有任务大到一层主管扛不住、必须再分一层组织时才升上来。它的硬边界正是<b>深度</b>。业界实践里"主管加工人"两层就覆盖了绝大多数生产场景；真要建第三层之前，先狠狠问一句——是真需要一层新组织，还是主管层的状态管理没做好？层级每加深一层，错误归因的难度都在指数上涨，到底是哪一层、哪个组、哪个工人出的错，排查成本会迅速失控。还有一条来自生产事故的教训：多层委派必须配上<b>交接计数和递归护栏</b>，给委派的层数和次数设硬上限——没有护栏的多层委派结构，等于一起还没发生的生产事故。

&emsp;&emsp;<b>Swarm 自主协作</b>，要先把定位讲清楚：它是<font color=red>多智能体早期提出的一种协作模式，到 2026 年其实已经不是主流了</font>。最直接的信号是，OpenAI 当初那个开创 handoff 风格的 Swarm 框架已经官方停更归档、明确建议改用 Agents SDK；业界整体也从这种平级去中心的协作，明显倒向了中心化的 Supervisor。它仍有自己适配的场景——没有明确工作流、对话焦点高度随用户漂移的业务，典型就是客服转接：用户问着物流忽然要退款，对话主导权在专员之间平级流转，第十章的客服案例就是这种情形。但它的边界也是四种模式里最多的：项目早期不该上它，业界共识是先用 Supervisor 跑起来、等数据证明瓶颈确实在主管来回再考虑切换；它没有中心节点，一次任务的轨迹散在各专员日志里，可观测性差、排障要重建接力链；成本也容易失控，handoff 型的多域任务一跑就是七八次模型调用、上万 token；加上第五章讲过的 LangChain 基准——交接关系随智能体数量平方级增长、还只能顺序执行。所以拿它去做有明确流程的任务编排是公认的反模式，那种场景回到 Supervisor；Claude Agent SDK 因为"一主多仆"的世界观，对它也明确不适配。一句话：Swarm 值得理解，但只在它那块窄场景里才该用。把上面这四组边界提炼成四个问题顺序问下来，就能落到对应的模式——<font color=red>工序固定</font>就是 Workflow，<font color=red>要中央把关</font>就是 Supervisor，<font color=red>团队要分层</font>就是 Hierarchical，<font color=red>对话跟人走</font>就是 Swarm。下面这棵决策树把这四问和四个落点画在了一起，可以当成选型时的速查图。

<div align=center><font size=2 color=#999999>四模式选型决策树：从业务需求出发，四个问题落到四种模式</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L6-decision-tree-abc20dbf.png" width=80%></div>

<br>

&emsp;&emsp;有了这棵决策树，再补一条更高层、也最贴合 2026 年现状的总原则：<font color=red>拿不准时，默认先上 Supervisor</font>。它对子智能体的假设最少、最容易控制、也最通用，又是当下绝对的主流默认，单智能体不够用时第一个该试的就是它（落地多用 supervisor-as-tool 那种"专家即工具"的形态）。在这个默认之上做减法和特化——工序其实固定、根本不需要模型临场决策，就降到更简单更稳的 Workflow；任务复杂庞大到一个主管扛不住，才升到 Hierarchical，且记住两层封顶；只有业务本质就是没有固定流程、对话焦点完全跟着人走，才横切到 Swarm。一句话收束：以 Supervisor 为默认中枢，按任务性质向 Workflow、Hierarchical 两侧特化，Swarm 留给那块窄场景，选型就八九不离十了。

### 6.3 四个案例项目预览

&emsp;&emsp;选型判断力立起来了，接下来就该把它落到真实项目里检验。从第七章开始的 第三部分，我们会用四个完整的项目，把四种模式一一变成能跑、能看的东西。在动手之前，先用一张总表把这四个项目预览一遍，让我们对接下来四章的走向心里有数。

&emsp;&emsp;这四个项目的安排，正好沿着我们刚立起来的选型判断走了一遍：每一个项目都对应一种协作模式，模式与框架的搭配也都和前面五章得出的选型结论吻合——工序固定的报表流水线用 CrewAI 做 Workflow、开放并行的研究编排做 Supervisor、复杂分层的软件交付用 LangGraph 做 Hierarchical、对话跟人走的在线客服用 OpenAI SDK 做 Swarm。下面这张表把四个项目的结构、框架和端口列在一起。

<div align=center><font size=2 color=#999999>第三部分 四个实战项目预览：案例 × 协作结构 × 框架 × 端口</font></div>

<div align="center">
<table width="80%">
<thead><tr>
<th>对应章</th><th>案例项目</th><th>协作结构</th><th>框架（版本）</th><th>服务端口</th>
</tr></thead>
<tbody>
<tr><td>第七章</td><td>销售数据报表流水线</td><td>Workflow 流程编排</td><td>CrewAI 1.14.6</td><td>8091</td></tr>
<tr><td>第八章</td><td>多轮研究编排</td><td>Supervisor 集中调度</td><td>Microsoft Agent Framework 1.8</td><td>8088</td></tr>
<tr><td>第九章</td><td>软件交付</td><td>Hierarchical 分层协同</td><td>LangGraph 1.2.4</td><td>8090</td></tr>
<tr><td>第十章</td><td>在线客服中心</td><td>Swarm 自主协作</td><td>OpenAI Agents SDK 0.17.4</td><td>8092</td></tr>
</tbody>
</table>
</div>

<br>

&emsp;&emsp;这四个项目接下来一章走一个，从业务需求出发，一路落到能跑起来的后端代码和能看协作过程的前端轨迹。它们用的模型统一是 deepseek-v4-pro（通过 OpenRouter 接入），这样四个项目的对照才公平，也方便我们把注意力集中在协作结构本身的差异上，而不是被不同模型的能力差异干扰。

&emsp;&emsp;还有一点要在进入 第三部分 之前先讲清楚：这四个项目虽然协作模式各不相同，但底下垫着的是同一套<font color=red>工程底座</font>。这套底座包括——后端统一用 FastAPI 搭建、用 uvicorn 启动；过程轨迹统一通过 SSE（服务器推送事件）实时流到前端，让我们能亲眼看到智能体之间一步步怎么协作；会话状态统一存在 SQLite 数据库里，支持不限轮数的多轮对话；前端则是统一的一套界面，左边是对话区、右边是协作轨迹图，节点能步进播放、能点开看每个智能体的提示词和工具调用。这套底座四个项目完全共用，所以后面四章不会重复讲它，每一章只聚焦在那个项目独有的协作结构、智能体分工和踩过的坑上——底座的事，记住"四项目共用同一套 FastAPI + SSE + SQLite + 统一前端"这一句就够了。

&emsp;&emsp;再交代一下这四个项目怎么在本机跑起来，省得到了第七章第一条启动命令前卡住。四个项目的源码都在课件目录下的 `代码/` 文件夹里，每个项目占一个子目录——报表流水线在 `代码/workflow-sales-report/`、研究编排在 `代码/flagship-supervisor-tool/`、软件交付在 `代码/hierarchical-software-delivery/`、在线客服在 `代码/swarm-customer-service/`。每个项目都自带一份 `requirements.txt` 和一个 `.env.example` 模板，并各用一个独立的虚拟环境（就建在项目目录下、叫 `.venv`）——四个框架的依赖各自隔离。以报表流水线为例，第一次运行前进项目目录，把环境建好、依赖装上、再把密钥填进 `.env`：

```bash
# macOS / Linux —— 进项目目录,建 venv + 装依赖 + 配 .env
cd 代码/workflow-sales-report
python3 -m venv .venv
./.venv/bin/pip install -r requirements.txt
cp .env.example .env          # 再用编辑器打开 .env,把 OPENROUTER_API_KEY 填成你自己的 key
```

```powershell
# Windows PowerShell
cd 代码\workflow-sales-report
python -m venv .venv
.\.venv\Scripts\pip.exe install -r requirements.txt
copy .env.example .env        # 再编辑 .env,填上 OPENROUTER_API_KEY
```

```bash
# Windows Git Bash
cd 代码/workflow-sales-report
python -m venv .venv
./.venv/Scripts/pip install -r requirements.txt
cp .env.example .env          # 再编辑 .env,填上 OPENROUTER_API_KEY
```

&emsp;&emsp;其余三个项目照同样的做法：进各自目录、`python -m venv .venv`、装它自己的 `requirements.txt`、`cp .env.example .env` 再填密钥。四个项目都通过 OpenRouter 调 deepseek，密钥统一放在各项目目录的 `.env` 文件里——项目代码用 python-dotenv 自动读取，绝不硬编码在代码里，`.env` 本身也不进版本库（仓库里只放 `.env.example` 模板）。

&emsp;&emsp;环境建好、`.env` 填好密钥，就可以按每章给出的启动命令把服务跑起来——命令的形态都是"进项目目录、用项目自带的 `.venv` 跑 uvicorn"。

&emsp;&emsp;预览到此为止。第三部分 的第一站，是四种模式里门槛最低、也最贴近日常业务的那一种——<font color=red>Workflow 流程编排</font>。第七章我们就用 CrewAI 搭一条销售数据报表流水线：自然语言提一个报表需求，让它依次走完查库、统计、出图、成文这四道固定工序，把"工序固定即流水线"这条选型判断，变成手边一个能真查数据库、能出图表的完整项目。

---

## <center>第七章 案例一 · 销售数据报表流水线（CrewAI）</center>

&emsp;&emsp;从这一章起，我们正式踏进 第三部分。前面六章我们把四种协作模式的来龙去脉和五个框架的最小实现都摸过了一遍，现在该把判断力落到真实项目里检验。第一个落地的案例，是 第三部分 四个项目里业务最朴素、也最容易上手的一个——一条销售数据报表流水线：我们用自然语言提一个分析需求，比如"按品类统计总销售额，从高到低"，它就依次走完查库、统计、出图、成文这四道固定工序，最后吐出一份带图表的销售分析报告。

&emsp;&emsp;这个项目的关键词是"固定工序"。报表这件事的流程事先就能划清楚，不会因为换个需求就临时改道——它天生就是 Workflow 流程编排的主场。我们在第二章 2.3 节用 CrewAI 的 `Process.sequential` 写过一条最小流水线（研究员、写作者、编辑三道工序逐个跑），那是流水线的"骨架样本"；这一章要做的，是把同一套角色化写法扩成一个能真查数据库、能出图表、还能把每一步实时画到前端的完整项目。下面这张图先把这条流水线的全貌摆出来：左侧一个数据库作为源头，四道工序竖着排成一条生产线，工件一路往下流，最底部长成一份带图表的报告。

<div align=center><font size=2 color=#999999>第七章章节图：销售数据报表流水线的四道竖排工序</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L7-sales-pipeline-729f06e3.png" width=80%></div>

<br>

&emsp;&emsp;这一章我们分五步走：先在 7.1 把业务需求和选型理由讲透，重点说清一个判断——为什么这条流水线不用 CrewAI 全自动的顺序流程，而是用代码确定性地把工序串起来；7.2 先把项目跑起来，看前端的工位逐个亮起、看实测结果，直观感受效果；7.3 看清楚这条流水线脚下踩的数据底座，一个由四张表组成的星型库；7.4 是本章的核心，挑几段关键代码看 CrewAI 的角色化写法怎么落到每一道工序上；7.5 收束几条实测里踩出来的提示。本章项目和 第三部分 的另外三个一样，统一用 deepseek-v4-pro 这个深度思考模型——项目要的是真效果，又需要在模型输出不规范时有兜底，这个量级的模型配上后面会讲的代码兜底，组合起来才稳。

### 7.1 业务与选型

&emsp;&emsp;先把业务需求说清楚。这个项目要解决的事很具体：业务方不想写 SQL，只想用一句大白话提需求——"按品类统计总销售额"、"哪个区域经理管的盘子最大"、"白金会员都买了些什么"——然后拿到一份能直接看的报告，里面有数字、有图表、有结论和建议。把这件事拆开，会发现它天然分成四道前后衔接的工序：第一道把自然语言需求翻译成 SQL 并真去数据库取数，第二道从取回的数据里提炼关键指标，第三道为这些数据选一种合适的图型、给出图表配置，第四道把前面这些综合成一份有洞察的报告。

&emsp;&emsp;这四道工序的顺序是死的——必须先查到数，才谈得上统计；必须先有统计，才谈得上画图和成文。不会出现"换个需求就要先成文再查库"这种临时改道。第六章我们立下的选型第一问是"工序固定吗"，这条流水线给出的答案是斩钉截铁的"固定"，所以它落到 Workflow 流程编排这一格，没有悬念。

> <font size=2>【名词解释】<b><font color=red>NL→SQL</font>（Natural Language to SQL，自然语言转 SQL）</b>:把人话形式的查询需求（如"按品类统计销售额"）交给大模型，让它结合数据库的表结构，生成一句可执行的 SQL 查询语句。它是这条流水线第一道工序的核心任务，也是"业务方不写 SQL 也能取数"的关键一环。</font>

&emsp;&emsp;模式定了是 <font color=red>Workflow</font>，接下来是这一节最值得讲的一个判断：用 CrewAI 落地这条流水线，到底是用它全自动的 `Process.sequential`，还是自己用代码把工序串起来。第二章 2.3 节我们见过 `Process.sequential` 的写法——把几个 Agent 和几个 Task 装进一个 Crew，声明这是顺序流程，框架就自动接管了"前一个产出喂给后一个"这件衔接的事。那种写法对纯文本流水线（研究要点 → 写成博客 → 精简成导读）非常顺手，因为工序之间除了传文本不需要别的动作。

&emsp;&emsp;但这条销售报表流水线有两个 `Process.sequential` 这个黑盒满足不了的硬需求。第一，工序之间要插入"真查数据库"这个动作——查询工程师产出的不是最终答案，而是一句 SQL，这句 SQL 必须由我们的代码真去 sales.db 上执行、拿回真实的数据行，再喂给统计工序。第二，每一道工序跑完，它的产出要立刻通过 SSE 推到前端，让我们能亲眼看着工位一个接一个亮起来。这两件事都要求我们能在工序的缝隙里插手，而全自动的 `Process.sequential` 把整条链路包成了一个黑盒，缝隙是焊死的，伸不进手。

&emsp;&emsp;所以这个项目的做法是：<b>把"衔接"这件事从框架收回到我们自己的代码里，但每一道工序仍然交给一个 CrewAI 的 Agent 去跑</b>。也就是说，CrewAI 的招牌——用 role、goal、backstory 给每道工序配一个有专业人设的角色——一点没丢，丢掉的只是"让框架自动串工序"这一项。查询工程师、统计分析师、可视化工程师、报告撰写仍然是四个有角色设定的 CrewAI Agent，只不过它们之间的接力棒，是由我们的代码一棒一棒地递的。这换来的是稳和可观测：工序之间能真查库、能发事件，而且因为控制流固定在代码里，deepseek 这类模型不会因为框架的自动编排而发散。这正是第六章那条"能简单就别上复杂、能确定就别交给模型"原则在项目层面的又一次体现。

### 7.2 先跑起来看效果

&emsp;&emsp;在拆开代码之前，我们先把这个项目跑起来，直观感受一下它的效果——后面再回头看每一行是怎么实现的。这个项目和 第三部分 其余三个共用同一套 FastAPI + SSE + SQLite 工程底座，启动方式也一致——进项目目录，用项目共用的虚拟环境跑 uvicorn，监听在本机 8091 端口。下面给三个平台的启动命令，按各自的终端环境挑一条。

```bash
# macOS / Linux:进项目目录，用上一级的共享虚拟环境启动，监听 8091
cd 代码/workflow-sales-report
.venv/bin/python -m uvicorn backend.app:app --host 127.0.0.1 --port 8091
```

```powershell
# Windows PowerShell:路径分隔符用反斜杠，虚拟环境的 python 在 Scripts 目录
cd 代码\workflow-sales-report
.venv\Scripts\python.exe -m uvicorn backend.app:app --host 127.0.0.1 --port 8091
```

```bash
# Windows Git Bash:路径写法和 macOS 一致，正斜杠即可
cd 代码/workflow-sales-report
.venv/bin/python -m uvicorn backend.app:app --host 127.0.0.1 --port 8091
```

&emsp;&emsp;服务起来后，浏览器打开 `http://127.0.0.1:8091` 就是这条流水线的操作界面。在输入框里提一个分析需求，比如"按品类统计总销售额，从高到低"，按下发送，就能看到这条流水线最直观的一幕：界面中央是一排竖着摆的工位卡——查询工程师、统计分析师、可视化工程师、报告撰写各占一张，卡上有人物头像、角色名和当前的工作动作文案。随着流水线往下走，这四张卡会被一个接一个地点亮，工件像在传送带上一样从上往下流。点开「查看数据库」，能分四张表、每页 50 行翻看 `sales.db` 里的真实数据；流水线跑完，报告会在一个独立弹窗里呈现，里面有画好的图表、报告正文、这一轮用到的 SQL 和查回的数据，还能下载。

<div align=center><font size=2 color=#999999>销售数据报表流水线运行界面（http://127.0.0.1:8091）</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L7-run-c27a63a3.png" width=80%></div>

&emsp;&emsp;这套流程不是纸上谈兵，是实测跑通的。下面这张表把 lab-records 里这个项目的实测结果列出来，都是端到端跑出来的数字。

<div align=center><font size=2 color=#999999>销售报表流水线实测结果（deepseek-v4-pro，端到端验证）</font></div>

<div align="center">
<table width="80%">
<thead><tr>
<th>验证维度</th><th>实测结果</th>
</tr></thead>
<tbody>
<tr><td>基础轮次</td><td>5 轮全部跑通，回归再测 5 轮仍全部跑通</td></tr>
<tr><td>跨表 JOIN 正确率</td><td>100%（供应商、城市、会员等级、区域经理四类维度追问全对）</td></tr>
<tr><td>图表选型</td><td>智能匹配：占比类自动选饼图、趋势类选折线、排名类选柱状图</td></tr>
<tr><td>链式追问</td><td>"只看最高的那个品类"自动推出 <code>WHERE category='数码'</code> 并带 JOIN</td></tr>
<tr><td>单轮耗时</td><td>25 到 60 秒，最快一轮 21 秒</td></tr>
</tbody>
</table>
</div>

<br>

&emsp;&emsp;这张表里最值得玩味的是图表选型和链式追问两行。图表选型说明可视化工程师这个角色的人设（懂柱、饼、折线各自适合表达什么）真的起了作用——它会按数据的性质挑图型，而不是一律画柱状图。链式追问那行则说明 7.4 节末尾讲的历史拼接确实生效了——模型从上一轮的结论里读出了"最高的品类是数码"，并把它落成了 SQL 的过滤条件。这两点合起来，恰好印证了 Workflow 流程编排的特点：每道工序的角色越专、产出越规整，整条线串起来的效果就越好。

&emsp;&emsp;效果看到了，下面我们就回过头，把这套效果背后的实现一层层拆开——先看它脚下的数据底座，再看每道工序的代码怎么写。

### 7.3 数据底座：星型四表

&emsp;&emsp;先看清楚这条流水线脚下踩的数据底座。一条报表流水线的分析能有多丰富，上限是由它能查的数据决定的。这个项目的数据库 `sales.db` 没有用一张大宽表草草了事，而是搭成了一个由四张表组成的星型结构——这样查询工序才能写出 JOIN，做"按供应商汇总""按会员等级分群""看哪个区域经理超额"这类跨表的分析。

> <font size=2>【名词解释】<b><font color=red>星型模型</font>（star schema，星型数据模型）</b>:数据仓库里的经典建模方式。中心是一张记录业务流水的"事实表"，四周围着若干描述性的"维表"，事实表靠几个关联键连到各张维表，整体形状像一颗星。它的好处是事实表保持精简、维度信息按需 JOIN 进来，既省空间又便于多角度分析。</font>

&emsp;&emsp;这个星型库的四张表，行数和职责都是实测跑出来的，列在下面这张表里。中心是事实表 `sales`，记录每一笔销售流水；外围三张维表分别从产品、客户、地区三个角度补充描述信息。

<div align=center><font size=2 color=#999999>sales.db 星型四表：一张事实表 + 三张维表</font></div>

<div align="center">
<table width="80%">
<thead><tr>
<th>表名</th><th>角色</th><th>行数</th><th>关键字段</th>
</tr></thead>
<tbody>
<tr><td>sales</td><td>事实表</td><td>380</td><td>date 日期、customer 客户、product 产品、category 品类、region 地区、qty 数量、amount 销售额</td></tr>
<tr><td>products</td><td>维表</td><td>20</td><td>product 产品（主键）、unit_price 单价、supplier 供应商、stock 库存</td></tr>
<tr><td>customers</td><td>维表</td><td>30</td><td>customer 客户（主键）、level 会员等级（普通/银卡/金卡/白金）、city 城市</td></tr>
<tr><td>regions</td><td>维表</td><td>4</td><td>region 地区（主键）、manager 区域经理、quarterly_target 季度目标</td></tr>
</tbody>
</table>
</div>

<br>

&emsp;&emsp;三张维表靠三个关联键挂到事实表上：`sales.product` 连 `products.product`、`sales.customer` 连 `customers.customer`、`sales.region` 连 `regions.region`。有了这三条关联，查询工序就能把"销售额"这个事实，跟"供应商是谁""客户是不是白金会员""哪个区域经理负责"这些维度交叉起来分析。

&emsp;&emsp;光有表还不够，得让查询工序知道这些表长什么样、怎么 JOIN。这就是 `db_schema` 这个工具的职责——它返回一段把表结构和关联键都写清楚的说明文字，作为提示喂给查询工程师 Agent，让它照着写 SQL。下面这段就是它返回的内容，注意最后一行专门把三个 JOIN 关联键点了出来，这是引导模型写对跨表查询的关键。

In [ ]:
def db_schema() -> str:
    """返回 sales.db 的星型库结构说明，作为查询工序写 SQL 的参考。"""
    # 把四张表的字段和含义写清楚；末行显式给出三个 JOIN 关联键
    return ("【星型库，可 JOIN 关联查询】\n"
            "sales(id, date 'YYYY-MM-DD', customer 客户, product 产品, category 品类, region 地区, qty 数量, amount 销售额/元) — 事实表\n"
            "products(product 主键, category 品类, unit_price 单价, supplier 供应商, stock 库存)\n"
            "customers(customer 主键, level 会员等级[普通/银卡/金卡/白金], city 城市, join_date 注册日)\n"
            "regions(region 主键, manager 区域经理, quarterly_target 季度目标)\n"
            "关联键:sales.product=products.product;sales.customer=customers.customer;sales.region=regions.region")

&emsp;&emsp;这段 `db_schema` 做的事，是把数据库的"地图"用大模型能读懂的方式描述出来。它在流水线里的位置是第一道工序的输入参考——查询工程师拿到它，才知道有哪些表、哪些字段、怎么关联。它对整个项目的意义在于：分析的丰富度上限，就是由这张"地图"画得有多清楚决定的，末行那条 JOIN 提示越明确，模型写出正确跨表查询的概率就越高。lab-records 实测里，正是靠这条提示，链式追问"供应商""城市""会员等级""区域经理"的 JOIN 正确率达到了 100%。

&emsp;&emsp;查询工程师写出来的 SQL，要由另一个工具 `run_sql` 真去 `sales.db` 上执行。这里有一道必须设的安全闸——大模型生成的 SQL 不可全信，万一它写出删表、改数据、或者一次返回几万行的语句，都会出问题。所以 `run_sql` 只放行单条只读的 SELECT 查询，并把返回行数限制在 200 行以内。

In [ ]:
def run_sql(sql: str) -> dict:
    """只允许单条 SELECT，真打 sales.db。"""
    s = (sql or "").strip().rstrip(";").strip()
    # 第一道闸:只放行 SELECT 开头的查询，挡掉一切写操作
    if not re.match(r"(?is)^\s*select\b", s):
        return {"ok": False, "error": "只允许 SELECT 查询", "columns": [], "rows": []}
    # 第二道闸:出现分号说明想塞多条语句，一律拒绝
    if ";" in s:
        return {"ok": False, "error": "只允许单条语句", "columns": [], "rows": []}
    try:
        c = sqlite3.connect(SALES_DB)
        cur = c.execute(s)
        cols = [d[0] for d in cur.description]
        rows = [list(r) for r in cur.fetchall()]
        c.close()
        return {"ok": True, "columns": cols, "rows": rows[:200]}  # 最多回 200 行，防超量
    except Exception as e:
        # SQL 写错不抛异常，而是把错误文本带回去，供工序内重试用
        return {"ok": False, "error": f"{type(e).__name__}: {e}", "columns": [], "rows": []}

&emsp;&emsp;这段 `run_sql` 实现的是"真查库"这个动作，是整条流水线和数据库之间唯一的接触点。它在流程里的作用是承上启下——上承查询工程师产出的 SQL，下启统计工序拿到的真实数据。它对项目的意义有两层：一层是安全，两道正则闸把模型可能写出的危险语句和超量查询挡在门外；另一层是为返工留了接口——它在 SQL 写错时不直接报错崩溃，而是把错误文本原样带回去，让上游的查询工程师能拿着这段错误重写一次。这个"修一次"的小闭环怎么接上，下一节走读流水线代码时就会看到。完整的 `tools.py` 还包含给前端分页看库的 `db_preview`，那部分跟教学主线关系不大，可以到项目源码里翻看。

### 7.4 流水线源码走读

&emsp;&emsp;数据底座清楚了，这一节进入本章核心：看 CrewAI 的角色化写法到底怎么落到每一道工序上。我们不会把整个 `agent.py` 逐行搬过来——完整的流水线编排、所有的事件推送细节都在项目源码里，跑起来看最直观。这里只挑三段最能说明问题的代码，看清楚三件事：CrewAI 的 Agent 怎么以"即用即抛"的粒度跑一道工序、自然语言怎么变成 SQL 再真查库、以及模型输出不规范时怎么用代码兜底。先用一张图把工件流经四道工序、逐步长成报告的过程拆开。

<div align=center><font size=2 color=#999999>7.4 配图：一个分析需求流经四道工序，逐步长成报告</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L7.3-stage-handover-6c60245a.png" width=80%></div>

<br>

&emsp;&emsp;第一段代码，是这条流水线跑每一道工序的"统一方法"，叫 `_crew_sync`。它解决的问题是：怎么用一个 CrewAI Agent 跑完一道工序、拿到产出，然后立刻把这个 Agent 丢掉。它接收三个参数——工序的编号、给这道工序的任务描述、期望的产出格式，返回这道工序的文本产出。

In [ ]:
# 每道工序的角色配置都登记在 STAGES 里（这里摘前两道工序示意）
STAGES = [
    {"id": "query", "role": "查询工程师",
     "goal": "把分析需求翻译成正确的 SQLite 查询并取数",
     "backstory": "精通 SQL,只产出可执行查询", "tools": ["db_schema", "run_sql"]},
    {"id": "stats", "role": "统计分析师",
     "goal": "从查询结果提炼关键指标",
     "backstory": "擅长从数字里看出总量、占比、Top 与趋势", "tools": []},
    # …可视化工程师、报告撰写两道工序同理，完整配置见项目源码
]
_STAGE = {s["id"]: s for s in STAGES}

def _crew_sync(stage_id: str, desc: str, expected: str) -> str:
    """用一个 CrewAI Agent 跑一道工序（同步，外部用 asyncio.to_thread 包）。"""
    s = _STAGE[stage_id]
    # 临时建一个角色化 Agent：role/goal/backstory 三件套就是 CrewAI 的招牌
    a = Agent(role=s["role"], goal=s["goal"], backstory=s["backstory"], llm=llm, verbose=False)
    # 一个 Task 绑这个 Agent，写明任务描述和期望产出
    t = Task(description=desc, expected_output=expected, agent=a)
    # 单 Agent + 单 Task 装进 Crew，kickoff 跑一次，拿到产出即返回——用完即抛
    return str(Crew(agents=[a], tasks=[t], verbose=False).kickoff())  # 启动 CrewAI 团队跑起来

&emsp;&emsp;这段 `_crew_sync` 用到的全是第二章 2.3 节那套 CrewAI API——`Agent` 的 role/goal/backstory 三件套、`Task` 绑定 Agent、`Crew` 装起来 `kickoff`。但它的用法和第二章有一处关键差别：第二章是一次性把三个 Agent、三个 Task 装进一个 Crew，整体 `kickoff` 一把跑完；这里是<b>每跑一道工序，就临时建一个只装单个 Agent、单个 Task 的 Crew，跑完拿到结果就丢掉</b>。这种"即用即抛"的写法，正是 7.1 节那条选型理由的代码落地——因为衔接被收回到了外层代码里，CrewAI 的职责被收缩到了最小粒度："只负责跑一道工序的那一次 LLM 调用"。工序与工序之间的接力，由外层代码亲手来递。这么做对项目的意义是：每道工序成了一个独立的、可以在它前后插入任意代码（真查库、发事件）的原子单元，整条流水线的控制权牢牢攥在我们自己手里。

&emsp;&emsp;第二段代码，是第一道工序"查询"的核心——也是整条流水线唯一和数据库打交道的地方。它要把自然语言需求变成 SQL、真去查库，并且在 SQL 写错时让查询工程师修一次。下面这段截取了这道工序的主干（完整版还包含若干 SSE 事件推送，那些发事件的细节这里略去，看项目源码里的 `run_events` 最完整）。

In [ ]:
# 工序 1：把需求 + 表结构喂给查询工程师，让它产出一句 SQL
schema = tools.db_schema()
raw = await _stage_llm("query",
        f"数据表结构:{schema}\n\n分析需求:{topic}\n\n请写一句 SQLite SELECT 来回答这个需求。只输出 SQL,不要解释。",
        "一句可执行的 SQLite SELECT")
sql = _extract_sql(raw)                        # 从模型回复里抠出干净的 SELECT 语句
res = await asyncio.to_thread(tools.run_sql, sql)   # 代码真去 sales.db 执行这句 SQL

# 报错就让查询工程师拿着错误信息修一次——工序内的小返工闭环
if not res.get("ok"):
    raw = await _stage_llm("query",
            f"数据表结构:{schema}\n\n需求:{topic}\n\n上一条 SQL 执行报错:{res.get('error')}\n"
            f"出错的 SQL:{sql}\n请修正,只输出正确的 SQLite SELECT。",
            "一句可执行的 SQLite SELECT")
    sql = _extract_sql(raw)
    res = await asyncio.to_thread(tools.run_sql, sql)   # 用修正后的 SQL 重查一次

&emsp;&emsp;这段代码实现的是 NL→SQL→真查库这一整条动作，是 7.1 节"为什么要代码确定性衔接"那个判断最直白的证据。它在流程里的作用，是把笼统的分析需求落成具体的数据：`_stage_llm` 调用查询工程师 Agent 产出 SQL，`_extract_sql` 把模型回复里可能裹着的代码块、解释文字剥掉只留下干净的 SELECT，然后 `run_sql` 代码真去查库。最值得注意的是底下那个 `if not res.get("ok")` 分支——它正好接上了 7.3 节 `run_sql` 把错误带回来留下的那个接口：SQL 写错时不崩溃，而是把错误信息连同出错的 SQL 一起回喂给查询工程师，让它修正后重查一次。这个"修一次"的小闭环对项目的意义是把容错做在了工序内部，让一次写歪的 SQL 不至于让整条流水线报废，是真实项目里很实用的一招。

&emsp;&emsp;第三段代码，是可视化工序的产出解析，叫 `_parse_chart`。它要解决的是一个所有"让大模型产出结构化配置"的场景都会碰到的问题——模型说要给我们一段规规矩矩的 JSON，但它偶尔会给歪。这道工序让可视化工程师 Agent 输出一段图表配置 JSON（图型是 bar、pie 还是 line，加上标签和数值），代码这边必须能把它稳稳解析出来，解析不了也得有个兜底。

In [ ]:
def _parse_chart(t: str, res: dict) -> dict:
    """解析图表配置 JSON；失败则用查询结果前两列兜底成柱状图。"""
    try:
        m = re.search(r"\{.*\}", t or "", re.S)        # 从模型回复里捞出 JSON 片段
        if m:
            cfg = json.loads(m.group(0))
            if cfg.get("labels") and cfg.get("values"):  # 标签和数值都在才算解析成功
                cfg.setdefault("type", "bar")            # 没给图型就默认柱状图
                cfg.setdefault("title", "分析结果")
                return cfg
    except Exception:
        pass
    # 兜底:解析失败时，直接拿查询结果的前两列拼一张柱状图
    cols, rows = res.get("columns", []), res.get("rows", [])
    if len(cols) >= 2 and rows:
        return {"type": "bar", "title": f"{cols[0]} × {cols[1]}",
                "labels": [str(r[0]) for r in rows[:12]],
                "values": [float(r[1]) if isinstance(r[1], (int, float)) else 0 for r in rows[:12]]}
    return {"type": "bar", "title": "结果", "labels": [], "values": []}

&emsp;&emsp;这段 `_parse_chart` 实现的是"解析模型的结构化产出，解析失败就兜底"。它在流程里的作用是把可视化工程师那句话变成前端能直接画的图表配置；它对项目的意义，是把整门课反复强调的那条主题线——<b>模型输出不可全信，确定性兜底必须就位</b>——落到了一段具体代码上。`try` 块里它努力把模型给的 JSON 解析成合法配置，一旦解析失败、或者 JSON 里缺了标签和数值，它不会让前端拿到一段坏配置而画不出图，而是退一步，直接用查询结果的前两列拼一张最朴素的柱状图。这意味着哪怕模型这一步彻底跑歪，用户也总能看到一张图，而不是一片空白。

&emsp;&emsp;最后补一句多轮对话的事。这条流水线支持链式追问——问完"按品类统计销售额"，接着追一句"只看最高的那个品类"，它能接着上文往下分析。这件事的实现不在 `agent.py` 里，而在服务端：每一轮把历史的"需求 + 结论摘要"拼进新一轮的提示。lab-records 的实测里，正是靠这套历史拼接，那句"只看最高的那个品类"被查询工程师自动推成了 `WHERE category='数码'` 并带上了 JOIN——模型从上文里读懂了"最高的品类是数码"，把它落成了 SQL 的过滤条件。这套服务端拼接历史的逻辑，在项目源码里能完整看到。

### 7.5 实测复盘

&emsp;&emsp;最后把这个项目实测里值得记住的几条收束一下，都是真实跑下来踩出来的经验，留给我们自己上手时少走弯路。

> <font size=2>【提示】图表配置一定要有兜底。可视化工序让大模型输出 JSON 图表配置，但模型偶尔会给出不规范的 JSON——缺字段、多一段解释文字、或者数值类型不对。7.4 节的 `_parse_chart` 之所以写了那段 fallback，就是为了在解析失败时退回到"拿查询结果前两列拼一张柱状图"，保证用户任何时候都能看到一张图，而不是一片空白。凡是让模型产出结构化配置的工序，都该配一个这样的兜底。</font>

> <font size=2>【提示】改了表结构要 DROP 重建。这个项目的播种逻辑里，普通的 `ensure_seed` 只在表为空时才灌数据——如果数据已经在了，它不会动。所以一旦改动了表结构（比如新增一张维表、给某张表加一列），光重启服务是不够的，旧表还在，新结构不会生效。正确做法是用 `seed(force=True)` 先把四张表 DROP 掉再重建，让新 schema 真正落地。</font>

> <font size=2>【提示】单轮 25 到 60 秒是正常的。这条流水线用的是 deepseek-v4-pro 这个深度思考模型，又是四道工序串行跑，单轮花上 25 到 60 秒属于预期之内。前端用 SSE 把每道工序的进展实时推出来、工位逐个亮起，就是为了让这段等待变得可感知，不至于让人对着空白页面干等。要留意的是，通过 OpenRouter 接入时偶尔会遇到长尾波动，个别请求会明显更慢，这是第三方端点的常态，给请求都加上超时和重试即可。</font>

&emsp;&emsp;到这里，第七章就走完了。我们用 CrewAI 把 Workflow 流程编排落成了一个能真查库、能出图、能多轮追问的完整项目，看清了"代码确定性衔接、每道工序一个角色化 Agent"这套写法在真实业务里的样子，也把第二章那条最小流水线扩成了手边能跑起来的东西。完整的工程代码——包含本章略去的事件推送、服务端多轮拼接、前端那套流水线动画——都在项目源码 `代码/workflow-sales-report/` 里，建议照着 7.2 的命令亲手跑一遍，看着工位一个个亮起来，体感最深。

&emsp;&emsp;第三部分 的第二站，我们要从"工序固定"换到"开放并行"。下一章是一个多轮研究编排项目：给它一个开放的研究主题，让一个中央主管把任务拆给好几位各有专长的分析专家并行去查，再收口综合成一份简报。它对应的正是第六章选型里那条"拿不准时默认先考虑"的 Supervisor 集中调度，用的框架则换成微软的 Agent Framework。

## <center>第八章 案例二 · 多轮研究编排：Supervisor 落地</center>

&emsp;&emsp;上一章我们用 CrewAI 把 <font color=red>Workflow 流程编排</font>落成了一个能真查库、能出图的销售报表项目，看清了"工序固定、代码确定性衔接"在真实业务里的样子。但流程编排有一个前提：工序是事先排好的，谁先谁后固定在代码里。这一章我们换一种业务形态——开放研究。给系统一个开放的研究主题，比如"当前多智能体协作有哪几种主流架构、各自的生态如何"，它事先并不知道该查哪几个维度、该派几个专家，得由一个中央主管现场判断、动态决定。这正是第六章选型里那条"拿不准时默认先考虑"的 Supervisor 集中调度。

&emsp;&emsp;这一章我们要把第三章讲过的两块拼图在一个完整项目里合起来落地。第三章 3.3 OpenAI Agents SDK 那一节，我们讲清了 agents-as-tools 这种形态——主管把专家当工具调用、控制权始终攥在自己手里；第三章 3.5 我们见过 MAF 的重型 Magentic 主管，也实测过它的高层 builder 要分个体看——这个生产项目最终选了更底层的 `as_agent` 自己搭主管，换取轻量与完全可控。本章这个研究编排项目，正是把这两点合在一起：用 MAF 的 `as_agent` 把七个领域专家各自包装成主管可调用的函数工具，主管按主题挑四到六个并行调用，最后综合成一份研究简报。

&emsp;&emsp;我们这一章会分五步走：先看业务为什么落到 Supervisor、为什么模型从 第二部分 最小代码的 flash 换成了深度思考的 pro（8.1）；再把项目跑起来，看星形派单动画和实测结果，直观感受效果（8.2）；接着看专家是怎么被包装成工具的，以及每个专家执行时为什么走"确定性检索 + 单轮总结"而不开自主工具循环（8.3）；然后看主管怎么综合、以及当深度思考模型偶发"思考吃掉正文"时，工程上怎么用一个兜底综合器把它救回来（8.4）；最后把这个项目实测踩出来的经验收束成几条提示（8.5）。这一章是整门课"模型不确定性、确定性兜底"这条主题线展开得最完整的一章——确定性检索、兜底综合、参考来源确定性追加，三处兜底会接连登场。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L8-research-star-673a4d22.png" width=80%></div>

### 8.1 业务与选型

&emsp;&emsp;先把业务讲清楚。我们要做的是一个研究编排系统：用户提一个开放的研究主题，系统把它拆成多个分析维度，派不同的专家分头联网检索，最后收口综合成一份简报。这个系统手下养了七个领域专家——市场分析、技术分析、趋势分析、竞品分析、政策法规、风险分析，外加一个专门盯"某个技术或框架是否还在活跃维护、是否已经停更或被取代"的时效维护专家。但不是每个主题都要七个全上：问一个偏市场态势的题，主管会挑市场、竞品、趋势几个；问一个偏技术选型的题，主管会挑技术、风险、时效维护几个。通常一轮挑四到六个并行调用，挑哪几个由主管现场判断——这个"挑专家"的决策权，正是 Supervisor 模式的价值所在。

&emsp;&emsp;为什么这个业务落到 Supervisor，而不是上一章的 Workflow？判断标准就是第六章给的那条判据：问题是不是开放的、并行有没有收益。这里两条都成立。研究主题是开放的，事先不知道该查哪几个维度，需要一个中央大脑现场决策，这是 Supervisor 的典型场景；而多个维度的检索互不依赖，市场专家查市场、技术专家查技术，完全可以同时进行，并行能把一轮的耗时从"七个串着查"压到"挑中的几个一起查"。开放问题加并行收益，结论就很清楚：用 Supervisor，且用它的 agents-as-tools 变体——把专家当工具挂在主管身上。

> <font size=2>【名词解释】<b><font color=red>supervisor-as-tools</font>（Supervisor as Tools，把专家当工具的集中调度）</b> — Supervisor 模式的一种实现变体：中央主管把每个专家都当成挂在自己身上的一个函数工具，主管自主判断该调哪几个、给每个派什么任务，调用后专家把结论当返回值交回，控制权始终留在主管手里。它和第三章 3.3 讲的 agents-as-tools 是同一个思路在集中调度场景下的落地。</font>

&emsp;&emsp;这里还有一个跟前面几章不同的选型决策需要点明：模型。第二部分 那些跑通最小代码的章节，我们统一用了 `deepseek-v4-flash` 这个轻快模型，目的是让最小示例跑得快、好演示。但这个研究编排项目对回答质量要求高——要综合多个专家的资料、要判断问题类型、要组织一份有结论有论证的简报，所以本案例换成了深度思考的 `deepseek-v4-pro`。

&emsp;&emsp;这个换型不是随手做的。pro 推理更强、综合质量更好，但它有一个深度思考模型特有的毛病——偶发"思考吃掉正文"，即把额度都花在内部推理上、最后留给正文的输出被挤没了。这个毛病恰恰是 8.4 节那个兜底综合器存在的原因。换句话说，用 pro 拿效果、用兜底补它的不稳定，这一对取舍本身就是本章的一条主线。

> <font size=2>【名词解释】<b><font color=red>MAF</font>（Microsoft Agent Framework，微软智能体框架）</b> — 微软出品的企业级智能体编排框架，第一章讲过它是 AutoGen 与 Semantic Kernel 合并后的接棒者。本案例基于它的 agent-framework-core 1.8 实测，沿用第三章定下的"高层编排原语配第三方端点不稳，退回底层 `as_agent` 自己搭"的写法。</font>

### 8.2 先跑起来看效果

&emsp;&emsp;在拆开代码之前，我们先把项目跑起来，直观感受一下它的效果——后面再回头看每一处实现。这个项目和 第三部分 其余三个共用同一套 FastAPI + SSE + SQLite 工程底座，启动方式也一致——进项目目录，用项目共用的虚拟环境跑 uvicorn，监听在本机 8088 端口。下面给三个平台的启动命令，按各自的终端环境挑一条。

```bash
# macOS / Linux：进项目目录，用上一级的共享虚拟环境启动，监听 8088
cd 代码/flagship-supervisor-tool
.venv/bin/python -m uvicorn backend.app:app --host 127.0.0.1 --port 8088
```

```powershell
# Windows PowerShell：路径分隔符用反斜杠，虚拟环境的 python 在 Scripts 目录
cd 代码\flagship-supervisor-tool
.venv\Scripts\python.exe -m uvicorn backend.app:app --host 127.0.0.1 --port 8088
```

```bash
# Windows Git Bash：路径写法和 macOS 一致，正斜杠即可
cd 代码/flagship-supervisor-tool
.venv/bin/python -m uvicorn backend.app:app --host 127.0.0.1 --port 8088
```

&emsp;&emsp;服务起来后，浏览器打开 `http://127.0.0.1:8088` 就是这个<font color=red>研究编排</font>的操作界面。在输入框里提一个研究主题，比如"当前多智能体协作有哪些主流架构、各自生态如何"，按下发送，就能看到这个项目最有辨识度的一幕：界面中央是一枚编排总管的圆形徽章居于圆心，七个专家卡片等角分布在它四周的圆周上，连成一个星形。

<div align=center><font size=2 color=#999999>多轮研究编排运行界面（http://127.0.0.1:8088）</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L8-run-67fc53f3.png" width=80%></div>

&emsp;&emsp;主管开始派单时，被挑中的几个专家和总管之间的连线会同时亮起、呈橙色流动的虚线——这就是"并行派发"的视觉化；专家检索完、把结论交回时，连线转成绿色回流。因为多个专家是真并行的，它们的检索和回流事件时间戳自然错开，界面下方还有一条并行甘特图把这种"几个专家同时在干活"的重叠关系画出来。点开任意一个专家节点，能看到它的系统提示词和它用到的检索工具；整轮跑完，综合简报会在一个独立弹窗里呈现，末尾带着那串确定性追加的参考来源。

&emsp;&emsp;这套流程不是纸上谈兵，是实测跑通的。下面这张表把 lab-records 里这个项目的实测结果列出来，都是端到端跑出来的数字。

<div align=center><font size=2 color=#999999>多轮研究编排实测结果（deepseek-v4-pro，端到端验证）</font></div>

<div align="center">
<table width="80%">
<thead><tr>
<th>验证维度</th><th>实测结果</th>
</tr></thead>
<tbody>
<tr><td>单轮并行</td><td>6 个专家并行检索，一轮约 95 到 160 秒</td></tr>
<tr><td>简报篇幅</td><td>通常 2 到 6 千字，最长一次跑出 11309 字</td></tr>
<tr><td>多轮记忆</td><td>第三轮追问"用一句话总结"时，主管零派发、直接用前几轮记忆作答，140 字、26 秒</td></tr>
<tr><td>兜底命中</td><td>pro 思考吃掉正文时，兜底综合器用专家完整结论补出完整简报</td></tr>
<tr><td>来源追加</td><td>参考来源全部来自专家真实检索命中的网页，无模型编造 URL</td></tr>
</tbody>
</table>
</div>

<br>

&emsp;&emsp;这张表里最值得玩味的是多轮记忆那行。前两轮主管真派了专家、查了资料、综合了长简报，到第三轮用户只追问一句"用一句话总结"，主管这时没有再派任何专家，而是直接从前几轮的对话记忆里提炼出一句话作答——零派发、140 字、26 秒。这说明主管不是机械地每轮都派单，它会判断"这一问是不是真需要新检索"，能用记忆答的就不浪费一轮检索。这背后是服务端把该会话前几轮的"问题加结论摘要"拼进了本轮输入，主管据此理解追问的指代，这套多轮上下文拼接的逻辑在项目源码里能完整看到。

&emsp;&emsp;效果看到了，下面我们回过头，把这套效果背后的实现一层层拆开。

### 8.3 专家即工具 + 确定性 RAG

&emsp;&emsp;这一节是本章的技术核心，要回答两个问题：专家是怎么变成主管手里一个工具的，以及每个专家执行时到底干了什么。先看第一个问题。在 MAF 里，把一个专家挂成主管的工具，本质就是把它写成一个普通的异步函数，再把这一组函数塞进主管的 `tools` 参数。下面这段是主管的构造代码，它是整个 supervisor-as-tools 模式的精髓所在，也是把"高层封装不合适时退回底层原语自己搭"这条工程思路真正落到完整项目里的一行。

In [ ]:
from agent_framework.openai import OpenAIChatClient  # Microsoft Agent Framework 聊天客户端，as_agent 能把它变成一个 agent

# 用 config 里的连接参数建 MAF 客户端（OpenRouter 端点 + deepseek-v4-pro）
client = OpenAIChatClient(model=MODEL, api_key=KEY, base_url=BASE_URL)

# 总管 = 一个挂了 7 个"专家工具"的 agent。这一行就是 supervisor-as-tools 的精髓
supervisor = client.as_agent(  # 把 MAF 客户端封成一个具名 agent
    name="research_supervisor",
    instructions=SUP_INSTRUCTIONS,                      # 综合规则写在系统提示词里，见 8.3
    tools=[make_expert_tool(w) for w in WORKERS],       # 7 个专家各被 make_expert_tool 包成一个工具
)

&emsp;&emsp;这段代码实现的是<font color=red>主管的组装</font>：`OpenAIChatClient` 用项目的连接参数建好 MAF 客户端，`as_agent` 把它转成一个能独立运行的智能体，关键在 `tools` 这个参数——我们用一个列表推导式，把七个专家逐个交给 `make_expert_tool` 包装成函数工具，整组挂给主管。

&emsp;&emsp;它在流程里的作用是确立"主管 + 七个工具"这个结构；它对项目的意义在于，这一行就是第三章那两块拼图的合体——既是 agents-as-tools（专家是工具、控制权不转移），又是 MAF 的 `as_agent` 底层写法（不碰高层编排原语，因为它配 OpenRouter 加 deepseek 不稳）。真正决定专家行为的，是 `make_expert_tool` 包出来的那个函数，我们接着看它内部。

&emsp;&emsp;现在回答第二个问题：专家执行时干了什么。这里有一个本案例最重要的设计决策——专家不开自主工具循环，而是走确定性检索加单轮总结。要解释为什么，得先讲一个早期版本踩到的坑。最初的写法是让每个专家 agent 自带一个联网检索工具、由它自主决定搜不搜、搜几次，也就是标准的 agentic 工具循环。实测在 deepseek 上有两个毛病：一是它常把"工具返回的资料"误当成"用户发来的链接或输入"，于是答非所问，回一句"您提供了链接，但还没提出问题"；二是它会反复搜、把一轮拖得很慢。两个毛病都源于让模型自主控制检索这件事在 deepseek 上不靠谱。

> <font size=2>【名词解释】<b><font color=red>RAG</font>（Retrieval-Augmented Generation，检索增强生成）</b> — 先检索出相关资料、再把资料连同问题一起交给模型生成回答的一种范式。本案例的"确定性 RAG"是它的一个变体：检索这一步由编排层的代码确定性地执行一次，而不是交给模型自主决定，这样模型只负责"基于给定资料总结"，行为可控。</font>

&emsp;&emsp;解决办法就是把检索从模型手里收回到代码里：编排层在调用专家"之前"，先用代码确定性地联网检索一次，把检索到的真实资料直接塞进专家的提示词，专家只管基于这份资料做一次总结。这就是确定性 RAG——检索是代码确定做的，不是模型自主决定的；总结是单轮的，不开循环。下面这段是 `make_expert_tool` 包出来的专家函数的核心，把这个三步流程拆给我们看。

In [ ]:
async def call_expert(task: str) -> str:
    # 时效窗：趋势分析要看近几年演变（不限时间），其余专家越新越好（只取近一年）
    tl = None if tool_name == "trend_analysis" else "y"
    try:
        # 第一步：编排层确定性检索一次（不是让专家自主决定搜不搜）
        data = await web_search(tool_name, task, timelimit=tl)
        # 第二步：专家基于真实资料做单轮总结，不开工具循环 → 稳、快、不跑偏
        out = await _summarize(expert, task, data, dims)
        # 第三步：若资料不相关/不足（专家明说"无法基于资料分析"），换全球区域再搜一次重总结
        if any(m in out for m in _INSUFFICIENT):
            more = await web_search(tool_name, task, region="wt-wt", timelimit=tl)
            out = await _summarize(expert, task, data + "\n\n【补充检索（全球）】\n" + more, dims)
    except Exception as e:
        # 单个专家失败不拖垮整轮：回个错误说明，主管照样能综合其余专家
        out = f"(专家执行失败: {type(e).__name__}: {e})"
    # 完整结论存一份（供 8.4 的兜底综合器使用）
    EXPERT_RESULTS.get().append({"tool": tool_name, "output": str(out)})
    return out

&emsp;&emsp;这段代码实现的是单个专家被调用时的完整执行链：检索、总结、必要时换区重搜。它在流程里的作用是把"专家"这个概念落成一个行为可控的函数——主管调它时只管传一个研究任务 `task`，函数内部确定性地把资料备好、让专家总结好、把结论交回。它对项目的意义在于，整个 agentic 工具循环的不确定性被这段代码消化掉了：模型不再决定搜不搜、搜几次，它只做它擅长的那件事——读资料、写总结。

&emsp;&emsp;这段代码里还藏着两个实测打磨出来的细节，值得单独点出来。第一个是检索的时效窗。绝大多数专家都希望资料越新越好，所以默认只取近一年的检索结果（`timelimit="y"`）；唯独趋势分析是例外，它要看的是近几年的演变脉络，限在近一年反而看不出趋势，所以它传 `None` 不限时间。第二个是检索预算。某些维度本身就没什么公开资料，如果不加约束，主管发现专家说"没找到"就会反复重派、专家又反复重搜，一轮能拖到失控。所以底层的检索函数给每个专家整轮的真检索次数封了顶——最多三次，超过就回放上次结果、不再真搜。

> <font size=2>**提示**：让模型自主控制检索流程，在 deepseek 这类模型上不稳。专家自带工具自主决定搜不搜，实测会把工具返回的资料误当成用户输入、答非所问，还会反复搜拖慢一轮。把检索收回到编排层的代码里确定性执行一次、专家只做单轮总结，是本案例稳定下来的关键写法——这也是整门课"确定性兜底"思想在专家层的体现。</font>

### 8.4 综合与兜底

&emsp;&emsp;专家们各自交回了结论，接下来是主管的活：把这些结论<font color=red>综合</font>成一份直接回答用户问题的简报。综合这件事看起来简单，实则最容易出问题，因为一不小心主管就会写成"市场分析说……技术分析说……"这种把各专家资料原样堆一遍的流水账，而不是真正回答用户的问题。所以主管的系统提示词里写了几条硬规则，把综合的方向定死。

&emsp;&emsp;第一条，先判断问题类型再组织回答。如果是概念性、原理性的问题，比如"多智能体协作有哪几种架构"，答案的骨架要用主管自己的知识直接、完整地给出，专家检索的资料只用来补充最新的例证和数据；如果是事实性、时效性的问题，比如最新版本、市场态势，那就以专家检索结果为准、具体数字不能编造。第二条，开头第一段就给出针对问题的明确结论，再围绕问题组织论证，不要写成逐专家的资料堆。第三条，不被偏题资料带跑——如果检索回来的资料和问题主旨有偏离（问的是协作架构、资料却都在讲框架产品对比），要以回答问题为先，偏离的资料降为例证或舍弃。第四条，禁文档体包装——不许把回答包装成《报告》《白皮书》之类，也不许编造版本号、日期、作者这些文档元信息。

&emsp;&emsp;规则写好了，但深度思考模型还有一个绕不开的毛病要兜。前面 8.1 提过，pro 偶发"思考吃掉正文"——它把额度花在内部推理上，最后真正交给综合正文的输出被挤没了，结果就是主管跑完了、简报却几乎是空的。这个毛病不能靠改提示词根治，只能在工程上加一道确定性的兜底。做法是：主管跑完后，检查它综合出的正文长度，如果短得不正常（少于两百字），就说明正文被思考吃掉了，这时把本轮各专家攒下来的完整结论交给一个不带任何工具的综合器，强制让它再综合一次。

> <font size=2>【名词解释】<b><font color=red>ContextVar</font>（Context Variable，上下文变量）</b> — Python 标准库 `contextvars` 提供的一种变量，能在同一个异步任务的调用链里隐式传递数据，且不同并发任务之间互不干扰。本案例用它（`EXPERT_RESULTS`）在一次请求的范围内收集各专家的完整结论，供兜底综合器随时取用，不必把这些结论一层层当参数传下去。</font>

&emsp;&emsp;下面这段是服务端处理这道兜底的核心，它跑在主管返回之后。我们顺带把"参考来源"那段也放进来一起看——它是本章第三处确定性兜底，思路一脉相承。

In [ ]:
# 主管跑完，拿到它综合的正文
result = await supervisor.run(
    model_input,
    options={"temperature": 0.5},                       # 不设 max_tokens：综合作答不限字数，把问题答透
    function_invocation_kwargs={"max_iterations": 3},   # 1 轮派发 +（可选）补派 1 轮 + 1 轮综合
)
brief = getattr(result, "text", None) or str(result)

# 兜底综合：pro 偶发"思考吃掉正文"（text 极短）。此时用各专家完整结论强制再综合一次
ers = EXPERT_RESULTS.get() or []
if len(brief.strip()) < 200 and ers:
    joined = "\n\n".join(f"【{e['tool']}】\n{e['output']}" for e in ers)
    r2 = await synthesizer.run(                         # synthesizer 是个不带工具的纯综合器
        f"用户的问题：{query}\n\n各专家的研究结论：\n{joined}\n\n请直接回答用户的问题（开头先给结论）。",
        options={"temperature": 0.5},
    )
    b2 = getattr(r2, "text", None) or ""
    if len(b2.strip()) > len(brief.strip()):            # 兜底结果更完整才采用
        brief = b2

# 参考来源：不让模型编 URL，用各专家真实检索命中的网页按 href 去重后确定性追加
refs = _dedup_by_href(SEARCH_LOG.get() or [])
if refs:
    lines = "\n".join(f"{i+1}. [{it['title'] or it['href']}]({it['href']})" for i, it in enumerate(refs))
    brief = brief.rstrip() + "\n\n---\n\n## 参考来源\n\n" + lines

&emsp;&emsp;这段代码实现的是综合阶段的两道确定性兜底。它在流程里的作用是兜住主管输出的不确定性：第一道兜底盯着正文长度，正文被思考吃没了就用专家完整结论无工具地再综合一遍，保证用户一定拿得到一份有内容的简报；第二道兜底盯着参考来源，不让模型自己编 URL（它会瞎编一堆打不开的链接），而是把各专家真实检索命中的网页攒起来、去重后确定性地追加到简报末尾。它对项目的意义，是把本章的兜底主题补齐成三处——专家层的确定性检索、综合层的兜底再综合、来源层的确定性追加。三处合起来传达的是同一个工程判断：模型擅长的部分交给模型，模型不可靠的部分用代码兜住。

&emsp;&emsp;这里还要点一下主管运行时的两个参数，它们也是实测调出来的。一个是 `function_invocation_kwargs={"max_iterations": 3}`，它限制主管内部的工具调用轮数——一轮并行派发、可选地对"结论无效"的专家补派一轮、最后一轮综合，三轮够用且不会失控。另一个是主管这一步不设 `max_tokens`，因为综合作答需要把问题答透、不该被字数掐断；而专家那一步反过来要给足 `max_tokens`（给到 16000，要容纳最长 4096 字的正文加上 pro 的思考预算），原因下一节会讲。

### 8.5 实测复盘

&emsp;&emsp;最后把这个项目实测里值得记住的几条收束一下，都是真实跑下来踩出来的经验，留给我们自己上手时少走弯路。

> <font size=2>**提示**：深度思考模型要算清 max_tokens 和思考预算的账。pro 这类模型的 token 额度是"思考"和"正文"共用的，如果 max_tokens 给得太小，思考一占就把正文挤没了——实测把专家的 max_tokens 给到 1500，结果正文被截成了 8 个字的半截标题。所以本案例给专家足额的 max_tokens（16000，足以装下最长 4096 字的正文再加上 pro 的思考预算），主管综合那一步干脆不设上限。即便不设上限，pro 也偶发把额度全花在思考上、正文为空，这正是 8.4 兜底综合器存在的必要性。</font>

> <font size=2>**提示**：通过 OpenRouter 接第三方模型，要给请求都加上超时和重试。这个项目用的是 deepseek-v4-pro，又是多专家并行加综合，单轮花上 95 到 160 秒属于预期之内。但通过 OpenRouter 接入时会遇到长尾波动——多数请求 30 到 200 秒，个别请求会明显更慢。这是第三方端点的常态，给所有 LLM 调用都加上超时和重试，再用 SSE 把每一步进展实时推给前端，让这段等待变得可感知，体验才稳。</font>

> <font size=2>**提示**：中文区检索噪声大时，换全球区域重搜一次。本案例的检索默认走中文区（cn-zh），偏中文资讯，但它有时会返回跟问题不相关的热点新闻。8.3 那段 `call_expert` 里之所以写了"资料不足就换 `wt-wt` 全球区域再搜一次"的逻辑，就是为了兜这个噪声——全球区域的英文技术结果往往更准。所以一次没搜好不要急着让专家放弃，换个区域再给它一次机会，动画里也能看到"搜砸了再搜一次"这个真实过程。</font>

&emsp;&emsp;到这里，第八章就走完了。我们用 Microsoft Agent Framework 把 <font color=red>Supervisor 集中调度</font>落成了一个能真联网检索、能并行派单、能综合简报、能多轮追问的完整研究编排项目，看清了第三章那两块拼图——agents-as-tools 和 `as_agent` 底层写法——在真实业务里合体后的样子，也把整门课"模型不确定性、确定性兜底"这条主题线展开得最完整：专家层的确定性检索、综合层的兜底再综合、来源层的确定性追加，三处兜底环环相扣。

&emsp;&emsp;完整的工程代码——包含本章略去的检索预算控制、SSE 事件推送、服务端多轮记忆拼接、前端那套星形派单动画——都在项目源码 `代码/flagship-supervisor-tool/` 里，建议照着 8.2 的命令亲手跑一遍，看着七个专家在圆周上同时亮起，体感最深。

&emsp;&emsp;第三部分 的第三站，我们要把协作的复杂度再往上抬一层。前两个案例的主管都只管一层专家，但有些业务，比如"用户说做什么软件就给我造出来"，要的是分组分工、层层下钻、还要质检返工。下一章是一个多智能体软件交付项目：一个技术总监先动态定下接口定义，再把活分给前端、后端、测试三个组长，每个组长底下各带两三个工程师并行干活，测试不过还要带着错误堆栈跨组打回重写。它对应的正是第六章选型里那条产出物复杂、需要分层协同的 Hierarchical 分层协同，用的框架则回到 LangGraph。

## <center>第九章 多智能体软件交付（LangGraph 分层协同落地）</center>

&emsp;&emsp;这一章是整门课最重的一个项目。它要让一群智能体真造出一个能跑的软件，而且做什么软件由用户当场说了算——说做记账本就造记账本，说做待办清单就造待办清单。一个技术总监先定下接口定义，再把活分给前端、后端、测试三个组长，每个组长各带两三个工程师并行写代码、跑测试、做审查，测试不过还要带着错误堆栈跨组打回重写。底层我们用回最熟悉的 LangGraph，模型仍是 deepseek-v4-pro。

&emsp;&emsp;它的工程量也是四个案例里最大的——光后端的智能体编排和工具实现就有近千行。我们不会把这近千行逐行搬上来，那样既看不清重点、又把这一章撑爆。这一章只挑四段最能体现分层协同设计取舍的核心：怎么把控制流固定在图里、技术总监怎么动态定接口定义、组长怎么组织并行和返工、工具怎么真执行。完整工程都在项目源码里，建议照着本章最后的启动命令亲手跑一遍。

<div align=center><font size=2 color=#999999>第九章：技术总监、三个组长、若干工程师构成的三层组织，测试不过会带着错误堆栈打回后端重写</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L9-delivery-org-65f92c2d.png" width=80%></div>

<br>

&emsp;&emsp;上一章的 Supervisor 主管只管一层专家——七个专家全是它的直接下属，派单、收口都在主管和专家这一跨之间完成。这一章我们把组织再叠一层：技术总监不直接指挥工程师，而是只跟三个组长说话，组长再各自指挥手下的工程师。这正是第四章 4.1 层级机制里讲过的多层委派，也是第四章埋下的那个伏笔——当时我们用嵌套主管搭过一个"总监管两个组"的迷你层级，并说过"到 第三部分 会用最重的项目把分层协同彻底落地"，这一章就是来兑现它的。

&emsp;&emsp;接下来五个小节这样走：先讲清这个项目选 Hierarchical 的理由，以及控制流上一个关键的工程取舍；然后看技术总监怎么动态定下接口定义；再看三个组长怎么组织组内并行和质检返工，这一节会一并看清建图、返工、工具三段核心代码；接着把项目跑起来，看真实跑出来的结果；最后把这个项目踩过的几个坑收束成几条提示。

### 9.1 业务与选型

&emsp;&emsp;先看这个项目的需求长什么样：用户在输入框里提一句"做一个待办清单，能添加任务（标题、优先级）、查看列表、删除任务"，系统就要真把这个待办清单造出来——写好数据库、写好接口、写好页面，还要能点一下就在浏览器里跑起来。<b>产出物复杂、要分组分工、要质检返工</b>，这三个特征正是第六章选型里 Hierarchical 分层协同的典型业务画像：造软件天然要拆成前端、后端、测试几个工种，每个工种内部还要再分细活，而且产出的代码必须经过审查和测试才能算交付——这套"分组、分工、质检"的结构，单层的 Supervisor 接不住，正好是分层协同的主场。

&emsp;&emsp;选型定了，但真要落地，还有一个控制流上的关键取舍，它是这一章最值得记住的工程判断。乍一看，LangGraph 生态里有现成的 `create_supervisor`，第三章我们也用过——把一组智能体和一个主管模型交给它，它自动给主管注入派单工具，让主管自己决定"先派谁、再派谁、什么时候收口"。把控制流交给模型来发，听起来很优雅。但在 deepseek 上实测，这条路不稳。

> <font size=2>**提示**：把控制流交给模型自主决策，在 deepseek 上不稳定。我们用 `create_supervisor` 让主管模型自己发派单决策时，<b>同一份代码两次运行能跑出两种结果</b>——一次主管把三个组长全派了一遍，另一次它只派了一个组就自己收口了。模型对"接下来该交给谁"的判断带着随机性，而软件交付要求三个组都必须跑到、顺序还不能乱，这种不确定性是不能接受的。</font>

&emsp;&emsp;解决办法，是把"谁交给谁"这件事从模型手里收回来，固定在代码里。我们不让模型决定控制流，而是用 LangGraph 的 `StateGraph` 手工建一张图，把执行顺序固定成 `START → 前端组 → 后端组 → 测试组 → 技术总监综合 → END`，路由全部写在图的边上。模型只负责一件事——产出内容（写代码、做审查、综合总结），<b>控制流由代码定、内容由模型产</b>。这是第八章"确定性兜底"那条主题线的第二次回收：上一章我们用确定性的检索和兜底综合器，兜住了模型在检索和综合上的不可靠；这一章我们用确定性的图结构，兜住了模型在控制流决策上的不可靠。组长和工程师仍然是真的 LLM 在干活，工具也仍然是真执行，被固定的只是"流程怎么走"这一层。

### 9.2 先跑起来看效果

&emsp;&emsp;在拆开代码之前，我们先把项目跑起来，直观感受一下它的效果——后面再回头看每一处实现。这个项目和 第三部分 其余三个共用同一套工程底座，但它有自己独立的虚拟环境和端口——监听在本机 8090，虚拟环境是项目自带的 `.venv`。下面给三个平台的启动命令，按各自的终端环境挑一条。

```bash
# macOS / Linux：进项目目录，用项目专属的虚拟环境启动，监听 8090
cd 代码/hierarchical-software-delivery
.venv/bin/python -m uvicorn backend.app:app --host 127.0.0.1 --port 8090
```

```powershell
# Windows PowerShell：路径分隔符用反斜杠，虚拟环境的 python 在 Scripts 目录
cd 代码\hierarchical-software-delivery
.venv\Scripts\python.exe -m uvicorn backend.app:app --host 127.0.0.1 --port 8090
```

```bash
# Windows Git Bash：路径写法和 macOS 一致，正斜杠即可
cd 代码/hierarchical-software-delivery
.venv/bin/python -m uvicorn backend.app:app --host 127.0.0.1 --port 8090
```

&emsp;&emsp;服务起来后，浏览器打开 `http://127.0.0.1:8090` 就是这个<font color=red>软件交付项目</font>的操作界面。在输入框里提一个需求，比如"做一个待办清单，能添加任务（标题、优先级）、查看列表、删除任务"，按下发送，最先出现的一条轨迹是技术总监定接口定义——可以看到它把这个需求转成了一份接口定义，应用名、数据表、字段、接口路径、统计指标都在里面。接口定义定完，三个组长依次开工，界面中央是一棵组织树，节点是组长（不是模糊的团队框），每个组长底下挂着它派出去的工程师。

<div align=center><font size=2 color=#999999>多智能体软件交付运行界面（http://127.0.0.1:8090）</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L9-run-866d83e0.png" width=80%></div>

&emsp;&emsp;工程师写完一个文件，产出区那棵文件夹树就多一个文件，点开能看到这一版的完整内容和与上一版的逐行差异。测试与审查单独有一个标签页，代码审查、功能测试、性能测试的结论都在里面；如果功能测试没过，轨迹里会出现一条"打回重写"，紧接着后端工程师重新写一遍、测试再跑一次。全部跑完，技术总监给出一份交付总结，页面上出现"启动预览"按钮，点一下就用内嵌窗口把刚造出来的软件跑起来，能直接在浏览器里添加、查看、删除——一个完整的应用就这么造好了。

&emsp;&emsp;这套流程是实测跑通的。下面这张表把 lab-records 里这个项目的实测结果列出来，都是端到端跑出来的。

<div align=center><font size=2 color=#999999>多智能体软件交付实测结果（deepseek-v4-pro，端到端验证）</font></div>

<div align="center">
<table width="80%">
<thead><tr>
<th>验证维度</th><th>实测结果</th>
</tr></thead>
<tbody>
<tr><td>整体可交付率</td><td>deepseek-v4-pro 连跑 5 轮，5 轮造出的软件都能跑起来</td></tr>
<tr><td>动态接口定义</td><td>造过记账本（接口 <code>/api/expenses</code>）和待办清单（技术总监自己起名 <code>/api/tasks</code>）</td></tr>
<tr><td>业务聚合</td><td>待办清单的统计自带 <code>high_priority_count</code>、<code>today_count</code> 这类有业务含义的聚合</td></tr>
<tr><td>单轮规模</td><td>一轮约 70 个事件，端到端 165 到 310 秒（OpenRouter 偶发更慢）</td></tr>
<tr><td>真实返工</td><td>功能测试抓到过运行时崩溃，打回后端重写一次后变为可交付</td></tr>
</tbody>
</table>
</div>

<br>

&emsp;&emsp;这张表里最值得玩味的是动态接口定义那两行。同一套代码，用户说做记账本，技术总监就定下 `/api/expenses` 接口、金额和分类字段；用户说做待办清单，它就自己起名 `/api/tasks`，还给待办清单配了"高优先级任务数""今天的任务数"这种带业务含义的统计——这些都不是预先定好的，是模型读懂需求后现定的。固定骨架保证了它们都能测、能跑，动态接口定义保证了它们各自贴合自己的业务，这正是 9.3 那套"骨架固定、内容动态"设计在真实运行里的样子。

&emsp;&emsp;效果看到了，下面我们回过头，把这套效果背后的实现一层层拆开。

### 9.3 动态接口定义

&emsp;&emsp;分层协同要让三个组并行干活，有一个前提必须先解决：三个组得对"我们在造什么"有一致的理解。前端组写的页面元素 `id`、后端组写的接口路径、测试组要打的端点，必须严丝合缝地对上，否则各写各的，联调时全是错位。但这个项目的难点恰恰在于——<b>做什么软件是用户当场定的</b>，没法把字段和接口预先定在代码里。

&emsp;&emsp;这里用的办法是<b>动态接口定义</b>：在三个组开工之前，技术总监先用一次 LLM 调用，根据用户需求生成一份接口定义。接口定义是一个 JSON，写明这次要造的应用叫什么、数据表叫什么、有哪些字段、接口路径是什么、统计哪些指标、用什么数据做功能测试。这份接口定义随后发给所有工程师当公共上下文，三个组照着同一份接口定义实现，自然就对得上了。下面是技术总监生成接口定义的核心逻辑。

In [ ]:
# 解析失败时回退到这份记账本接口定义（字段已省略，完整版见源码），保证任意需求下流程都能继续
FALLBACK_CONTRACT = {
    "app_name": "极简记账本", "table": "expenses",
    "fields": [{"name": "amount", "type": "REAL", "label": "金额", "required": True}, ...],
    "api_base": "/api/expenses", "stats_elems": [{"id": "stat-total", "label": "总支出"}],
}

async def _make_contract(topic: str):
    """技术总监按用户需求定接口定义。返回 (接口定义 dict, 是否成功);解析失败回退记账本接口定义。"""
    try:
        # 让低温模型只输出一个接口定义 JSON，约束它取 2-4 个核心字段、1-2 个有意义的统计
        raw = await _llm(_CTO_CONTRACT_SYS, f"用户需求:{topic}\n\n只输出接口定义 JSON。", model=code_model)
        m = re.search(r"\{.*\}", raw, re.S)              # 从模型回复里抠出 JSON 那一段
        contract = _normalize_contract(json.loads(m.group(0)))   # 解析 + 校验字段合法性
        return contract, True
    except Exception:
        # 模型没按格式输出、JSON 解析失败、字段校验不过——任意一步出错都回退默认接口定义
        return json.loads(json.dumps(FALLBACK_CONTRACT)), False

&emsp;&emsp;这段代码实现的是接口定义的动态生成与解析兜底。它在流程里的作用是给整个交付链路定下统一的接口规格——后面三个组的所有实现都以这份接口定义为准。它在项目里的意义，是把"做什么由用户定"这件不确定的事，收敛成一份结构确定的 JSON。这里又能看到确定性兜底的影子：模型生成接口定义这一步可能失败（没按格式输出、JSON 残缺、字段非法），所以 `try` 块外面接了一个 `except`，任何一步出错都回退到那份预置的记账本接口定义，保证流程绝不会因为接口定义生成失败而中断。代码里的 `_normalize_contract` 是一道校验关——它检查表名、字段名是否合法、补全缺失的默认值，校验不过就抛异常交给外层回退，细节在项目源码里，这里理解它"校验不合法就让上层回退"的职责即可。

&emsp;&emsp;有了动态接口定义，还需要回答一个问题：既然每次造的软件都不一样，怎么保证造出来的东西一定能测、能跑？答案是<b>固定骨架 + 变接口定义内容</b>。变的只是接口定义里的字段和接口名，但有一整套骨架是固定的——文件永远是数据层、服务层、页面、交互脚本这四件套，数据层暴露的函数名永远是建表、新增、列表、删除、统计这五个，接口永远是"取列表、新增、删除、统计"这几个固定模式，页面元素的 `id` 永远等于字段名。三个组照着这套固定骨架实现，测试组也照着这套骨架去打——不管这次造的是记账本还是待办清单，骨架不变，测试逻辑就不用改。

> <font size=2>**提示**：动态接口定义的本质是在<b>自由度和可验证性之间找平衡</b>。完全固定字段，就只能造一种软件，没有自由度；完全放开让模型自由发挥，又没法写出通用的测试和校验。把"骨架固定、内容动态"这条线划好，既能造任意单表应用，又能用同一套测试和自检逻辑覆盖所有情况——这是这个项目能做到"任意需求都可测可启动"的关键设计。</font>

### 9.4 组织与返工

&emsp;&emsp;接口定义定好了，接下来就是三个组照着接口定义干活。这一节我们把这个项目最核心的三段代码一次看清：先看组织结构怎么用 `StateGraph` 建出来，再看组长怎么组织组内并行和返工，最后看工具怎么真执行。

&emsp;&emsp;先看组织结构。整个项目由三个组长撑起来：前端组长带 UI 工程师和交互工程师，后端组长带数据库工程师和 API 工程师，测试组长带代码审查、功能测试、性能测试三个角色。在图里，<b>每个节点就是一个组长</b>，组长内部再去调度自己手下的工程师。9.1 说过控制流要固定，下面这段就是把这套组织结构和执行顺序固定下来的建图代码。

In [ ]:
class DeliveryState(TypedDict):
    topic: str           # 用户的软件需求
    contract: dict       # 技术总监动态制定的接口定义
    leads_done: dict     # 各组长的交付小结
    final: str           # 技术总监最终综合的交付总结

def _build_graph():
    g = StateGraph(DeliveryState)
    for L in LEADS:                              # LEADS = [前端组长, 后端组长, 测试组长]
        g.add_node(L["id"], _make_lead_node(L))  # 每个组长是图里的一个节点
    g.add_node("cto_synthesize", cto_synthesize) # 技术总监综合节点
    g.add_edge(START, LEADS[0]["id"])            # START → 前端组
    for a, b in zip(LEADS, LEADS[1:]):           # 前端 → 后端 → 测试，顺序固定在边里
        g.add_edge(a["id"], b["id"])
    g.add_edge(LEADS[-1]["id"], "cto_synthesize")# 测试组 → 技术总监综合
    g.add_edge("cto_synthesize", END)            # 综合完 → 结束
    return g.compile()  # 把状态图编译成可执行的图

&emsp;&emsp;这段代码实现的是整个交付流程的骨架。它在流程里的作用是把"先前端、再后端、再测试、最后总监综合"这个顺序固定下来——所有的边都是手工 `add_edge` 连死的，没有任何一条路由交给模型去决定。它在项目里的意义正是 9.1 那个工程判断的落地：图结构本身就是那道确定性兜底，模型在每个节点里产出内容，但下一步走到哪，由图说了算。这和第四章 4.2 节嵌套主管那段代码正好是一对参照——那边每层主管自主派单、控制权层层让渡，这边把层级用图结构连死、模型只产内容不掌路由，同一个三层组织的两种做法，生产项目选了更可控的这一种。

&emsp;&emsp;再看每个组长节点内部干了什么。组长不是简单地把任务丢给工程师就完事，它要做三件事：把活并行派给手下的工程师、自检产出是否合格、不合格就打回重写。下面是组长节点的核心。

In [ ]:
def _make_lead_node(L: dict):
    """组长节点:组内并行派发 → 自检 →【不合格打回重写，最多 2 次返工】→ 汇总。"""
    async def lead_node(state: DeliveryState):
        topic, contract = state["topic"], state["contract"]
        # 组内并行:asyncio.gather 让本组两个工程师同时写各自负责的文件
        outs = list(await asyncio.gather(*[_run_worker(L["id"], w, topic, contract) for w in L["workers"]]))

        chk = await _lead_inspect(L, contract)      # 组长自检:真读文件，按接口定义逐项核对
        coders = [w for w in L["workers"] if w["kind"] == "coder"]
        attempt = 0
        while coders and not chk.get("ok") and attempt < 2:    # 自检不过，最多打回重写 2 次
            attempt += 1
            failed = [c["name"] for c in chk.get("checks", []) if not c["pass"]]
            fb = "组长自检未通过项:" + "、".join(failed) + "。请针对性修正后重写完整文件。"
            outs = await asyncio.gather(*[_run_worker(L["id"], w, topic, contract, feedback=fb) for w in coders])
            chk = await _lead_inspect(L, contract)  # 重写后再自检，直到通过或用完 2 次机会
        return {"leads_done": {**state.get("leads_done", {}), L["id"]: ...}}
    return lead_node

&emsp;&emsp;这段代码实现的是组长的"并行派发 + 质检返工"闭环。它在流程里的作用是把组内的工程师管起来——`asyncio.gather` 让本组的工程师同时干活（前端组的 UI 和交互工程师并行写页面和脚本，后端组的数据库和 API 工程师并行写存储和接口），写完组长用 `_lead_inspect` 真读文件、按接口定义逐项核对，不合格就把具体的未通过项作为反馈打回工程师重写，最多两次。它在项目里的意义是把"组内并行换吞吐、组长把关保质量"这套人事管理逻辑用代码表达了出来——组间是顺序下钻（一个组跑完才到下一个组，因为后端要等前端定下页面元素），组内则是并行换速度。

&emsp;&emsp;比组长自检更有意思的是<b>测试驱动返工</b>。组长的自检靠的是静态核对——读文件看函数名对不对、接口路径在不在，但这种静态检查抓不出实现层面的运行时错误。测试组长这里多了一道闭环：功能测试真跑起来失败时，它会带着错误堆栈跨组打回后端工程师重写，再重测一次。

In [ ]:
# 测试组专属:功能测试失败 → 带错误堆栈跨组打回后端工程师重写 → 重测一次
if L["id"] == "test":
    func = next((t for t in reversed(TESTS.get() or []) if t.get("kind") == "functional"), None)
    if func and not func.get("ok"):
        err = func.get("error") or ";".join(c["name"] for c in func.get("checks", []) if not c["pass"])
        # 把错误信息 + 堆栈片段一起塞进反馈，让后端工程师知道到底哪里崩了
        fb = f"功能测试未通过:{err}\n错误堆栈片段:\n{(func.get('trace') or '')[-600:]}\n请定位并修复后重写完整文件。"
        backend = next(x for x in LEADS if x["id"] == "backend")   # 跨组找到后端组
        await asyncio.gather(*[_run_worker("backend", w, topic, contract, feedback=fb)
                               for w in backend["workers"] if w["kind"] == "coder"])
        # 后端重写完，功能测试再打一次
        ftw = next(w for w in L["workers"] if w["kind"] == "functional")
        outs.append(await _run_worker(L["id"], ftw, topic, contract))

&emsp;&emsp;这段代码实现的是跨组的测试驱动返工。它在流程里的作用是补上静态检查的盲区——`py_compile` 这类语法检查只能保证代码"能编译"，但保证不了它"跑起来是对的"。它在项目里的意义，是给整个交付链路加了一道真实运行的兜底：功能测试是真用 FastAPI 的 `TestClient` 去打生成的应用，POST 一条数据再 GET 读回来看通不通，跑挂了就把错误堆栈原样喂回给后端工程师，让它照着真实的报错去修，而不是凭空猜。这道闭环在实测里抓到过真实的运行时崩溃，9.5 会专门讲那个案例。

> <font size=2>**提示**：组长自检和测试驱动返工是<b>两道不同层次的关</b>。组长自检是静态核对（函数名、接口路径、页面元素是否齐全），抓的是"写漏了、写错名字"这类问题，每个组都有；测试驱动返工是动态运行（真跑 POST 再 GET），抓的是"编译能过但运行时崩"这类深层问题，只有测试组才有。两道关一静一动叠在一起，才把 LLM 生成代码的质量兜住——只靠静态检查会放过运行时错误，只靠运行测试又太慢、定位不到具体哪个文件。</font>

&emsp;&emsp;最后说一下工具。上面这套并行、自检、返工能成立，全靠工程师手里的工具都是<b>真执行</b>的，不是摆设。

> <font size=2>【名词解释】<b><font color=red>TestClient</font>（Test Client，测试客户端）</b>:FastAPI 自带的测试工具，不用真启动服务器就能在进程内直接对应用发起请求，常用来验证接口读写是否打通。</font>

&emsp;&emsp;这个项目给工程师配了一整套真执行的工具，挑三个最关键的看一眼它们各自的签名和职责。

In [ ]:
# write_code_file(relpath, content) -> {path, diff, bytes}
#   把生成的代码真落盘到工作区，并用 difflib 算出与上一版的 unified diff，
#   前端那棵文件夹树点开就能看到这一轮改了哪几行
# run_functional_test(contract) -> {ok, checks}
#   用 FastAPI 的 TestClient 按接口定义真打接口：先 POST 一条测试数据，再 GET 把列表
#   读回来，断言刚写的那条记录能被读到——这是"读写真打通了没有"的硬证据
# launch_app() -> {url, port, pid}
#   技术总监验收通过后，用 uvicorn 在空闲端口把应用真起起来，健康检查通过后返回
#   可访问地址，前端用内嵌窗口直接预览——就是页面上"启动预览"按钮背后做的事

> <font size=2>【名词解释】<b><font color=red>unified diff</font>（unified diff，统一差异格式）</b>:展示两个文本版本差异的标准格式，用减号开头的行表示删掉的、加号开头的行表示新增的，是 git 等工具默认的差异呈现方式。</font>

&emsp;&emsp;这三个工具的共同点是<b>全程没有一步是假装</b>：`write_code_file` 真把文件写进磁盘，`run_functional_test` 真用 `TestClient` 在进程内把应用打一遍，`launch_app` 真用 `uvicorn` 把服务跑起来。"写代码 → 测读写 → 起服务"这条链路因此都是真实发生的，组长的自检、测试组的功能测试、技术总监的验收，核对的也都是这些工具真执行后留下的产物——这正是 9.4 那套并行、自检、返工能成立的底座。工具的完整实现（含静态检查、性能连打、组长自检等另外几个工具）在项目源码里。

### 9.5 实测复盘

&emsp;&emsp;最后把这个项目实测里值得记住的几条收束一下，都是真实跑下来踩出来的经验，留给我们自己上手时少走弯路。

> <font size=2>**提示**：`dict(row)` 是 LLM 生成 SQLite 代码的高频出错点，要用"事先约束 + 事后测试"双保险兜住。SQLite 的游标默认返回的是元组而不是字典，模型写数据层代码时很容易顺手写出 `dict(row)` 想把一行转成字典，运行时直接报错。这个项目用了两道保险：一是在给数据库工程师的任务说明里就<b>事先写明</b>"严禁直接 `dict(row)`，要么显式写键值对、要么先设 `row_factory = sqlite3.Row`"，从源头预防；二是 9.4 那道<b>测试驱动返工</b>兜底——万一模型还是写错了，功能测试真跑会立刻崩，带着错误堆栈打回后端重写。这正是测试驱动返工存在的价值，它在实测里真的抓到过这个崩溃。</font>

> <font size=2>**提示**：`(lst or []).append()` 会悄悄丢数据，要显式判空再就地追加。代码里想往一个可能为空的列表追加元素时，很容易顺手写成 `(lst or []).append(x)`，但这里有个陷阱：空列表在 Python 里是假值，`[] or []` 返回的是 `or` 右边那个<b>新建的临时列表</b>，`append` 加到了这个临时列表上，原来的列表一个元素都没加进去，数据就这么悄无声息地丢了。正确写法是先显式判断"如果列表不是 None"，再对原列表就地 `append`。</font>

> <font size=2>**提示**：接口定义动态化之后，手动测接口要先看这一轮的接口定义。因为接口路径和字段都是技术总监按需求现定的，造记账本时是 `/api/expenses`，造待办清单时就成了 `/api/tasks`，没有一个固定的端点可背。所以想用 `curl` 或浏览器手动验证接口时，第一步先去轨迹里看技术总监这一轮定下的接口定义，照着接口定义里的 `api_base` 和字段名来打，否则很容易对着上一轮的旧路径打、然后收到一个让人困惑的 404。</font>

&emsp;&emsp;到这里，第九章就走完了。我们用 LangGraph 把 <font color=red>Hierarchical 分层协同</font>落成了一个能真造软件的完整项目——技术总监动态定接口定义、三个组长各带工程师并行干活、测试不过带着错误堆栈跨组打回重写，把第四章 4.1 层级机制里埋下的"软件交付案例"伏笔彻底兑现了。更重要的是，这一章把整门课"模型不确定性、确定性兜底"这条主题线推到了最强：第八章我们用确定性的检索和兜底综合器，兜住模型在内容上的不可靠；这一章我们干脆把整个控制流固定在 `StateGraph` 的边里，模型只产内容、不碰流程，确定性兜底从"兜内容"升级到了"兜流程"。

&emsp;&emsp;完整的工程代码——包含本章略去的接口定义校验细节、四个工程师各自的任务说明、静态检查与性能测试工具、前端那棵文件夹树和启动预览——都在项目源码 `代码/hierarchical-software-delivery/` 里，建议照着 9.2 的命令亲手跑一遍，提一个自己想造的需求，看着三个组把它真造出来，体感最深。

&emsp;&emsp;第三部分 的最后一站，我们要换一种完全不同的协作形态。前三个案例不管是流水线、集中调度还是分层协同，都有一个明确的"指挥者"在掌控全局。但在线客服这种场景不一样——用户先问退款、又转头问物流、最后还想开发票，会话焦点一直跟着人走，没有一个固定的中心能接得住。下一章是一个在线客服中心项目：一个分诊台加六个专员全互联，谁接得住就由当前这个专员自己决定把会话交给谁，控制权在平级之间像接力棒一样传递。它对应的正是第五章讲过的 Swarm 自主协作，用的框架是 OpenAI Agents SDK。

## <center>第十章 案例四 · 在线客服中心（OpenAI Agents SDK 落地 Swarm）</center>

&emsp;&emsp;前三个案例落地的是三种<font color=red>"有中心"</font>的协作：第七章的销售报表，工序固定、谁先谁后固定在代码里；第八章的研究编排，一个中央主管攥着控制权调度专家；第九章的软件交付，技术总监到组长到工程师层层下钻。它们有一个共同点——总有一个角色在掌控全局，控制权始终向上汇集。这一章我们要落地第四种、也是控制最弱的那一种：Swarm 自主协作。在线客服正是它的典型场景，用户先问一句退款，问着问着又想查物流，最后还要开发票，会话焦点一路跟着人走，事先没有任何一个固定的中心能预判这条对话会拐到哪里。

&emsp;&emsp;这正好接上第五章 5.6 讲过的业界对 Swarm 的保留态度。当时我们的结论是平衡的：各家官方文档普遍把集中调度列为默认推荐，对去中心化的接力持保留态度，原因是控制流不可见、调试归因难——但这份保留态度针对的是"用 Swarm 做任务编排"，而对话流转这类会话焦点随用户漂移的业务，Swarm 仍是最贴合的工具。这一章就是用一个完整的客服项目把这句话证实：分诊台只负责把第一句话分流给对的专员，六个专员之间全互联，谁接手之后发现问题不归自己管，就由它自己决定把会话交给哪个同事，控制权在平级之间像接力棒一样传递。

&emsp;&emsp;我们这一章分五步走：先看业务为什么落到 Swarm、分诊台加六专员的组织怎么搭（10.1）；再把项目跑起来，看这张网上 handoff 真实发生的样子和实测结果，直观感受效果（10.2）；接着看专员和工具怎么写，重点是六专员全互联的建法和一个完整的开票闭环（10.3）；然后看多轮对话怎么续接，让控制权能在专员之间跨轮反复交换（10.4）；最后把这个项目实测踩出来的经验收束成几条提示，其中第一条正是第五章 5.2 那个招牌坑在完整项目里的实战复现（10.5）。本案例基于 OpenAI Agents SDK 的 `openai-agents` 0.17.4 实测，模型沿用全课统一的 `deepseek-v4-pro`，走 OpenRouter 端点。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L10-service-mesh-f6529e3f.png" width=80%></div>

### 10.1 业务与选型

&emsp;&emsp;先把业务讲清楚。我们要做的是一个在线客服中心：用户进来抛一个问题，系统判断这是什么类型的诉求，把它交给对应的专员处理；如果聊着聊着诉求变了，或者当前专员发现这事不归自己管，控制权还能再交给别的专员。这个客服团队由一个分诊台加六个专员组成——售前咨询、退款售后、技术支持、账单财务、物流查询、投诉处理。分诊台的职责很单一，它只判断用户第一句话的主要意图、立刻分流，绝不自己回答任何业务问题，也绝不寒暄展开。真正干活的是六个专员，每个专员手里握着一组能真查数据库的工具，按自己的专长据实回复。

&emsp;&emsp;这个业务落到 Swarm，判断标准还是第六章给的那条判据：会话焦点会不会随用户漂移、有没有一个角色能事先掌控全程。客服场景两条都指向 Swarm。用户的诉求事先不可预判，更要命的是它会在对话中途变——一个人进来问"我的包裹到哪了"，物流专员查完告诉他还在路上，他紧接着说"那太慢了我要退款"，这时会话焦点已经从物流漂到了退款，物流专员接不住，得把会话交给退款专员。没有任何一个中心能预先排好这条对话的走向，谁该接手只能由当前正在对话的那个专员临场判断。控制权交还给每一个 agent、让它们自主协作，这正是 Swarm 的主场。

> <font size=2>【名词解释】<b><font color=red>OpenAI Agents SDK</font>（OpenAI Agents SDK，OpenAI 智能体开发套件）</b> — OpenAI 推出的轻量级多智能体开发框架，第五章讲过它是早期 Swarm 实验库的生产化接棒者，handoff（控制权交接）是它的四个核心原语之一。本案例基于它的 `openai-agents` 0.17.4 实测。</font>

&emsp;&emsp;这里要呼应第五章 5.6 那个保留态度的结论，把它跟本章对齐。第五章我们说过，各家官方文档对 Swarm 持保留态度，是因为去中心化的接力让控制流变得不可见、出了问题难以归因，所以它们在做任务编排时普遍默认集中调度。但客服不是任务编排——它是对话流转。任务编排的目标是把一个明确的任务拆开、并行做完、收口汇总，那种场景确实更适合一个中央大脑统筹；而对话流转的目标是应对一个会拐弯的人，会话焦点在哪、下一步交给谁，本来就该由临场的那个专员决定。所以保留态度成立，本章的选型也成立，两者并不矛盾——它们针对的是不同的业务形态。

> <font size=2>【名词解释】<b><font color=red>handoff</font>（handoff，控制权交接）</b> — Swarm 模式里一个 agent 把整个会话的控制权移交给另一个 agent 的动作。在 OpenAI Agents SDK 里，给一个 Agent 配上 `handoffs=[其他 Agent]`，框架就会自动把"交接给某专员"这件事暴露成一个工具，agent 调用它就完成了控制权转移。这与第八章 agents-as-tools 的根本区别在于：handoff 之后控制权真的转走了，而 agents-as-tools 里控制权始终留在主管手里。</font>

### 10.2 先跑起来看效果

&emsp;&emsp;在拆开代码之前，我们先把项目跑起来，直观感受一下它的效果——后面再回头看每一处实现。这个项目和 第三部分 其余三个共用同一套工程底座，但有自己独立的虚拟环境和端口——监听在本机 8092，虚拟环境是项目自带的 `.venv`。下面给三个平台的启动命令，按各自的终端环境挑一条。

```bash
# macOS / Linux：进项目目录，用项目专属的虚拟环境启动，监听 8092
cd 代码/swarm-customer-service
.venv/bin/python -m uvicorn backend.app:app --host 127.0.0.1 --port 8092
```

```powershell
# Windows PowerShell：路径分隔符用反斜杠，虚拟环境的 python 在 Scripts 目录
cd 代码\swarm-customer-service
.venv\Scripts\python.exe -m uvicorn backend.app:app --host 127.0.0.1 --port 8092
```

```bash
# Windows Git Bash：路径写法和 macOS 一致，正斜杠即可
cd 代码/swarm-customer-service
.venv/bin/python -m uvicorn backend.app:app --host 127.0.0.1 --port 8092
```

&emsp;&emsp;服务起来后，浏览器打开 `http://127.0.0.1:8092` 就是这个<font color=red>在线客服项目</font>的操作界面。界面中央和前三个案例都不一样——七个等大节点摆成一圈、彼此之间都可以连线交接，组成一张无中心的网，分诊台和六个专员平级排布，这正是 Swarm 网状结构的样子。正中位置有一个小徽章，显示当前正在接待的是谁。在输入框里发一句话，比如"我的订单 SO-1003 到哪了，顺便问下能退货吗"，最先发生的是分诊台把它分流给某个专员，网上随即出现一条 handoff 连线箭头，中央徽章切换成接手的专员；如果对话过程中又发生了专员之间的交接，网上会再画出新的箭头，走过的交接路径留成虚线，控制权在专员间一步步传递的过程一目了然。

<div align=center><font size=2 color=#999999>在线客服中心运行界面（http://127.0.0.1:8092）</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L10-run-c2f48183.png" width=80%></div>

&emsp;&emsp;主对话区的呈现也专门为这种多专员协作做了设计。用户说的话是一种气泡，每个专员"回复用户"的话是另一种白底气泡、逐条常显——这样一通对话里哪句是退款专员答的、哪句是账单财务专员答的，看得清清楚楚；而专员查工具、做交接这些中间过程，则折叠成一行"过程·N 步"，想看细节再点开。界面顶部还有一个"查看数据库"面板，订单、发票、知识库三张表分页可看，方便我们对照专员的回答和数据库里的真实数据是不是对得上。

&emsp;&emsp;这套流程是实测跑通的。下面这张表把 lab-records 里这个项目的实测结果列出来，都是端到端跑出来的。

<div align=center><font size=2 color=#999999>在线客服中心实测结果（deepseek-v4-pro，OpenAI Agents SDK，端到端验证）</font></div>

<div align="center">
<table width="80%">
<thead><tr>
<th>验证维度</th><th>实测结果</th>
</tr></thead>
<tbody>
<tr><td>分诊准确</td><td>六类典型诉求分诊 6/6 全部交给了对的专员</td></tr>
<tr><td>专员互转</td><td>logistics→refund、refund→tech、refund→billing 等专员间互转链真实发生，一轮最多 4 跳</td></tr>
<tr><td>业务规则兜底</td><td>已退款订单依规拒开发票；tech 拦截过"前面说退款现在又要开票"的业务冲突</td></tr>
<tr><td>开票闭环</td><td>全流程三轮走完：查资格 → 收集抬头税号 → 真开票落库 <code>INV-2026-1003</code></td></tr>
<tr><td>跨轮续接</td><td>跨轮指代成立，用户说"她最近那单"能续上上一轮的上下文</td></tr>
<tr><td>单轮耗时</td><td>10 到 50 秒，是四个案例里最快的</td></tr>
</tbody>
</table>
</div>

<br>

&emsp;&emsp;这张表里最能体现 Swarm 价值的是专员互转那一行。`logistics→refund` 就是前面说的那个典型场景——用户查完物流转头要退款，物流专员接不住、把会话真交给了退款专员；而 `refund→billing` 则是退款专员发现用户其实想问发票、又把会话递给了账单财务专员。这些交接都不是事先排好的，是当前专员临场判断"这事不归我管"之后自己决定交给谁的，控制权就这么在平级之间一跳一跳地传下去。业务规则兜底那一行也值得看——已退款订单依规拒开发票，靠的正是 10.3 那个 `check_invoice_eligibility` 工具里硬编码的状态校验，模型再怎么想配合用户也开不出来，这就是去中心化协作里那道确定性栏杆在起作用。

&emsp;&emsp;效果看到了，下面我们回过头，把这套效果背后的实现一层层拆开。

### 10.3 专员与工具

&emsp;&emsp;这一节是本章的技术核心。在搭组织之前，先把一件基础设施的事情交代清楚——模型怎么接进来。我们用的是 `deepseek-v4-pro`，走 OpenRouter 端点，而 OpenAI Agents SDK 默认连的是 OpenAI 官方。把它指到 OpenRouter，标准做法是先建一个把 `base_url` 指向 OpenRouter 的 `AsyncOpenAI` 客户端，再用 `OpenAIChatCompletionsModel` 把这个客户端和模型名包起来，最后把追踪关掉。

In [ ]:
from openai import AsyncOpenAI  # OpenAI 官方异步客户端，指向 OpenRouter 兼容端点
from agents import OpenAIChatCompletionsModel, set_tracing_disabled  # OpenAI Agents SDK：模型接兼容端点 + 关闭轨迹上报

set_tracing_disabled(True)                                   # 关掉 OpenAI 官方追踪上报（端点不是官方）
client = AsyncOpenAI(base_url=BASE_URL, api_key=KEY)          # base_url 指向 OpenRouter
model = OpenAIChatCompletionsModel(model=MODEL, openai_client=client)  # 包成 SDK 能用的模型对象

&emsp;&emsp;这段代码实现的是把第三方端点接进 OpenAI Agents SDK。它在流程里的作用是给后面所有 Agent 提供一个统一的 `model` 对象——`AsyncOpenAI` 把请求指向 OpenRouter 而不是 OpenAI 官方，`OpenAIChatCompletionsModel` 再把它包装成 SDK 内部能用的模型实例，`set_tracing_disabled(True)` 则关掉默认的追踪上报（追踪要回连 OpenAI 官方，端点既然换了就关掉避免噪声）。它对项目的意义在于，这三行是本案例能用 `deepseek-v4-pro` 跑通的前提，后面建的每一个专员、分诊台，`model` 参数填的都是这里建好的这个对象。

&emsp;&emsp;接进模型之后，看组织是怎么搭起来的。Swarm 的结构是一张无中心的网，分诊台和六个专员是平级的节点，专员之间要能两两互相交接、还要能退回分诊台，这就是所谓的全互联 handoffs。这个全互联不能一步建成，因为建专员的时候它要交接的那些同事还没建出来，存在互相引用。下面这段是搭建这套结构的核心，它分三步把全互联建立起来。

In [ ]:
def build_agents():
    """先建 6 专员（handoffs 待补）→ 建分诊台 → 把专员互联，实现平级自由交接。"""
    agents = {}
    # 第一步：先建 6 个专员，handoffs 暂时留空——此刻别的专员还没建出来
    for s in SPECIALISTS:                    # SPECIALISTS 是 6 个专员的配置（英文 name + 中文展示）
        agents[s["name"]] = Agent(
            name=s["name"],                  # 关键：name 必须英文，原因见 10.5
            handoff_description=s["desc"],   # 交接时给别人看的"我负责什么"
            instructions=s["instr"],         # 中文指令全放这里
            model=model, tools=s["tools"])   # 每个专员挂自己那组真查库工具
    # 第二步：建分诊台，handoffs = 全部 6 个专员（它只分流，能交给任何一个）
    triage = Agent(name="triage", handoff_description="客服分诊台",
                   instructions=TRIAGE_INSTR, model=model,
                   handoffs=list(agents.values()))
    # 第三步：把专员互联——每个专员能交回分诊台 + 交给除自己外的所有其他专员
    for s in SPECIALISTS:
        agents[s["name"]].handoffs = [triage] + [agents[o["name"]] for o in SPECIALISTS
                                                 if o["name"] != s["name"]]
    return triage, agents

&emsp;&emsp;这段代码实现的是 Swarm 全互联结构的搭建。它在流程里的作用是把"分诊台只分流、六专员平级自由交接"这套组织关系真正建立起来——第一步先把六个专员建出来、handoffs 先留空，因为这时候要交接的同事都还没存在；第二步建分诊台，它的 handoffs 直接是全部六个专员，意味着它能把第一句话分流给任何一个；第三步回过头把每个专员的 handoffs 补上，让它既能交回分诊台、又能交给除自己之外的所有其他专员。它对项目的意义在于，这个三步建法是去中心化接力能成立的物理基础——六个专员两两之间都连着一条交接通道，所以退款专员发现用户其实想开发票时，能直接把会话递给账单财务专员，不必绕回分诊台。

&emsp;&emsp;这里第一行注释标的 `name` 必须英文，正是第五章 5.2 那个招牌坑在完整项目里的落地，10.5 会完整复盘它当初是怎么把整个 handoff 搞崩的。专员真正的能力来自它们手里的工具。每个专员都挂了一组用 `@function_tool` 写的工具，对客服数据库做真查询，不是凭空编。退款专员能按订单号或本人姓名查订单状态，物流专员能查物流单号，账单财务专员能查开票资格、还能真开发票。我们重点看账单财务专员手里那对开票工具，它把一个完整的多轮业务流程怎么在 Swarm 里落地讲得最清楚。

> <font size=2>【名词解释】<b><font color=red>@function_tool</font>（function tool，函数工具装饰器）</b> — OpenAI Agents SDK 提供的装饰器，给一个普通 Python 函数加上它，框架就会自动读取函数签名和文档字符串，把它包装成 agent 能调用的工具。本案例的查订单、查物流、查开票资格、真开发票，都是用它装饰的真查库函数。</font>

&emsp;&emsp;开票这件事在客服里天然是多轮的：不能用户一说"我要开发票"就直接开，得先查这个订单到底能不能开，能开还得问清楚开什么类型、抬头写什么。所以账单财务专员手里是两个配合的工具——先用 `check_invoice_eligibility` 查资格，资格通过、信息收齐后再用 `issue_invoice` 真开。先看查资格这个工具。

In [ ]:
_INVOICEABLE = ("已付款", "备货中", "已发货", "已签收")    # 只有这四种状态可以开票

@function_tool  # 把普通函数标成 agent 可调用的工具
def check_invoice_eligibility(order_id: str) -> str:
    """判断某订单能否开票：校验订单存在、状态可开、是否已开过，并返回下一步要收集的信息。"""
    oid = (order_id or "").strip().upper()
    o = _query_order(oid)                       # 查订单
    if not o:                                   # 关卡一：订单不存在
        return f"没查到订单 {order_id}，请确认订单号。"
    inv = _query_invoice(oid)                   # 查这个订单是否已开过发票
    if inv:                                     # 关卡二：已开过 → 拒绝重复开
        return f"订单 {oid} 已开过发票 {inv[0]}，不能重复开。"
    if o["status"] not in _INVOICEABLE:         # 关卡三：状态不可开（已退款/已取消等）
        return f"订单 {oid} 当前状态【{o['status']}】不符合开票条件。"
    # 三关都过 → 告诉专员该向用户收集哪些开票信息
    return (f"订单 {oid} 可以开票，金额 ¥{o['amount']}。请收集：① 发票类型"
            "（电子普通发票 / 增值税专用发票）；② 抬头；③ 专票还需税号。")

&emsp;&emsp;这段代码实现的是开票前的资格校验。它在流程里的作用是把"能不能开"这个判断从模型手里收回到代码里——订单存不存在、是不是已经开过、状态符不符合开票条件，这三道关卡全由确定性的代码逐条核对，模型不参与判断，它只负责把校验结果转述给用户。它对项目的意义在于，这正是 Swarm 这种最弱控制模式里加的一道确定性栏杆：控制权虽然交给了专员，但"已退款的订单不能开发票""一个订单不能重复开票"这类业务规则不能由模型自由发挥，必须用代码固定。校验通过后，工具不直接开票，而是返回一份需要向用户收集的信息清单，专员据此引导用户提供发票类型、抬头、税号——这就把多轮对话自然地接了起来。

&emsp;&emsp;信息收齐后，专员调用第二个工具真正开票落库。

In [ ]:
@function_tool  # 把普通函数标成 agent 可调用的工具
def issue_invoice(order_id: str, title: str, invoice_type: str, tax_no: str = "") -> str:
    """为订单真实开具发票并写入 invoices 表：普票要抬头，专票还要税号；返回发票号。"""
    oid = (order_id or "").strip().upper()
    if not title:                               # 抬头是必填，缺了先问
        return "缺少发票抬头，请先向用户确认抬头。"
    if "专用" in invoice_type and not tax_no:    # 专票必须有税号
        return "增值税专用发票必须提供税号，请向用户索要税号。"
    o = _query_order(oid)
    if not o or o["status"] not in _INVOICEABLE:        # 开票前再校验一次状态
        return f"订单 {oid} 当前不可开票。"
    if _query_invoice(oid):                              # 开票前再拦一次重复开
        return f"订单 {oid} 已开过发票，不能重复开。"
    n = _count_invoices()                                # 现有发票数，用来生成顺延单号
    inv_no = f"INV-2026-{1001 + n}"                       # 预置 2 张 → 第一张新开就是 INV-2026-1003
    _insert_invoice(inv_no, oid, title, tax_no, invoice_type, o["amount"])  # 真写进 invoices 表
    return f"开票成功：发票号 {inv_no}，抬头「{title}」，金额 ¥{o['amount']}。"

&emsp;&emsp;这段代码实现的是发票的真实开具与落库。它在流程里的作用是把多轮对话攒下来的信息一次性兑现成一条数据库记录——抬头缺了先问、专票没税号先要、开票前把状态和重复开两道关再核一遍，全过了才真的把发票写进 `invoices` 表并返回发票号。它对项目的意义在于完整展示了一个有状态、有校验、要落库的业务流程怎么在 Swarm 里落地：从查资格、引导收集信息到真开票，跨越好几轮对话，控制权可能中途还在专员之间转过手，但只要会话最终回到账单财务专员、信息也收齐了，这条开票流程就能闭合。发票号是按现有发票数顺延生成的，数据库里预置了两张，所以第一张新开出来的发票号就是 `INV-2026-1003`——这个号 10.2 的实测结果里我们会再见到。完整的工具集（含查订单、查物流、查商品、检索 FAQ 等另外几个）在项目源码里。

### 10.4 多轮续接

&emsp;&emsp;Swarm 的协作要真正成立，光有单轮的全互联交接还不够，控制权必须能跨轮续接。客服对话是连着的：用户这一轮在退款专员那里聊完退款方案，下一轮接着问"那这单的发票怎么办"，这时候该续接的不是分诊台、而是上一轮把他服务到最后的那个专员——他刚跟用户聊完，最清楚来龙去脉。如果每一轮都从分诊台重新分流，用户就得把背景重说一遍，对话的连续性就断了。所以我们要记住两样东西：完整的对话历史，以及上一轮最后接待的是哪个专员。下面这段是多轮续接的核心。

In [ ]:
# 会话状态：session_id -> {"input": 历史对话, "agent": 上一轮最后接待的专员}
SESSIONS = {}

async def run_events(query: str, session_id: str = "default") -> dict:
    state = SESSIONS.get(session_id)
    if state is None:
        # 服务重启后兜底：从 runs.db 把这个会话的历史轮重建成对话上下文
        hist = store.load_history(session_id)
        state = {"input": _rebuild(hist), "agent": "triage"} if hist else {}
    # 关键：从上一轮的专员续接，没有记录才回分诊台
    start_name = state.get("agent", "triage")
    start_agent = agents.get(start_name, triage)
    # 带着完整历史 + 这一轮的新问题往下跑；max_turns 只是单轮内部的防互踢保险丝
    input_list = list(state.get("input", [])) + [{"role": "user", "content": query}]
    result = Runner.run_streamed(start_agent, input_list, max_turns=25)
    # ...（中间是流式处理 handoff / 工具调用 / 回复事件，省略）...
    # 存回会话：下一轮带着完整历史 + 从这一轮最后接待的专员继续
    SESSIONS[session_id] = {"input": result.to_input_list(), "agent": result.last_agent.name}
    return {"final": result.final_output, "last_agent": result.last_agent.name}

&emsp;&emsp;这段代码实现的是会话的跨轮续接。它在流程里的作用是让控制权能在专员之间跨轮反复交换——`start_name` 取的是上一轮最后接待的专员，而不是固定回分诊台，所以用户连着问不同类型的问题时，会话会从上一轮的落点直接续上；跑完之后用 `result.to_input_list()` 把这一轮的完整对话存回去，同时把 `result.last_agent.name`（这一轮最后是谁在接待）记下来，作为下一轮的起点。它对项目的意义在于，这是 Swarm 去中心化协作在多轮维度上的延伸：单轮里控制权能在专员间横向传递，跨轮里这个落点被记了下来、还能继续传递，于是我们才能在实测里看到控制权在一通对话里于多个专员之间反复交换。

&emsp;&emsp;这里还有两个细节值得点出来。一个是重启兜底：`SESSIONS` 是内存里的字典，服务一重启就空了，所以当内存里查不到这个会话时，代码会从 `runs.db` 把它的历史轮重新读出来、拼回对话上下文，用户重启后还能接着之前的对话往下问。另一个是 `max_turns=25` 这个参数的真实含义，容易被误解。

> <font size=2>**提示**：`max_turns=25` 是单轮内部的回合保险丝，不是会话轮数上限。去中心化接力有一个隐患——两个专员可能互相把会话踢来踢去停不下来，A 觉得这事归 B、B 又觉得归 A。`max_turns` 限的就是单次运行内部模型回合的总数，给到 25 是为了留足正常交接和多次工具调用的余量，同时兜住"无限互踢"这种极端情况。它跟"用户能聊多少轮"完全是两回事——会话轮数本身不设上限，用户想问多少轮都行，每一轮都续接着上一轮往下走。</font>

### 10.5 实测复盘

&emsp;&emsp;最后把这个项目实测里值得记住的几条收束一下，都是真实跑下来踩出来的经验，留给我们自己上手时少走弯路。第一条就是第五章 5.2 那个招牌坑在完整项目里的实战复现，这次我们把它讲完整。

> <font size=2>**提示**：Agent 的 `name` 必须用英文，中文名会让整个 handoff 崩掉。这是这个项目踩过的最隐蔽的一个坑。handoff 在底层并不是什么特殊机制，它就是一个名叫 `transfer_to_<专员name>` 的工具——给专员起名"退款专员"，框架就生成一个叫 `transfer_to_退款专员` 的交接工具。问题出在 OpenAI 的 function-calling 对工具名有命名规则，非 ASCII 字符（也就是中文）会被统一替换成下划线，于是"退款专员""账单财务"这些不同的中文名，全都变成了同一个 `transfer_to_____`，六个交接工具撞成一个名字，handoff 直接抛出 ModelBehaviorError 崩掉。修法很简单但必须记牢：`Agent.name` 一律用英文（presale、refund、tech 等），所有中文展示放到 `instructions`、`handoff_description` 和前端 label 里去。本章 10.3 那段建专员的代码第一行注释标的就是这件事。</font>

> <font size=2>**提示**：必须在指令里明确要求"说转接就要真调交接工具，严禁口头转接"。去中心化接力的另一个坑是模型会"假装转接"——它嘴上跟用户说"好的，我帮您转接给账单财务同事"，但实际并没有调用那个交接工具，于是控制权根本没转走，下一句还是它自己在答，用户被晾在原地。所以每个专员的指令里都定死了一条硬规则：如果问题（或其中一部分）不归你管，必须立刻真正调用交接工具把用户转过去，严禁只在嘴上说"帮您转接"却不实际交接；先把归自己管的部分答完，再立即把剩下的交接出去。</font>

> <font size=2>**提示**：跨轮指代能成立，靠的是 10.4 那套"带完整历史续接"的设计。实测里用户说"她最近那单"这类带指代的表述，系统能正确续上是因为上一轮的完整对话被 `result.to_input_list()` 存了下来、这一轮整个带进了上下文，模型才能从历史里把"她"和"最近那单"对上。如果只传当前这一句、不带历史，这种指代就无从解析。这也提醒我们，多轮客服的上下文一定要完整带入，不能为了省 token 只传最后一句。</font>

> <font size=2>**提示**：演示环境里要让专员凭姓名或订单号就能代查，别把隐私当成拒绝的理由。这是个容易横跳的坑——模型有时会过度谨慎，用户报了本人姓名想查自己的订单，它却以"涉及隐私无法提供"为由拒绝，把一个本该顺畅的查询堵死。本案例是内部演示环境，用户围绕的都是自己的订单，所以专员指令里明确写了：用户报出订单号或本人姓名即可查询，不要拿隐私当拒绝理由。真实生产环境当然要做身份核验，但那是另一套机制，不该让模型在演示里凭感觉乱挡。</font>

&emsp;&emsp;到这里，第十章就走完了，第三部分 的四个案例也全部落地了。我们用四个能真跑的完整项目，把<font color=red>四种多智能体协作模式</font>从纸面机制带到了真实业务里——第七章的 Workflow 流程编排落成了销售数据报表，第八章的 Supervisor 集中调度落成了多轮研究编排，第九章的 Hierarchical 分层协同落成了多智能体软件交付，这一章的 Swarm 自主协作落成了在线客服中心。四种结构沿着"控制权从集中到分散"这条线一字排开：Workflow 把控制流固定在代码里，Supervisor 把控制权攥在一个中央主管手里，Hierarchical 让控制权在层级间逐层下钻，到了 Swarm 这里，控制权干脆交还给了每一个平级的 agent。

&emsp;&emsp;这一章也给整门课"模型不确定性、确定性兜底"这条主题线补上了最后一块。越是控制权分散的模式，越需要工程上的确定性栏杆来兜底：本章里 `Agent.name` 固定用英文规避撞名、开票资格用代码确定性校验而不让模型自由判断、`max_turns` 保险丝防专员互踢、续接落点记在会话状态里而不靠模型自己记，这几道兜底叠在一起，才让 Swarm 这种最弱控制的模式在真实客服业务里稳得住。完整的工程代码——包含本章略去的六专员完整定义、分诊台指令、另外几个查库工具、流式事件处理和那个无中心网状前端——都在项目源码 `代码/swarm-customer-service/` 里，建议照着 10.2 的命令亲手跑一遍，发一句会拐弯的问题，看着控制权在网上一跳一跳地传，体感最深。

&emsp;&emsp;到这里，四种协作模式和四个完整项目就全部讲完了。下一章我们做一次完整的课程回顾，把这一路从单个智能体的能力上限讲起、走过四种结构、落地四个项目的旅程串成一条线，带大家从终点回望来路，看清楚这门课到底带走了哪些能力。

## <center>第十一章 课程回顾</center>

&emsp;&emsp;一门课走到这里，最值得做的一件事不是急着合上笔记本，而是站到终点回头望一眼来路。十一章一路走下来，每一章都在解决一个具体的问题、留下一个具体的产物，单看像是十一段各自独立的旅程；可当它们串成一条线，才会显出这门课真正想交到我们手里的东西——不是某一个框架的 API，而是一套"拿到任何业务需求，知道该用哪种模式、配哪个框架、怎么让它在生产里稳得住"的判断力。

&emsp;&emsp;下面这张回顾图把整条来路重新铺开：从最左边那个被任务压垮的单个智能体出发，经过四种协作结构的演进，落到四个能在浏览器里跑起来的真实系统，终点是一张把所有模式和框架关系都摊开的全景。我们就顺着这条线，把走过的每一站重新认一遍。

<div align=center><font size=2 color=#999999>第十一章：从单个智能体到四种结构、四个系统的完整旅程回望</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/multi-agent/2026-06-11/L11-recap-ae6ea8b0.png" width=80%></div>

<br>

### 11.1 我们走过的来路

&emsp;&emsp;这门课的起点，是第一章那个单个智能体被任务压垮的画面。工具越挂越多、上下文塞到爆、一个提示词身兼数职——<font color=red>单个智能体的能力上限</font>不是不努力，而是一个人扛不下一整摊复杂的活。认清这个能力上限，才能理解后面所有模式存在的理由：把一个智能体扛不动的活，拆给一支智能体团队来分工。这就是从"一个智能体"到"一支智能体团队"的视角切换，也是整门课的第一块基石。有了这个动机，第二到第五章我们逐一拆透了四种主流的协作结构。Workflow 流程编排把控制流固定在代码里，工序一道接一道，最稳也最可预测；Supervisor 集中调度立起一个主管，由它派单给多个专家再综合，是最容易控制、也最通用的结构；Hierarchical 分层协同在主管之上再分层，总监管组长、组长管工程师，专治一个主管管不过来的超复杂项目；Swarm 自主协作则把"下一步交给谁"的控制权交还给每个智能体，让对话焦点跟着用户的需求在专员之间自由流转。

&emsp;&emsp;这四种模式沿着一条清晰的主线排开——控制权从代码手里一点点松开，组织结构从单链长成环状，自主性越来越强，可控性越来越弱。每一种模式我们都用 LangGraph、CrewAI、OpenAI Agents SDK、Microsoft Agent Framework、Claude Agent SDK 五个主流框架各写了一段能直接运行的最小实现，亲眼看清同一种模式在五家框架里分别长什么样、原生支持还是要拼装。第六章是承上启下的一座桥。把四种模式都见过之后，真正的难点不在"会不会写"，而在"该选哪个"。我们用一道四问决策树把"什么业务用什么模式"立成了可查的判断框架，并补上一条最高层的原则——拿不准时默认先考虑 Supervisor，再按任务性质做特化。这套选型判断力，正是这门课最想留下的核心能力。

&emsp;&emsp;第七到第十章，四个完整的案例项目把前面学的模式和框架真正落到了地：CrewAI 实现的销售数据报表流水线、Microsoft Agent Framework 实现的多轮研究编排、LangGraph 实现的多智能体软件交付、OpenAI Agents SDK 实现的在线客服中心。四个项目各对应一种协作结构，从需求一直跑到浏览器里能点开的界面，是简历级的完整作品。

### 11.2 学完带走了什么

&emsp;&emsp;一门工程课的可信度，最终要落在"学完能不能拿出真东西"上。这门课交到我们手里的，是三样可以立刻验证的产物：四个能在本机跑起来的完整案例系统、一张把四模式与五框架关系全摊开的支持矩阵、以及第二到第五章那二十段能直接运行的最小实现代码。下面这张表把四个系统的结构模式、框架、端口和启动入口逐一登记，照着就能逐个确认它们真的能跑。

<div align=center><font size=2 color=#999999>四个完整案例系统 self-check 一览（结构 / 框架 / 端口 / 项目目录）</font></div>

<div align="center">
<table width="80%">
<thead><tr>
<th>产物</th><th>结构模式</th><th>框架</th><th>端口</th><th>项目目录（代码/ 下）</th>
</tr></thead>
<tbody>
<tr><td>销售数据报表流水线</td><td>Workflow 流程编排</td><td>CrewAI</td><td>8091</td><td>workflow-sales-report</td></tr>
<tr><td>多轮研究编排</td><td>Supervisor 集中调度</td><td>Microsoft Agent Framework</td><td>8088</td><td>flagship-supervisor-tool</td></tr>
<tr><td>多智能体软件交付</td><td>Hierarchical 分层协同</td><td>LangGraph</td><td>8090</td><td>hierarchical-software-delivery</td></tr>
<tr><td>在线客服中心</td><td>Swarm 自主协作</td><td>OpenAI Agents SDK</td><td>8092</td><td>swarm-customer-service</td></tr>
</tbody>
</table>
</div>

<br>

&emsp;&emsp;除了四个系统，我们还带走了一张<font color=red>四模式 × 五框架支持矩阵</font>——它在第五章 5.6 业界态度与 第二部分 全景那一节就摊在了明面上，把每个框架对每种模式是原生、官方示例级、可拼装还是不适配标得清清楚楚。再加上第二到第五章每个框架小节那二十段最小可跑代码，它们都是可以直接拷进 Jupyter cell 运行的种子，下次要快速验证某个框架对某种模式的支持，翻出对应那段改改提示词就能跑。

&emsp;&emsp;要确认四个系统是不是真能跑起来，最直接的办法就是挑一个项目启动、再用一次健康检查看它有没有应答。四个项目的启动方式完全一致——进项目目录，用对应的虚拟环境把 uvicorn 起在 127.0.0.1 加各自的端口上。下面以销售报表流水线（8091）为例给出三平台的启动与验证命令，其余三个项目把目录、虚拟环境和端口换成上表对应的值即可。

```bash
# macOS / Linux：进项目目录，用对应虚拟环境启动，再 curl 健康检查确认应答
cd 代码/workflow-sales-report
.venv/bin/python -m uvicorn backend.app:app --host 127.0.0.1 --port 8091 &
sleep 3 && curl -s http://127.0.0.1:8091/ | head -c 200    # 能返回 HTML 即说明服务起来了
```

```powershell
# Windows PowerShell：路径分隔符用反斜杠，虚拟环境的 python 在 Scripts 目录
cd 代码\workflow-sales-report
.venv\Scripts\python -m uvicorn backend.app:app --host 127.0.0.1 --port 8091
# 另开一个终端：Invoke-WebRequest http://127.0.0.1:8091/
```

```bash
# Windows Git Bash：路径写法和 macOS 一致，正斜杠即可
cd 代码/workflow-sales-report
.venv/Scripts/python -m uvicorn backend.app:app --host 127.0.0.1 --port 8091
```

&emsp;&emsp;启动成功后，在浏览器里打开 `http://127.0.0.1:8091`，就能看到那条竖排的报表流水线界面。把四个项目都这样跑一遍，端口分别换成 8088、8090、8092，看着四种不同的协作结构在浏览器里分别动起来——流水线一道道往下传、星形派单并行回流、组织树层层打回返工、客服网状结构上控制权一跳一跳地转——这门课带走的东西就实打实地摆在眼前了。

### 11.3 全课关键提示速查

&emsp;&emsp;这一节把全课的关键提示收进一张<b>速查表</b>——它收的是贯穿 第一部分 到 第三部分、和模式选型与框架原语相关的那些落点，外加四个案例项目里实打实踩过、影响面最大的几条工程坑。下面这张表按"落点 / 提示 / 修法"汇总全课各章最值得记牢的提示。

<div align=center><font size=2 color=#999999>全课关键提示速查表（落点 / 提示 / 修法）</font></div>

<div align="center">
<table width="80%">
<thead><tr>
<th>落点</th><th>提示</th><th>修法</th>
</tr></thead>
<tbody>
<tr><td>第一章 立项算账</td><td>多智能体 token 消耗约为聊天交互的 15 倍，上线后看账单才发现就晚了</td><td>立项时按"调用量 × 约 15 倍 token"先估成本，扛得住再拆</td></tr>
<tr><td>第二章 MAF 顺序编排</td><td><code>agent-framework-orchestrations</code> 子包还是候选版，配第三方端点加深度思考模型时不稳，产出会损坏成碎片</td><td>退回更底层的 <code>as_agent</code> 手动串工序，生产用 MAF 高层编排配官方推荐端点</td></tr>
<tr><td>第二章 CrewAI 接入</td><td>litellm 走 OpenRouter 时模型名漏掉 <code>openrouter/</code> 前缀，报找不到模型提供方</td><td>模型名写成 <code>openrouter/模型名</code> 再传给 <code>LLM(...)</code></td></tr>
<tr><td>第三章 Supervisor 控制流</td><td><code>create_supervisor</code> 让模型自主决定派单，deepseek 同代码两跑一次全派一次不派</td><td>关键流程改 <code>StateGraph</code> 显式建图，路由固定在边里，模型只产内容</td></tr>
<tr><td>第四章 多层自主</td><td>嵌套主管层级越深，transfer 轮次成倍增加，偶发的不规范交接更容易撞上</td><td>层级两层封顶；各层主管换思考型模型，或加一层重试兜住偶发失败</td></tr>
<tr><td>第五章 Swarm handoff</td><td><code>Agent.name</code> 用中文，<code>transfer_to_&lt;name&gt;</code> 工具名里非 ASCII 被替换成下划线，多专员撞成同名导致崩溃</td><td><code>Agent.name</code> 一律英文，中文角色名放 <code>instructions</code> / <code>handoff_description</code></td></tr>
<tr><td>第五章 多轮状态</td><td>langgraph-swarm 不配 checkpointer，跨轮丢"当前接待者"，每轮都从分诊台重来</td><td><code>compile(checkpointer=...)</code> 配上并带 <code>thread_id</code>，接待状态跨轮保留</td></tr>
<tr><td>第六章 模式选型</td><td>被"效果更好"吸引先搭起来，忽略了多智能体的成本和失控风险</td><td>用四问决策树定位模式，拿不准默认 Supervisor，再按任务性质做特化</td></tr>
<tr><td>第八章 专家工具循环</td><td>专家开 agentic 工具循环时把工具结果误当用户输入，答非所问</td><td>改确定性 RAG：代码先检索、资料进提示词、专家单轮总结</td></tr>
<tr><td>第八章 输出截断</td><td>reasoning 模型把 token 花在思考上，正文被截成半截甚至空白</td><td>专家给足 max_tokens，正文太短时用专家完整结论兜底再综合一次</td></tr>
<tr><td>第九章 LLM 生成代码</td><td><code>dict(row)</code> 对 sqlite tuple 崩、<code>(lst or []).append()</code> 对空列表丢数据</td><td>用接口定义预防字段与结构，再用测试驱动返工兜底</td></tr>
</tbody>
</table>
</div>

<br>

&emsp;&emsp;这张表覆盖了从立项决策、模式选型到框架原语的全链路提示。真到自己动手搭多智能体系统时，遇到拿不准的落点，先翻这张表对一对——多数高频的坑，前人都已经替我们踩过、也记下了修法。

&emsp;&emsp;回到这门课最开始那个画面——一个被任务压垮的单个智能体。我们从它的能力上限出发，认识了四种让一支智能体团队分工协作的结构，看清了五个主流框架各自的世界观，立起了"什么业务用什么模式"的选型判断，又亲手把四个完整的系统从需求跑到了浏览器里能点开的界面。从"一个智能体"到"一支智能体团队"，这条路我们已经完整走过一遍。真正带走的，不是某一段代码，而是那杆能在任何业务场景里掂量"该用哪种模式、配哪个框架、怎么兜住不确定性"的秤。带着这套选型判断力上路，下一个多智能体系统，我们就有底气从第一行代码开始把它搭对。